# Dataset Loading and Merging

In [ ]:
# 🧩 1. Dataset Overview & Initial Checks
# Machine Learning-Based Forecasting of Food Commodity Price Volatility in Somalia
# Author: Asma Abdiwali Hassan

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load both datasets to compare columns

wb_exchange = pd.read_csv('data/raw/exchange_rate.csv')
food_data = pd.read_csv('data/raw/wfp_food_prices_som.csv')

print("COLUMN COMPARISON")
print("="*50)

print("\nWorld Bank Exchange Rate columns:")
for i, col in enumerate(wb_exchange.columns):
    print(f"{i+1:2d}. {col}")

print(f"\nWFP Food Price Dataset columns:")
for i, col in enumerate(food_data.columns):
    print(f"{i+1:2d}. {col}")

print("\nGEOGRAPHIC NAME COMPARISON")
print("="*30)

print(f"\nWorld Bank regions (adm1_name):")
print(wb_exchange['adm1_name'].unique())

print(f"\nWFP dataset regions (admin1):")
print(food_data['admin1'].unique())

print(f"\nWorld Bank markets (mkt_name):")
print(f"Total unique markets: {wb_exchange['mkt_name'].nunique()}")
print("Sample markets:", wb_exchange['mkt_name'].unique()[:10])

print(f"\nWFP dataset markets:")
print(f"Total unique markets: {food_data['market'].nunique()}")
print("Sample markets:", food_data['market'].unique()[:10])

print("\nDATE RANGE COMPARISON")
print("="*25)
print(f"World Bank: {wb_exchange['year'].min()}-{wb_exchange['year'].max()}")

# Check what's in the WFP date column first
print(f"\nWFP date column sample:")
print(food_data['date'].head(10))
print(f"Date column type: {food_data['date'].dtype}")

In [ ]:
print("DATASET SIZE COMPARISON")
print("="*40)

print(f"\nWorld Bank Exchange Rate Dataset:")
print(f"Total records: {len(wb_exchange):,}")
print(f"Records with valid exchange rates: {wb_exchange['exchange_rate_unofficial'].notna().sum():,}")
print(f"Date range: {wb_exchange['year'].min()}-{wb_exchange['year'].max()}")

print(f"\nWFP Food Price Dataset:")
print(f"Total records: {len(food_data):,}")
print(f"Records with valid prices: {food_data['price'].notna().sum():,}")

# Check food data date range properly
food_data['date'] = pd.to_datetime(food_data['date'])
print(f"Date range: {food_data['date'].dt.year.min()}-{food_data['date'].dt.year.max()}")

print(f"\nOVERLAP ANALYSIS:")
print(f"WFP records from 2007 onwards: {(food_data['date'].dt.year >= 2007).sum():,}")
print(f"WFP records before 2007: {(food_data['date'].dt.year < 2007).sum():,}")

print(f"\nCOMMODITY BREAKDOWN:")
print(f"Food commodities: {(~food_data['category'].str.contains('non-food', na=False)).sum():,}")
print(f"Non-food items: {(food_data['category'].str.contains('non-food', na=False)).sum():,}")

In [ ]:
# Check unit distribution before filtering
print("UNIT DISTRIBUTION ANALYSIS")
print("=" * 40)

# Overall unit distribution
unit_counts = food_data['unit'].value_counts()
print(f"Total unique units: {len(unit_counts)}")
print(f"\nUnit distribution:")
for unit, count in unit_counts.items():
    percentage = (count / len(food_data)) * 100
    print(f"  {unit}: {count:,} ({percentage:.1f}%)")

# Check which units might be affected by non-food filtering
non_food_mask = food_data['category'].str.contains('non-food', na=False)
non_food_units = food_data[non_food_mask]['unit'].value_counts()

print(f"\nUnits in non-food items that will be removed:")
for unit, count in non_food_units.items():
    remaining = unit_counts[unit] - count
    print(f"  {unit}: {count:,} removed, {remaining:,} remaining")

print(f"\nUnits that will have very low counts after filtering:")
for unit, count in unit_counts.items():
    non_food_count = non_food_units.get(unit, 0)
    remaining = count - non_food_count
    if remaining < 100:  # Threshold for "small"
        print(f"  {unit}: {remaining:,} records remaining")

In [ ]:
# Filter out non-food items
food_only = food_data[~food_data['category'].str.contains('non-food', na=False)]

print(f"Original records: {len(food_data):,}")
print(f"Food-only records: {len(food_only):,}")
print(f"Removed records: {len(food_data) - len(food_only):,}")

In [ ]:
# Before and After Unit Distribution Comparison
print("BEFORE vs AFTER FILTERING: UNIT DISTRIBUTION")
print("=" * 60)

# Before filtering (original data)
before_units = food_data['unit'].value_counts()
total_before = len(food_data)

print("BEFORE FILTERING:")
print(f"Total records: {total_before:,}")
for unit, count in before_units.items():
    percentage = (count / total_before) * 100
    print(f"  {unit}: {count:,} ({percentage:.1f}%)")

# After filtering
after_units = food_only['unit'].value_counts()
total_after = len(food_only)

print(f"\nAFTER FILTERING:")
print(f"Total records: {total_after:,}")
for unit, count in after_units.items():
    percentage = (count / total_after) * 100
    print(f"  {unit}: {count:,} ({percentage:.1f}%)")

# Change summary
print(f"\nCHANGE SUMMARY:")
print(f"Records removed: {total_before - total_after:,}")
print(f"Percentage retained: {(total_after/total_before)*100:.1f}%")

print(f"\nUNIT-SPECIFIC CHANGES:")
all_units = set(before_units.index) | set(after_units.index)
for unit in sorted(all_units):
    before_count = before_units.get(unit, 0)
    after_count = after_units.get(unit, 0)
    removed = before_count - after_count

    if removed > 0:
        retention_rate = (after_count / before_count) * 100 if before_count > 0 else 0
        print(f"  {unit}: {before_count:,} → {after_count:,} ({removed:,} removed, {retention_rate:.1f}% retained)")
    else:
        print(f"  {unit}: {before_count:,} → {after_count:,} (no change)")

In [ ]:
# Create a detailed comparison table
wb_regions = set(wb_exchange['adm1_name'].dropna())
food_regions = set(food_only['admin1'].dropna())

matching = wb_regions.intersection(food_regions)
wb_only = wb_regions - food_regions
food_only_regions = food_regions - wb_regions

print("REGION COMPARISON TABLE")
print("="*40)

print(f"\nEXACT MATCHES ({len(matching)}):")
for region in sorted(matching):
    print(f"   {region}")

print(f"\nWORLD BANK ONLY ({len(wb_only)}):")
for region in sorted(wb_only):
    print(f"  - {region}")

print(f"\nFOOD DATA ONLY ({len(food_only_regions)}):")
for region in sorted(food_only_regions):
    print(f"  - {region}")

print(f"\nSUMMARY:")
print(f"  Match Rate: {len(matching)}/{len(food_regions)} = {len(matching)/len(food_regions)*100:.1f}%")
print(f"  Perfect region alignment!")

In [ ]:
# Remove the "Market Average" entries from World Bank data
wb_clean = wb_exchange[wb_exchange['adm1_name'] != 'Market Average']

print(f"Original WB records: {len(wb_exchange):,}")
print(f"After removing 'Market Average': {len(wb_clean):,}")
print(f"Removed records: {len(wb_exchange) - len(wb_clean):,}")

In [ ]:
# Add year/month columns to food data for matching
food_only['year'] = food_only['date'].dt.year
food_only['month'] = food_only['date'].dt.month

# Check date distribution
print("DATE ALIGNMENT CHECK")
print("="*30)
print(f"\nWorld Bank data points per year:")
wb_yearly = wb_clean.groupby('year').size()
print(wb_yearly.head())

print(f"\nFood data points per year:")
food_yearly = food_only.groupby('year').size()
print(food_yearly.head())

print(f"\nSample date formats:")
print(f"WB: year={wb_clean['year'].iloc[0]}, month={wb_clean['month'].iloc[0]}")
print(f"Food: year={food_only['year'].iloc[0]}, month={food_only['month'].iloc[0]}")

In [ ]:
# Fix the copy warning by creating a proper copy
food_clean = food_only.copy()
food_clean['year'] = food_clean['date'].dt.year
food_clean['month'] = food_clean['date'].dt.month

print("MATCHING POTENTIAL CHECK")
print("="*30)

# Check how many region-month combinations we can potentially match
wb_combinations = wb_clean.groupby(['adm1_name', 'year', 'month']).size().reset_index(name='wb_count')
food_combinations = food_clean.groupby(['admin1', 'year', 'month']).size().reset_index(name='food_count')

print(f"World Bank region-month combinations: {len(wb_combinations):,}")
print(f"Food data region-month combinations: {len(food_combinations):,}")

# Test merge to see potential matches
test_merge = food_combinations.merge(
    wb_combinations,
    left_on=['admin1', 'year', 'month'],
    right_on=['adm1_name', 'year', 'month'],
    how='inner'
)

print(f"Potential direct matches: {len(test_merge):,}")
print(f"Match rate: {len(test_merge)/len(food_combinations)*100:.1f}%")

In [ ]:
# Check if  food markets match World Bank markets
wb_markets = set(wb_clean['mkt_name'].unique())
food_markets = set(food_clean['market'].unique())
market_matches = wb_markets.intersection(food_markets)

print("MARKET MATCHING ANALYSIS")
print("="*30)
print(f"World Bank markets: {len(wb_markets)}")
print(f"Food data markets: {len(food_markets)}")
print(f"Exact market matches: {len(market_matches)}")
print(f"Market match rate: {len(market_matches)/len(food_markets)*100:.1f}%")

print(f"\nSample matching markets:")
print(sorted(list(market_matches)[:10]))

print(f"\nFood markets without WB match:")
unmatched = food_markets - wb_markets
print(f"Count: {len(unmatched)}")
print(sorted(list(unmatched)[:10]))

In [ ]:
# Merge at market level for precise exchange rates
final_data = food_clean.merge(
    wb_clean[['adm1_name', 'mkt_name', 'year', 'month', 'exchange_rate_unofficial']],
    left_on=['admin1', 'market', 'year', 'month'],
    right_on=['adm1_name', 'mkt_name', 'year', 'month'],
    how='left'
)

print("MARKET-LEVEL MERGE RESULTS")
print("="*30)
print(f"Original food records: {len(food_clean):,}")
print(f"Final merged records: {len(final_data):,}")
print(f"Records with exchange rates: {final_data['exchange_rate_unofficial'].notna().sum():,}")
print(f"Missing exchange rates: {final_data['exchange_rate_unofficial'].isna().sum():,}")

# Check the unmatched records
unmatched = final_data[final_data['exchange_rate_unofficial'].isna()]
if len(unmatched) > 0:
    print(f"\nUnmatched records breakdown:")
    print(unmatched['market'].value_counts())

print(f"\nExchange rate range: {final_data['exchange_rate_unofficial'].min():.0f} to {final_data['exchange_rate_unofficial'].max():.0f}")

In [ ]:
# 📍 Fix missing values for Owdweyne & mkt_name
print("=== FIXING MISSING VALUES ===")

# 1. Fill exchange_rate_unofficial (Owdweyne with regional averages)
owdweyne_region = final_data.loc[final_data['market'] == 'Owdweyne', 'admin1'].iloc[0]

regional_avg = (
    wb_clean[wb_clean['adm1_name'] == owdweyne_region]
    .groupby(['year','month'])['exchange_rate_unofficial']
    .mean().reset_index()
)

# Merge and fill with regional averages
final_data = final_data.merge(
    regional_avg, on=['year','month'], how='left', suffixes=('', '_regional')
)
final_data['exchange_rate_unofficial'] = final_data['exchange_rate_unofficial'].combine_first(
    final_data['exchange_rate_unofficial_regional']
)

# 2. Fill missing mkt_name using market
final_data['mkt_name'] = final_data['mkt_name'].combine_first(final_data['market'])


# ✅ Summary
print(f"Remaining missing exchange_rate_unofficial: {final_data['exchange_rate_unofficial'].isna().sum()}")
print(f"Remaining missing mkt_name: {final_data['mkt_name'].isna().sum()}")

In [ ]:
# ✅ Verify all fills worked correctly
print("\n=== VERIFICATION SUMMARY ===")

# 1. market vs mkt_name
mismatched_market = final_data[final_data['market'] != final_data['mkt_name']]
print(f"📍 Market mismatched records: {len(mismatched_market)}")
if not mismatched_market.empty:
    print(mismatched_market[['market', 'mkt_name']].head())
else:
    print("✅ All market names match")

# 2. admin1 vs adm1_name
if "adm1_name" in final_data.columns:  # only if the column exists
    mismatched_admin = final_data[
        (final_data['admin1'].notna()) & (final_data['adm1_name'].notna()) &
        (final_data['admin1'] != final_data['adm1_name'])
    ]
    print(f"\n📍 Admin mismatched records: {len(mismatched_admin)}")
    if not mismatched_admin.empty:
        print(mismatched_admin[['admin1', 'adm1_name']].head())
    else:
        print("✅ All admin1 and adm1_name match")
else:
    print("\n📍 Column 'adm1_name' already dropped. Using only 'admin1'.")

# 3. Exchange rate check
missing_exchange = final_data['exchange_rate_unofficial'].isna().sum()
print(f"\n📍 Missing exchange_rate_unofficial after fill: {missing_exchange}")
if missing_exchange == 0:
    print("✅ All exchange rates filled")
else:
    print("⚠️ Some exchange rates are still missing")

# Overall dataset summary
print("\n=== DATASET SUMMARY ===")
print(f"Total records       : {len(final_data):,}")
print(f"Markets             : {final_data['market'].nunique():,}")
print(f"Regions (admin1)    : {final_data['admin1'].nunique():,}")
print(f"Districts (admin2)  : {final_data['admin2'].nunique():,}")

In [ ]:
# Check if the merge created duplicates
print("DUPLICATION CHECK")
print("="*20)
print(f"Original food records: {len(food_clean):,}")
print(f"Merged records: {len(final_data):,}")
print(f"Difference: {len(final_data) - len(food_clean):,}")

# Check for actual duplicates in the World Bank data at market level
wb_market_duplicates = wb_clean.groupby(['adm1_name', 'mkt_name', 'year', 'month']).size()
market_duplicates = wb_market_duplicates[wb_market_duplicates > 1]

print(f"\nMarket-level duplicates in WB data: {len(market_duplicates)}")
if len(market_duplicates) > 0:
    print("Sample duplicates:")
    print(market_duplicates.head())

# Check if food data has any internal duplicates
food_duplicates = food_clean.groupby(['admin1', 'market', 'year', 'month']).size()
food_dups = food_duplicates[food_duplicates > 1]
print(f"\nFood data duplicates: {len(food_dups)}")

In [ ]:
# Check what's causing the food data duplicates
food_duplicates_detail = food_clean.groupby(['admin1', 'market', 'year', 'month']).size().reset_index(name='count')
duplicate_entries = food_duplicates_detail[food_duplicates_detail['count'] > 1]

print("FOOD DATA DUPLICATE ANALYSIS")
print("="*30)
print(f"Combinations with duplicates: {len(duplicate_entries)}")
print(f"Max duplicates per combination: {duplicate_entries['count'].max()}")
print(f"Average duplicates: {duplicate_entries['count'].mean():.1f}")

# Show what's causing duplicates - likely different commodities
sample_duplicate = food_clean[
    (food_clean['admin1'] == duplicate_entries.iloc[0]['admin1']) &
    (food_clean['market'] == duplicate_entries.iloc[0]['market']) &
    (food_clean['year'] == duplicate_entries.iloc[0]['year']) &
    (food_clean['month'] == duplicate_entries.iloc[0]['month'])
]

print(f"\nSample duplicate records:")
print(sample_duplicate[['admin1', 'market', 'commodity', 'year', 'month']].head())

In [ ]:
# ===============================
#  Dataset Structure Analysis
# ===============================
def dataset_overview(df, date_col='date'):
    print("="*60)
    print("🧩 DATASET OVERVIEW & INITIAL CHECKS")
    print("="*60)
    print(f"📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    # --- Structure ---
    print("\n" + "="*40)
    print("📋 DATASET STRUCTURE ANALYSIS")
    print("="*40)
    print(f"💾 Memory usage: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
    for i, col in enumerate(df.columns, 1):
        miss = df[col].isna().sum()
        dtype_str = str(df[col].dtype)   # ✅ fix here
        print(f"{i:2d}. {col:<15} | {dtype_str:<10} "
              f"| Non-null: {df[col].notna().sum():,} | Missing: {miss:,} "
              f"({miss/len(df)*100:5.1f}%)")

    # --- Data types ---
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    cat = df.select_dtypes(include=['object']).columns.tolist()
    dt  = [c for c in df.columns if 'date' in c.lower()]
    print("\n🔍 DATA TYPE ANALYSIS")
    print(f"🔢 Numeric: {num}\n📝 Categorical: {cat}\n📅 Potential datetime: {dt}")

    # --- Temporal ---
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        valid = df.dropna(subset=[date_col])
        if not valid.empty:
            start, end = valid[date_col].min(), valid[date_col].max()
            print("\n📅 TEMPORAL COVERAGE ANALYSIS")
            print(f"📅 Date range: {start.date()} to {end.date()}")
            print(f"⏱️  Total span: {(end-start).days:,} days ({(end-start).days/365.25:.1f} years)")
            counts = valid[date_col].dt.year.value_counts().sort_index()
            for y, c in counts.items():
                print(f"{y}: {c:>6,} ({c/len(valid)*100:5.1f}%)")

    # --- Geographic ---
    print("\n🌍 GEOGRAPHIC COVERAGE ANALYSIS")
    if 'admin1' in df: print(f"🏛️ Regions: {df['admin1'].nunique()}")
    if 'admin2' in df: print(f"🏘️ Districts: {df['admin2'].nunique()}")
    if 'market' in df:
        print(f"🏪 Markets: {df['market'].nunique()}")
        print(df['market'].value_counts().head(10))
    if 'latitude' in df and 'longitude' in df:
        coords = df[['latitude','longitude']].dropna()
        print(f"🗺️ Records with coordinates: {len(coords):,} ({len(coords)/len(df)*100:.1f}%)")

    # --- Commodities ---
    if 'category' in df:
        print("\n🍽️ COMMODITY DISTRIBUTION")
        for k,v in df['category'].value_counts().items():
            print(f"{k:<25}: {v:>6,} ({v/len(df)*100:5.1f}%)")
    if 'commodity' in df:
        print(f"📦 Unique commodities: {df['commodity'].nunique()}")
        print(df['commodity'].value_counts().head(15))

    # --- Currency & Prices ---
    if 'currency' in df:
        print("\n💰 CURRENCY & PRICE ANALYSIS")
        for k,v in df['currency'].value_counts().items():
            print(f"{k}: {v:,} ({v/len(df)*100:5.1f}%)")
    for col in ['price','usdprice']:
        if col in df:
            vals = pd.to_numeric(df[col], errors='coerce').dropna()
            if not vals.empty:
                print(f"\n💲 {col.upper()} → Mean {vals.mean():.2f}, Min {vals.min()}, Max {vals.max()}, Std {vals.std():.2f}")

    # --- Data Quality ---
    print("\n✅ DATA QUALITY ASSESSMENT")
    miss = df.isna().sum()
    for c in df.columns:
        if miss[c]>0:
            status = "🔴" if miss[c]/len(df)>0.5 else "🟡" if miss[c]/len(df)>0.1 else "🟢"
            print(f"{status} {c:<15}: {miss[c]:,} ({miss[c]/len(df)*100:5.1f}%)")
        else:
            print(f"🟢 {c:<15}: Complete")

# Run
dataset_overview(final_data)

# 🧹 2. Data Cleaning & Preprocessing

In [ ]:
# Rename the dataframe
dataset = final_data.copy()

In [ ]:
# Drop unnecessary columns
cols_to_drop = [
    "exchange_rate_unofficial_regional",
    "adm1_name",
    "mkt_name",
    "commodity_id",
    "market_id"
]

dataset.drop(columns=[c for c in cols_to_drop if c in dataset.columns], inplace=True)

# ✅ Verify
print("Dropped columns successfully!")
print("Remaining columns:")
print(dataset.columns.tolist())

In [ ]:
# Step 2: Drop Unnecessary Columns
# Based on our analysis, drop redundant ID columns

print("Original columns:")
print(final_data.columns.tolist())
print(f"\nOriginal shape: {final_data.shape}")
print(f"\nAfter dropping unnecessary columns:")
print(f"New shape: {dataset.shape}")
print(f"Dropped columns: {cols_to_drop}")
print(f"\nRemaining columns:")
print(dataset.columns.tolist())

In [ ]:
# Import Libraries and Load Data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Create summary table
summary = pd.DataFrame({
    "Data Type": dataset.dtypes,
    "Non-Null Count": dataset.notnull().sum(),
    "Missing Count": dataset.isnull().sum(),
    "Missing %": (dataset.isnull().sum() / len(dataset) * 100).round(1)
})

print(f"Dataset Shape: {dataset.shape}\n")
print("📋 Dataset Overview:")
display(summary)

In [ ]:
# Summarize missing values for admin1, admin2, latitude, and longitude by market
missing_summary = (
    dataset.groupby('market')[['admin1', 'admin2', 'latitude', 'longitude']]
    .apply(lambda x: x.isna().sum())
    .reset_index()
)

# Sort by markets with most missing values
missing_summary = missing_summary.sort_values(
    by=['admin1', 'admin2', 'latitude', 'longitude'], ascending=False
)

print("📊 Missing Regions, Districts, Latitude, and Longitude by Market")
print(missing_summary)

In [ ]:
# Step 3: Fix Data Types & Summarize
import pandas as pd

# Conversions
dataset['date'] = pd.to_datetime(dataset['date'], errors='coerce')
for col in ['price','usdprice','latitude','longitude']:
    dataset[col] = pd.to_numeric(dataset[col], errors='coerce')

# Summary table
dtype_summary = pd.DataFrame({
    "Column": dataset.columns,
    "Data Type": dataset.dtypes.astype(str),
    "Non-Null Count": dataset.notna().sum(),
    "Missing Count": dataset.isna().sum(),
    "Missing %": (dataset.isna().sum()/len(dataset)*100).round(1)
})

print(f"Dataset Shape: {dataset.shape}")
dtype_summary

In [ ]:
# Rename columns for clarity:
dataset.rename(columns={
    'admin1': 'region',
    'admin2': 'district'
}, inplace=True)

In [ ]:
# Step 4: Handle Missing Values - Geographic Data
# Address missing admin1, admin2, latitude, longitude using systematic mapping
from geopy.geocoders import Nominatim

# Create a geolocator object with user_agent
geolocator = Nominatim(user_agent="geoapi")

# Geocode the location name
location = geolocator.geocode("Wadajir Market, Somalia")

# Manually assign Wadajir's correct coordinates from Google Maps
dataset.loc[(dataset['market'] == 'Wadajir') & (dataset['latitude'].isnull()), 'latitude'] = 2.0212614
dataset.loc[(dataset['market'] == 'Wadajir') & (dataset['longitude'].isnull()), 'longitude'] = 45.2946934
# Impute region and district
dataset.loc[(dataset['market'] == 'Wadajir') & (dataset['region'].isnull()), 'region'] = 'Banadir'
dataset.loc[(dataset['market'] == 'Wadajir') & (dataset['district'].isnull()), 'district'] = 'Wadajir'

In [ ]:
# Impute Missing values of USDPrice and calculate as new USD prices

dataset['new_usdprice'] = dataset['price'] / dataset['exchange_rate_unofficial']

print("FINAL DATASET SUMMARY")
print("="*25)
print(f"Total records: {len(dataset):,}")
print(f"Complete exchange rates: {dataset['exchange_rate_unofficial'].notna().sum():,}")
print(f"Exchange rate range: {dataset['exchange_rate_unofficial'].min():.0f} to {dataset['exchange_rate_unofficial'].max():.0f}")
print(f"New USD price range: ${dataset['new_usdprice'].min():.3f} to ${dataset['new_usdprice'].max():.3f}")
print(f"Date range: {dataset['year'].min()}-{dataset['year'].max()}")

In [ ]:
dataset.isnull().sum()

In [ ]:
# Drop the old usdprice column to avoid confusion
dataset = dataset.drop('usdprice', axis=1)
dataset = dataset.rename(columns={'new_usdprice': 'usd_price'})

In [ ]:
# Show first 5 rows
print(dataset.head())

# Show dataset shape (rows, columns)
print("\nDataset shape:", dataset.shape)

# Show column names
print("\nColumns:", dataset.columns.tolist())

# Show quick info
print("\nDataset info:")
print(dataset.info())

In [ ]:
# Clean text: remove spaces, commas, lowercase
dataset['priceflag'] = dataset['priceflag'].replace({'actual,aggregate': 'aggregate'})
# Check unique values again
print(dataset['priceflag'].unique())

In [ ]:
# Summary statistics for numerical columns
print("DATASET SUMMARY STATISTICS")
print("=" * 40)

# Get numerical columns only
numerical_cols = dataset.select_dtypes(include=['float64', 'int32', 'int64']).columns
print(f"Numerical columns: {list(numerical_cols)}\n")

# Display summary statistics in one-line per column
summary = dataset[numerical_cols].describe().T  # transpose
print(summary.round(2).to_string())

# Additional useful stats
print("\nADDITIONAL INSIGHTS:")
print("-" * 25)
print(f"Price range: {dataset['price'].min():,.0f} - {dataset['price'].max():,.0f} (local currency)")
print(f"USD price range: ${dataset['usd_price'].min():.3f} - ${dataset['usd_price'].max():.3f}")
print(f"Time span: {dataset['year'].min()} - {dataset['year'].max()} ({dataset['year'].max() - dataset['year'].min() + 1} years)")
print(f"Exchange rate range: {dataset['exchange_rate_unofficial'].min():,.0f} - {dataset['exchange_rate_unofficial'].max():,.0f}")
print(f"Geographic spread: {dataset['latitude'].min():.2f}°N - {dataset['latitude'].max():.2f}°N")
print(f"                   {dataset['longitude'].min():.2f}°E - {dataset['longitude'].max():.2f}°E")

# Check Outliers

In [ ]:
# Check top values
print(dataset[['price', 'usd_price', 'exchange_rate_unofficial']].sort_values('price', ascending=False).head(10))

In [ ]:
# Plot Distributions: Local Price, USD Price, and Exchange Rate
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18,12))

# ---- Local Price ----
plt.subplot(3,2,1)
sns.histplot(dataset['price'], bins=50, kde=True)
plt.title("Local Price Distribution")

plt.subplot(3,2,2)
sns.boxplot(x=dataset['price'])
plt.title("Local Price Boxplot")

# ---- USD Price ----
plt.subplot(3,2,3)
sns.histplot(dataset['usd_price'], bins=50, kde=True)
plt.title("USD Price Distribution")

plt.subplot(3,2,4)
sns.boxplot(x=dataset['usd_price'])
plt.title("USD Price Boxplot")

# ---- Exchange Rate (Unofficial) ----
plt.subplot(3,2,5)
sns.histplot(dataset['exchange_rate_unofficial'], bins=50, kde=True)
plt.title("Exchange Rate Distribution")

plt.subplot(3,2,6)
sns.boxplot(x=dataset['exchange_rate_unofficial'])
plt.title("Exchange Rate Boxplot")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for USD Price Distribution by Commodity
import seaborn as sns
import matplotlib.pyplot as plt

# Create facet grid of boxplots (one per commodity)
g = sns.catplot(
    data=dataset,
    x='commodity',
    y='usd_price',
    kind='box',
    col='commodity',
    col_wrap=4,    # Wrap every 4 plots in a row
    sharey=False,  # Allow each plot to have its own y-axis scale
    height=4       # Height of each subplot
)

# Set individual subplot titles (commodity names)
g.set_titles("{col_name}")

# Rotate x-tick labels for readability
g.set_xticklabels(rotation=45)

# Add one overall title for the entire grid
plt.subplots_adjust(top=0.92)  # adjust spacing for the main title
g.fig.suptitle("📦 USD Price Distribution by Commodity (Boxplots)", fontsize=16)

plt.show()

In [ ]:
# 📊 Distribution of Categorical Variables
import matplotlib.pyplot as plt
import seaborn as sns

# Columns to analyze
categorical_cols = ['region', 'market', 'category', 'commodity', 'priceflag', 'unit']

# Create grid (2 rows × 3 columns for 5 plots)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    # For high-cardinality columns, show only top 20 values
    top_values = dataset[col].value_counts().head(20).index
    data_to_plot = dataset[dataset[col].isin(top_values)]

    sns.countplot(
        y=data_to_plot[col],
        order=data_to_plot[col].value_counts().index,
        ax=axes[i]
    )

    axes[i].set_title(
        f'Distribution of {col.capitalize()} (Top {min(20, len(dataset[col].unique()))})'
        if len(dataset[col].unique()) > 20
        else f'Distribution of {col.capitalize()}',
        fontsize=12
    )
    axes[i].set_xlabel("Count")
    axes[i].set_ylabel(col.capitalize())

# Remove empty subplot if cols < grid size
for j in range(len(categorical_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Temporal Trends – Average Price Over Time
import matplotlib.pyplot as plt
import seaborn as sns

dataset['date'] = pd.to_datetime(dataset['date'])

price_trend = dataset.groupby('date')['usd_price'].mean().reset_index()

plt.figure(figsize=(14,6))
sns.lineplot(data=price_trend, x='date', y='usd_price')
plt.title("Average USD Price Over Time")
plt.xlabel("Date")
plt.ylabel("Avg USD Price")
plt.tight_layout()
plt.show()


In [ ]:
# Seasonal Pattern: Monthly Price Trend
dataset['month'] = dataset['date'].dt.month
monthly_avg = dataset.groupby('month')['usd_price'].mean()

plt.figure(figsize=(10,5))
sns.lineplot(x=monthly_avg.index, y=monthly_avg.values)
plt.title("Monthly Average USD Price")
plt.xlabel("Month")
plt.ylabel("Avg USD Price")
plt.xticks(range(1,13))
plt.show()


In [ ]:
# Time Series per Region (Line Plot)
plt.figure(figsize=(16, 6))
for region in dataset['region'].unique():
    subset = dataset[dataset['region'] == region]
    subset = subset.groupby('date')['usd_price'].mean()
    plt.plot(subset.index, subset.values, label=region)

plt.legend()
plt.title("USD Price Trend by Region Over Time")
plt.ylabel("USD Price")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Price Trend for Specific Market (e.g., Bakaara)
# Choose a specific market
market_name = 'Bakaara'
subset = dataset[dataset['market'] == market_name]

plt.figure(figsize=(14, 6))
subset.groupby('date')['usd_price'].mean().plot()
plt.title(f'📍 USD Price Trend in {market_name}')
plt.xlabel('Date')
plt.ylabel('Average USD Price')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# Time Series Decomposition (Trend, Seasonality, Residual)
# ======================================================

from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# 1. Prepare the monthly average USD price
monthly_avg = dataset.copy()
monthly_avg['date'] = pd.to_datetime(monthly_avg['date'])
monthly_avg = monthly_avg.set_index('date')
monthly_avg = monthly_avg['usd_price'].resample('M').mean()

# Fill missing values using linear interpolation
monthly_avg = monthly_avg.interpolate(method='linear')

# 2. Decompose the time series
# Check if there are still NaNs after interpolation
if monthly_avg.isnull().sum() > 0:
    print("Warning: Missing values still present after interpolation. Filling remaining NaNs with mean.")
    monthly_avg = monthly_avg.fillna(monthly_avg.mean())


decomposition = seasonal_decompose(monthly_avg, model='additive', period=12)

# 3. Plot the components
plt.figure(figsize=(14, 8))
decomposition.plot()
plt.suptitle('USD Price Decomposition: Trend, Seasonality, Residuals', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Analysis
import seaborn as sns
import matplotlib.pyplot as plt
corr = dataset[['price', 'usd_price', 'exchange_rate_unofficial']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")

In [ ]:
# Drop 'price' since it's highly correlated with 'usd_price'
dataset = dataset.drop(columns=['price', 'currency'])

# Define target variable
target = 'usd_price'
features = [col for col in dataset.columns if col != target]

print("Target variable:", target)
print("Feature variables:", features)

In [ ]:
# Check if outliers are from specific periods/locations
extreme_cases = dataset[dataset['usd_price'] > 100]
print(extreme_cases[['date', 'market', 'commodity']])

In [ ]:
# Quick check\

import numpy as np
from scipy import stats

print("TRANSFORMATION DECISION HELPER")
print("="*35)

# 1. Check for zeros (log transform issue)
zero_prices = (dataset['usd_price'] == 0).sum()
print(f"Zero prices: {zero_prices} (if >0, log transform needs adjustment)")

# 2. Measure skewness
skewness = stats.skew(dataset['usd_price'])
print(f"Skewness: {skewness:.2f}")
print(f"Interpretation: {abs(skewness):.1f} {'(highly skewed - consider log)' if abs(skewness) > 1 else '(mildly skewed - optional)' if abs(skewness) > 0.5 else '(not skewed - no transform needed)'}")

# 3. Check price range spread
price_std = dataset['usd_price'].std()
price_mean = dataset['usd_price'].mean()
cv = price_std / price_mean
print(f"Coefficient of variation: {cv:.2f}")
print(f"Interpretation: {'High variation - log transform recommended' if cv > 1 else 'Moderate variation - log optional' if cv > 0.5 else 'Low variation - no transform needed'}")

# 4. Visual test: Would log help?
min_price = dataset['usd_price'].min()
max_price = dataset['usd_price'].max()
ratio = max_price / max(min_price, 0.001)  # Avoid division by zero
print(f"Price ratio (max/min): {ratio:.1f}x")
print(f"Interpretation: {'Log recommended' if ratio > 100 else 'Log optional' if ratio > 10 else 'Linear fine'}")

print(f"\nRECOMMENDATION:")
if abs(skewness) > 1 or cv > 1 or ratio > 100:
    print("✅ USE LOG TRANSFORM")
    print("Your data shows high skewness/variation")
    print("Code: dataset['log_usd_price'] = np.log1p(dataset['usd_price'])")
else:
    print("⚪ LOG TRANSFORM OPTIONAL")
    print("Try both and see which gives better model performance")

print(f"\nOTHER TRANSFORMS TO CONSIDER:")
print("- Standardization: For features with different scales")
print("- Date features: Extract year, month, seasonality")
print("- Categorical encoding: For region, commodity, etc.")

# Corrected Outlier handling

In [ ]:
# 📌 Outlier Treatment + Log Transform
import numpy as np

print("OUTLIER TREATMENT + LOG TRANSFORM")
print("=" * 40)

# Step 0: Snapshot of current data
n_records = len(dataset)
usd_min, usd_max = dataset['usd_price'].min(), dataset['usd_price'].max()
print(f"Before treatment: {n_records:,} records")
print(f"USD Price range: ${usd_min:.2f} – ${usd_max:.2f}")

# Step 1: Outlier treatment (keep all outliers – they reflect real market/crisis prices)
dataset_clean = dataset.copy()
print(f"Outlier handling: No records removed (all retained)")

# Step 2: Log transform to reduce skewness
dataset_clean['log_usd_price'] = np.log1p(dataset_clean['usd_price'])

# Step 3: Post-treatment summary
print(f"\nAfter treatment: {len(dataset_clean):,} records")
print(f"Records removed: 0")
print(f"Original skewness: {dataset['usd_price'].skew():.2f}")
print(f"Log-transformed skewness: {dataset_clean['log_usd_price'].skew():.2f}")

print("\n✅ Completed: Outlier treatment + Log transform")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# Distribution of Prices
# ==============================

plt.figure(figsize=(14,6))

# Original distribution
plt.subplot(1, 2, 1)
sns.histplot(dataset['usd_price'], bins=50, kde=True, color='#e75480')
plt.title("Original USD Price Distribution")
plt.xlabel("USD Price")
plt.ylabel("Frequency")

# Log-transformed distribution
plt.subplot(1, 2, 2)
sns.histplot(dataset_clean['log_usd_price'], bins=50, kde=True, color='#4682B4')
plt.title("Log-Transformed USD Price Distribution")
plt.xlabel("Log(USD Price)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

# Feature Engineering

In [ ]:
# ==============================================================
# 📌 STEP 1: TEMPORAL TRAIN-TEST SPLIT
# ==============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")

def create_temporal_split(df, split_ratio=0.8):
    """Split data chronologically to prevent data leakage"""
    df_sorted = df.sort_values("date").reset_index(drop=True)
    split_idx = int(len(df_sorted) * split_ratio)

    train_df = df_sorted.iloc[:split_idx].copy()
    test_df = df_sorted.iloc[split_idx:].copy()

    print("TEMPORAL TRAIN-TEST SPLIT")
    print("=" * 30)
    print(f"Train period: {train_df['date'].min()} → {train_df['date'].max()}")
    print(f"Test period:  {test_df['date'].min()} → {test_df['date'].max()}")
    print(f"Train size: {len(train_df):,}, Test size: {len(test_df):,}\n")

    return train_df, test_df

# Apply temporal split
train_raw, test_raw = create_temporal_split(dataset_clean, split_ratio=0.8)


In [ ]:
# ==============================================================
# 📌 STEP 2: FEATURE ENGINEERING (RESPECT TEMPORAL ORDER)
# ==============================================================

def create_features(df, is_training=True, reference_stats=None):
    """Create features while respecting temporal ordering"""
    df_processed = df.copy()
    df_processed = df_processed.sort_values(["commodity", "market", "date"])

    print(f"Creating features for {'training' if is_training else 'test'} data...")

    # --- 1. Temporal features ---
    df_processed["year"] = df_processed["date"].dt.year
    df_processed["month"] = df_processed["date"].dt.month
    df_processed["quarter"] = df_processed["date"].dt.quarter
    df_processed["day_of_year"] = df_processed["date"].dt.dayofyear

    # --- 2. Cyclical encoding ---
    df_processed["month_sin"] = np.sin(2 * np.pi * df_processed["month"] / 12)
    df_processed["month_cos"] = np.cos(2 * np.pi * df_processed["month"] / 12)
    df_processed["day_sin"] = np.sin(2 * np.pi * df_processed["day_of_year"] / 365)
    df_processed["day_cos"] = np.cos(2 * np.pi * df_processed["day_of_year"] / 365)

    # --- 3. Season mapping (Somalia context) ---
    def assign_season(month):
        if month in [4, 5, 6]:
            return "Gu"       # Main rainy season
        elif month in [7, 8, 9]:
            return "Xagaa"    # Dry season
        elif month in [10, 11, 12]:
            return "Deyr"     # Short rains
        else:
            return "Jilaal"   # Dry season

    df_processed["season"] = df_processed["month"].apply(assign_season)

    return df_processed

In [ ]:
# ===================================
# Show Engineered Features with Meaning
# ===================================

feature_descriptions = {
    "year": "Year extracted from date",
    "month": "Month extracted from date (1–12)",
    "quarter": "Quarter of the year (1–4)",
    "day_of_year": "Day number in the year (1–365)",

    "month_sin": "Sine transformation of month (cyclical encoding)",
    "month_cos": "Cosine transformation of month (cyclical encoding)",
    "day_sin": "Sine transformation of day_of_year (cyclical encoding)",
    "day_cos": "Cosine transformation of day_of_year (cyclical encoding)",

    "season": "Somalia-specific season mapping (Gu, Xagaa, Deyr, Jilaal)"
}

print("📌 Engineered Features Summary")
print("="*60)
for feature, desc in feature_descriptions.items():
    print(f"{feature:12} → {desc}")

In [ ]:
# Add lag, rolling & change features

def add_time_series_features(df, is_training=True):
    """Add lag, rolling statistics, and price change features"""
    df_processed = df.copy()

    # --- 4. Lag features ---
    lags = [1, 3, 6, 12]  # months
    for lag in lags:
        df_processed[f"usd_price_lag_{lag}"] = (
            df_processed.groupby(["commodity", "market"])["usd_price"].shift(lag)
        )
        df_processed[f"exchange_rate_lag_{lag}"] = (
            df_processed.groupby(["commodity", "market"])["exchange_rate_unofficial"].shift(lag)
        )

    # --- 5. Rolling features ---
    windows = [3, 6, 12]
    for window in windows:
        df_processed[f"price_rolling_mean_{window}"] = (
            df_processed.groupby(["commodity", "market"])["usd_price"]
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        df_processed[f"price_rolling_std_{window}"] = (
            df_processed.groupby(["commodity", "market"])["usd_price"]
            .transform(lambda x: x.rolling(window, min_periods=1).std())
        )
        df_processed[f"exchange_rolling_std_{window}"] = (
            df_processed.groupby(["commodity", "market"])["exchange_rate_unofficial"]
            .transform(lambda x: x.rolling(window, min_periods=1).std())
        )

    # --- 6. Price change features ---
    df_processed["price_pct_change_1m"] = (
        df_processed.groupby(["commodity", "market"])["usd_price"].pct_change(1)
    )
    df_processed["price_pct_change_3m"] = (
        df_processed.groupby(["commodity", "market"])["usd_price"].pct_change(3)
    )
    df_processed["price_pct_change_12m"] = (
        df_processed.groupby(["commodity", "market"])["usd_price"].pct_change(12)
    )

    return df_processed

In [ ]:
# ==============================================================
# 📌 Engineered Features Summary (Lag, Rolling, Change Features)
# ==============================================================

def summarize_time_series_features():
    summary = {
        "usd_price_lag_[1,3,6,12]": "Lagged USD prices for 1, 3, 6, and 12 months (captures past price memory)",
        "exchange_rate_lag_[1,3,6,12]": "Lagged unofficial exchange rate for 1, 3, 6, and 12 months",
        "price_rolling_mean_[3,6,12]": "Rolling mean of USD price over 3, 6, 12 months (trend smoothing)",
        "price_rolling_std_[3,6,12]": "Rolling standard deviation of USD price over 3, 6, 12 months (volatility)",
        "exchange_rolling_std_[3,6,12]": "Rolling standard deviation of exchange rate (volatility of FX)",
        "price_pct_change_[1m,3m,12m]": "Percentage change in USD price over 1, 3, and 12 months (momentum/returns)"
    }

    print("\n📌 Engineered Features Summary (Time-Series)\n" + "="*60)
    for feature, desc in summary.items():
        print(f"{feature:<30} → {desc}")

# Call the summary
summarize_time_series_features()

In [ ]:
# Crisis indicators, missing-value handling & execution
def finalize_features(df, is_training=True):
    """Add crisis indicators and handle missing values"""
    df_processed = df.copy()

    # --- 7. Crisis indicators ---
    df_processed["is_2008_crisis"] = (
        ((df_processed["year"] >= 2007) & (df_processed["year"] <= 2009)).astype(int)
    )
    df_processed["is_covid_period"] = (
        ((df_processed["year"] >= 2020) & (df_processed["year"] <= 2022)).astype(int)
    )

    # --- 8. Fill missing values in lag/rolling features ---
    lag_roll_cols = [
        col for col in df_processed.columns
        if any(x in col for x in ["lag_", "rolling_", "pct_change"])
    ]
    for col in lag_roll_cols:
        df_processed[col] = (
            df_processed.groupby(["commodity", "market"])[col].fillna(method="ffill")
        )
        df_processed[col] = df_processed[col].fillna(0)

    # Return stats only for training
    stats_dict = {"feature_cols": lag_roll_cols} if is_training else None

    return df_processed, stats_dict


# ==============================================================
# 📌 APPLY FEATURE ENGINEERING PIPELINE
# ==============================================================

print("FEATURE ENGINEERING")
print("=" * 30)

# Training set
train_features = create_features(train_raw, is_training=True)
train_features = add_time_series_features(train_features, is_training=True)
train_features, training_stats = finalize_features(train_features, is_training=True)

# Test set
test_features = create_features(test_raw, is_training=False)
test_features = add_time_series_features(test_features, is_training=False)
test_features, _ = finalize_features(test_features, is_training=False)

print("\n✅ Feature engineering completed!")
print(f"Train shape: {train_features.shape}")
print(f"Test shape: {test_features.shape}")
print(
    f"New features created: {len([col for col in train_features.columns if col not in dataset_clean.columns])}"
)

In [ ]:
# ==============================================================
# 📌 Engineered Features Summary (Crisis Indicators & Missing Values)
# ==============================================================

def summarize_crisis_features():
    summary = {
        "is_2008_crisis": "Indicator for the 2008 global food price crisis (2007–2009)",
        "is_covid_period": "Indicator for the COVID-19 pandemic period (2020–2022)",
        "lag_roll_cols (NaN handling)": "All lag, rolling, and % change features filled using forward-fill per commodity/market; remaining missing values replaced with 0"
    }

    print("\n📌 Engineered Features Summary (Crisis & Missing Value Handling)\n" + "="*70)
    for feature, desc in summary.items():
        print(f"{feature:<25} → {desc}")

    print("\n✅ Final feature engineering pipeline applied to both training and test sets")

# Call the summary
summarize_crisis_features()

In [ ]:
# View all features before encoding

print("CURRENT FEATURES SUMMARY")
print("="*30)

print(f"Train dataset shape: {train_features.shape}")
print(f"Total columns: {len(train_features.columns)}")

# Separate by data type
numerical_cols = train_features.select_dtypes(include=['float64', 'int64', 'int32']).columns
categorical_cols = train_features.select_dtypes(include=['object', 'category']).columns
datetime_cols = train_features.select_dtypes(include=['datetime64']).columns

print(f"\nNUMERICAL FEATURES ({len(numerical_cols)}):")
print(list(numerical_cols))

print(f"\nCATEGORICAL FEATURES ({len(categorical_cols)}):")
print(list(categorical_cols))

print(f"\nDATETIME FEATURES ({len(datetime_cols)}):")
print(list(datetime_cols))

# Show sample of new engineered features
new_features = [col for col in train_features.columns if col not in dataset_clean.columns]
print(f"\nNEW ENGINEERED FEATURES ({len(new_features)}):")
print(new_features[:10] if len(new_features) > 10 else new_features)

In [ ]:
# Categorical encoding - fit on train, transform on test

from sklearn.preprocessing import LabelEncoder
import pandas as pd

print("CATEGORICAL ENCODING")
print("="*20)

def encode_features(train_df, test_df):
    train_enc = train_df.copy()
    test_enc = test_df.copy()
    encoders = {}

    # Label encode high cardinality categorical features
    label_cols = ['region', 'district', 'market', 'category', 'commodity']
    for col in label_cols:
        if col in train_df.columns:
            le = LabelEncoder()
            # Fit on training only
            train_enc[f'{col}_encoded'] = le.fit_transform(train_enc[col].astype(str))
            encoders[col] = le

            # Transform test, handle unseen categories
            test_categories = test_enc[col].astype(str)
            seen_categories = set(le.classes_)
            test_enc[f'{col}_encoded'] = [
                le.transform([cat])[0] if cat in seen_categories else 0
                for cat in test_categories
            ]

    # One-hot encode low cardinality features
    onehot_cols = ['unit', 'priceflag', 'season', 'pricetype']
    for col in onehot_cols:
        if col in train_df.columns:
            # Get unique categories from training only
            categories = train_enc[col].unique()
            for cat in categories:
                col_name = f'{col}_{cat}'
                train_enc[col_name] = (train_enc[col] == cat).astype(int)
                test_enc[col_name] = (test_enc[col] == cat).astype(int)

    return train_enc, test_enc, encoders

# Apply encoding
train_encoded, test_encoded, fitted_encoders = encode_features(train_features, test_features)

# Prepare final datasets
drop_cols = ['region', 'district', 'market', 'category', 'commodity',
             'unit', 'priceflag', 'season', 'pricetype', 'date']

# Use log_usd_price as target (your volatility prediction target)
target_col = 'log_usd_price'

X_train = train_encoded.drop(columns=drop_cols + [target_col, 'usd_price'])
y_train = train_encoded[target_col]
X_test = test_encoded.drop(columns=drop_cols + [target_col, 'usd_price'])
y_test = test_encoded[target_col]

print(f"Encoding completed!")
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Target: {target_col} (log-transformed for volatility)")

In [ ]:
# ==============================================================
# 📌 VALIDATION: Categorical Encoding Summary
# ==============================================================

def validate_encoding(train_encoded, test_encoded, drop_cols, target_col):
    print("\n📌 VALIDATION OF CATEGORICAL ENCODING")
    print("=" * 60)

    # 1. Show which encoded columns exist
    encoded_cols = [col for col in train_encoded.columns if col not in drop_cols + [target_col, 'usd_price']]
    print(f"Total encoded feature columns: {len(encoded_cols)}\n")
    print("Sample of encoded columns:")
    print(encoded_cols[:15])  # show first 15 for brevity

    # 2. Confirm that original categorical columns were dropped
    print("\nDropped original categorical columns:")
    print(drop_cols)

    # 3. Check dataset shapes
    print(f"\nTrain dataset shape: {train_encoded.shape}")
    print(f"Test dataset shape:  {test_encoded.shape}")

    # 4. Show sample rows to confirm encoding
    print("\nSample encoded training data (first 3 rows):")
    print(train_encoded[encoded_cols].head(3))

# Run validation
validate_encoding(train_encoded, test_encoded, drop_cols, target_col)

In [ ]:
# Standardization - fit on train, transform on test

from sklearn.preprocessing import StandardScaler

print("STANDARDIZATION")
print("="*15)

# Identify numerical columns only (exclude binary encoded features)
numerical_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
binary_cols = [col for col in X_train.columns if X_train[col].nunique() <= 2]
cols_to_scale = [col for col in numerical_cols if col not in binary_cols]

print(f"Columns to standardize: {len(cols_to_scale)}")

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

if len(cols_to_scale) > 0:
    X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])
    print("Standardization completed")
else:
    print("No columns to standardize")

print(f"Final train shape: {X_train_scaled.shape}")
print(f"Final test shape: {X_test_scaled.shape}")
print("Ready for modeling!")

In [ ]:
# Check train and test periods

print("TRAIN/TEST PERIOD CHECK")
print("="*25)

# Get original data with dates (before we dropped 'date' column)
train_dates = train_features['date']
test_dates = test_features['date']

print(f"Train period: {train_dates.min()} to {train_dates.max()}")
print(f"Test period: {test_dates.min()} to {test_dates.max()}")
print(f"Train records: {len(train_dates):,}")
print(f"Test records: {len(test_dates):,}")

# Feature Selection

In [ ]:
# ===============================
# Correlation Matrix Analysis
# Somalia Food Price Forecasting
# ===============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("CORRELATION MATRIX ANALYSIS - SOMALIA FOOD PRICE FORECASTING")
print("=" * 70)

# ===============================
# 1. Data Preparation
# ===============================
def prepare_correlation_data():
    """Prepare data for correlation analysis"""
    print("\nStep 1: Preparing data for correlation analysis...")

    # Combine features and target
    # Assuming you have X_train_scaled and y_train available
    try:
        # Create full dataset for correlation
        correlation_data = X_train_scaled.copy()
        correlation_data['target_usd_price'] = y_train.values

        print(f"Dataset shape: {correlation_data.shape}")
        print(f"Features: {correlation_data.shape[1]-1}")
        print(f"Target: log_usd_price")

        return correlation_data

    except Exception as e:
        print(f"Error preparing data: {e}")
        return None

# ===============================
# 2. Basic Correlation Matrix
# ===============================
def create_correlation_matrix(data, method='pearson'):
    """Create and display correlation matrix"""
    print(f"\nStep 2: Computing {method} correlation matrix...")

    # Calculate correlation matrix
    if method == 'pearson':
        corr_matrix = data.corr()
    elif method == 'spearman':
        corr_matrix = data.corr(method='spearman')

    return corr_matrix

# ===============================
# 3. Target Variable Correlations
# ===============================
def analyze_target_correlations(corr_matrix, top_n=20):
    """Analyze correlations with target variable"""
    print(f"\nStep 3: Analyzing target variable correlations (Top {top_n})...")

    # Get correlations with target
    target_corr = corr_matrix['target_usd_price'].drop('target_usd_price')
    target_corr_abs = target_corr.abs().sort_values(ascending=False)

    # Display top correlations
    print(f"\n📊 TOP {top_n} FEATURES CORRELATED WITH USD PRICE:")
    print("-" * 65)
    print(f"{'Feature':<30} {'Correlation':<12} {'Abs Value':<12}")
    print("-" * 65)

    for i, (feature, corr_val) in enumerate(target_corr_abs.head(top_n).items(), 1):
        original_corr = target_corr[feature]
        print(f"{i:2d}. {feature:<27} {original_corr:>8.4f} {corr_val:>11.4f}")

    return target_corr_abs

# ===============================
# 4. High Correlation Pairs
# ===============================
def find_multicollinearity(corr_matrix, threshold=0.8):
    """Find highly correlated feature pairs (multicollinearity)"""
    print(f"\nStep 4: Identifying multicollinearity (|r| > {threshold})...")

    # Get upper triangle of correlation matrix
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    # Find high correlation pairs
    high_corr_pairs = []
    for col in upper_tri.columns:
        for idx in upper_tri.index:
            if abs(upper_tri.loc[idx, col]) > threshold:
                if not pd.isna(upper_tri.loc[idx, col]):
                    high_corr_pairs.append({
                        'Feature_1': idx,
                        'Feature_2': col,
                        'Correlation': upper_tri.loc[idx, col]
                    })

    if high_corr_pairs:
        print(f"\n⚠️  MULTICOLLINEARITY DETECTED ({len(high_corr_pairs)} pairs):")
        print("-" * 80)
        print(f"{'Feature 1':<25} {'Feature 2':<25} {'Correlation':<12}")
        print("-" * 80)

        # Sort by absolute correlation
        high_corr_pairs.sort(key=lambda x: abs(x['Correlation']), reverse=True)

        for pair in high_corr_pairs[:15]:  # Show top 15
            print(f"{pair['Feature_1']:<25} {pair['Feature_2']:<25} {pair['Correlation']:>8.4f}")
    else:
        print(f"✅ No multicollinearity issues detected (threshold: {threshold})")

    return high_corr_pairs

# ===============================
# 5. Correlation Visualization
# ===============================
def visualize_correlations(corr_matrix, target_corr, figsize=(20, 15)):
    """Create correlation visualizations"""
    print("\nStep 5: Creating correlation visualizations...")

    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('Somalia Food Price Forecasting - Correlation Analysis', fontsize=16, fontweight='bold')

    # 1. Full correlation heatmap (sample of features if too many)
    ax1 = axes[0, 0]
    if corr_matrix.shape[0] > 50:
        # Sample features for readability
        sample_features = list(target_corr.abs().sort_values(ascending=False).head(30).index) + ['target_usd_price']
        corr_sample = corr_matrix.loc[sample_features, sample_features]
    else:
        corr_sample = corr_matrix

    sns.heatmap(corr_sample, annot=False, cmap='RdBu_r', center=0,
                ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title('Feature Correlation Matrix\n(Top 30 features)', fontweight='bold')
    ax1.tick_params(axis='both', labelsize=8)

    # 2. Target correlations bar plot
    ax2 = axes[0, 1]
    top_features = target_corr.abs().sort_values(ascending=False).head(15)
    colors = ['red' if target_corr[feat] < 0 else 'green' for feat in top_features.index]

    bars = ax2.barh(range(len(top_features)), [target_corr[feat] for feat in top_features.index], color=colors, alpha=0.7)
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels([feat[:20] + '...' if len(feat) > 20 else feat for feat in top_features.index], fontsize=8)
    ax2.set_xlabel('Correlation with USD Price')
    ax2.set_title('Top 15 Features - Target Correlation', fontweight='bold')
    ax2.axvline(x=0, color='black', linestyle='-', alpha=0.5)
    ax2.grid(True, alpha=0.3)

    # 3. Correlation distribution
    ax3 = axes[1, 0]
    target_corr_values = target_corr.values
    ax3.hist(target_corr_values, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    ax3.axvline(target_corr_values.mean(), color='red', linestyle='--',
                label=f'Mean: {target_corr_values.mean():.3f}')
    ax3.set_xlabel('Correlation Coefficient')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Distribution of Feature-Target Correlations', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Feature importance based on correlation
    ax4 = axes[1, 1]
    correlation_importance = target_corr.abs().sort_values(ascending=True).tail(15)
    bars = ax4.barh(range(len(correlation_importance)), correlation_importance.values, color='orange', alpha=0.7)
    ax4.set_yticks(range(len(correlation_importance)))
    ax4.set_yticklabels([feat[:20] + '...' if len(feat) > 20 else feat for feat in correlation_importance.index], fontsize=8)
    ax4.set_xlabel('Absolute Correlation')
    ax4.set_title('Feature Importance by Correlation Strength', fontweight='bold')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# ===============================
# 6. Feature Recommendations
# ===============================
def recommend_features(target_corr, high_corr_pairs, min_correlation=0.1):
    """Provide feature selection recommendations"""
    print(f"\nStep 6: Feature selection recommendations...")

    # Strong predictors
    strong_predictors = target_corr.abs()[target_corr.abs() > 0.3].sort_values(ascending=False)
    moderate_predictors = target_corr.abs()[(target_corr.abs() > min_correlation) &
                                            (target_corr.abs() <= 0.3)].sort_values(ascending=False)
    weak_predictors = target_corr.abs()[target_corr.abs() <= min_correlation]

    print(f"\n🔍 FEATURE ANALYSIS SUMMARY:")
    print("-" * 50)
    print(f"Strong predictors (|r| > 0.3): {len(strong_predictors)}")
    print(f"Moderate predictors (0.1 < |r| ≤ 0.3): {len(moderate_predictors)}")
    print(f"Weak predictors (|r| ≤ 0.1): {len(weak_predictors)}")

    print(f"\n✅ RECOMMENDED FEATURES FOR MODELING:")
    print("-" * 50)
    if len(strong_predictors) > 0:
        print("High Priority Features:")
        for i, (feat, corr) in enumerate(strong_predictors.head(10).items(), 1):
            print(f"  {i:2d}. {feat} (r = {target_corr[feat]:.4f})")

    if len(moderate_predictors) > 0:
        print(f"\nModerate Priority Features (consider for ensemble):")
        for i, (feat, corr) in enumerate(moderate_predictors.head(5).items(), 1):
            print(f"  {i:2d}. {feat} (r = {target_corr[feat]:.4f})")

    print(f"\n❌ FEATURES TO CONSIDER REMOVING:")
    print(f"   • {len(weak_predictors)} features with |r| ≤ {min_correlation}")
    if len(high_corr_pairs) > 0:
        print(f"   • Consider removing one feature from {len(high_corr_pairs)} highly correlated pairs")

    return {
        'strong': strong_predictors,
        'moderate': moderate_predictors,
        'weak': weak_predictors
    }

# ===============================
# 7. Main Execution Function
# ===============================
def run_correlation_analysis():
    """Run complete correlation analysis"""
    try:
        # Prepare data
        data = prepare_correlation_data()
        if data is None:
            print("❌ Could not prepare data for correlation analysis")
            return

        # Create correlation matrix
        corr_matrix = create_correlation_matrix(data, method='pearson')

        # Analyze target correlations
        target_corr = analyze_target_correlations(corr_matrix, top_n=20)

        # Find multicollinearity
        high_corr_pairs = find_multicollinearity(corr_matrix, threshold=0.8)

        # Create visualizations
        visualize_correlations(corr_matrix, corr_matrix['target_usd_price'].drop('target_usd_price'))

        # Feature recommendations
        feature_recommendations = recommend_features(
            corr_matrix['target_usd_price'].drop('target_usd_price'),
            high_corr_pairs
        )

        print(f"\n{'='*70}")
        print("CORRELATION ANALYSIS COMPLETED")
        print(f"✅ Matrix computed: {corr_matrix.shape[0]} x {corr_matrix.shape[1]}")
        print(f"✅ Multicollinearity pairs: {len(high_corr_pairs)}")
        print(f"✅ Strong predictors: {len(feature_recommendations['strong'])}")
        print(f"{'='*70}")

        return {
            'correlation_matrix': corr_matrix,
            'target_correlations': target_corr,
            'multicollinearity': high_corr_pairs,
            'recommendations': feature_recommendations
        }

    except Exception as e:
        print(f"❌ Error in correlation analysis: {e}")
        return None

# Execute analysis
if __name__ == "__main__":
    correlation_results = run_correlation_analysis()

In [ ]:
# ===============================
# Feature Selection for Food Price Forecasting
# ===============================
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print("=" * 60)
print("FEATURE SELECTION - SOMALIA FOOD PRICE FORECASTING")
print("=" * 60)

def remove_multicollinearity(X_train, threshold=0.95):
    """Remove highly correlated features"""
    print(f"\nStep 1: Removing multicollinearity (threshold: {threshold})")
    print("-" * 50)

    # Calculate correlation matrix
    corr_matrix = X_train.corr().abs()

    # Find highly correlated pairs
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find features to drop
    features_to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

    print(f"Original features: {X_train.shape[1]}")
    print(f"Features to remove: {len(features_to_drop)}")

    if features_to_drop:
        print("Removing features:")
        for i, feat in enumerate(features_to_drop[:10], 1):  # Show first 10
            print(f"  {i:2d}. {feat}")
        if len(features_to_drop) > 10:
            print(f"  ... and {len(features_to_drop)-10} more")

    # Remove correlated features
    X_clean = X_train.drop(columns=features_to_drop)
    print(f"Features after multicollinearity removal: {X_clean.shape[1]}")

    return X_clean, features_to_drop

def manual_feature_removal(X_train):
    """Remove specific problematic features based on correlation analysis"""
    print(f"\nStep 2: Manual removal of problematic features")
    print("-" * 50)

    # Features to remove based on your correlation analysis
    features_to_remove = [
        # Perfect correlations (-1.0 or 1.0)
        'priceflag_aggregate',  # Keep priceflag_actual
        'day_of_year',          # Keep month

        # Highly redundant price features (keep only the best ones)
        'price_rolling_mean_6',  # Keep price_rolling_mean_3
        'price_rolling_mean_12', # Keep price_rolling_mean_3
        'usd_price_lag_1',       # Too similar to rolling mean
        'usd_price_lag_6',       # Keep usd_price_lag_3
        'usd_price_lag_12',      # Keep usd_price_lag_3
    ]

    # Remove features that exist in the dataset
    existing_features_to_remove = [f for f in features_to_remove if f in X_train.columns]

    print(f"Manually removing {len(existing_features_to_remove)} features:")
    for i, feat in enumerate(existing_features_to_remove, 1):
        print(f"  {i:2d}. {feat}")

    X_manual = X_train.drop(columns=existing_features_to_remove)
    print(f"Features after manual removal: {X_manual.shape[1]}")

    return X_manual, existing_features_to_remove

def statistical_feature_selection(X_train, y_train, k=20):
    """Use statistical methods for feature selection"""
    print(f"\nStep 3: Statistical feature selection (top {k} features)")
    print("-" * 50)

    # Method 1: F-test (linear relationships)
    f_selector = SelectKBest(score_func=f_regression, k=k)
    X_f_selected = f_selector.fit_transform(X_train, y_train)
    f_feature_names = X_train.columns[f_selector.get_support()].tolist()
    f_scores = f_selector.scores_[f_selector.get_support()]

    print("Top features by F-test:")
    f_results = list(zip(f_feature_names, f_scores))
    f_results.sort(key=lambda x: x[1], reverse=True)
    for i, (feat, score) in enumerate(f_results[:10], 1):
        print(f"  {i:2d}. {feat:<25} (F-score: {score:.2f})")

    # Method 2: Mutual Information (non-linear relationships)
    mi_selector = SelectKBest(score_func=mutual_info_regression, k=k)
    X_mi_selected = mi_selector.fit_transform(X_train, y_train)
    mi_feature_names = X_train.columns[mi_selector.get_support()].tolist()
    mi_scores = mi_selector.scores_[mi_selector.get_support()]

    print(f"\nTop features by Mutual Information:")
    mi_results = list(zip(mi_feature_names, mi_scores))
    mi_results.sort(key=lambda x: x[1], reverse=True)
    for i, (feat, score) in enumerate(mi_results[:10], 1):
        print(f"  {i:2d}. {feat:<25} (MI-score: {score:.3f})")

    # Combine both methods (intersection)
    common_features = list(set(f_feature_names) & set(mi_feature_names))
    print(f"\nFeatures selected by both methods ({len(common_features)}):")
    for i, feat in enumerate(sorted(common_features), 1):
        print(f"  {i:2d}. {feat}")

    return {
        'f_test': f_feature_names,
        'mutual_info': mi_feature_names,
        'common': common_features,
        'f_scores': dict(f_results),
        'mi_scores': dict(mi_results)
    }

def create_final_feature_set(X_train, selection_results):
    """Create final recommended feature set"""
    print(f"\nStep 4: Creating final feature set")
    print("-" * 50)

    # Strategy: Use common features + top from each method
    recommended_features = set(selection_results['common'])

    # Add top F-test features not already included
    f_top = selection_results['f_test'][:15]
    recommended_features.update(f_top)

    # Add top MI features not already included
    mi_top = selection_results['mutual_info'][:15]
    recommended_features.update(mi_top)

    # Convert to list and sort
    final_features = sorted(list(recommended_features))

    print(f"Final recommended features ({len(final_features)}):")
    for i, feat in enumerate(final_features, 1):
        f_score = selection_results['f_scores'].get(feat, 0)
        mi_score = selection_results['mi_scores'].get(feat, 0)
        print(f"  {i:2d}. {feat:<25} (F:{f_score:.1f}, MI:{mi_score:.3f})")

    return final_features

def visualize_feature_selection(selection_results):
    """Create visualizations for feature selection"""
    print(f"\nStep 5: Creating feature selection visualization")
    print("-" * 50)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # F-test scores
    f_features = list(selection_results['f_scores'].keys())[:15]
    f_scores = [selection_results['f_scores'][f] for f in f_features]

    axes[0].barh(range(len(f_features)), f_scores, color='skyblue', alpha=0.7)
    axes[0].set_yticks(range(len(f_features)))
    axes[0].set_yticklabels([f[:20] + '...' if len(f) > 20 else f for f in f_features], fontsize=8)
    axes[0].set_xlabel('F-test Score')
    axes[0].set_title('Top 15 Features - F-test Selection')
    axes[0].grid(True, alpha=0.3)

    # Mutual Information scores
    mi_features = list(selection_results['mi_scores'].keys())[:15]
    mi_scores = [selection_results['mi_scores'][f] for f in mi_features]

    axes[1].barh(range(len(mi_features)), mi_scores, color='lightcoral', alpha=0.7)
    axes[1].set_yticks(range(len(mi_features)))
    axes[1].set_yticklabels([f[:20] + '...' if len(f) > 20 else f for f in mi_features], fontsize=8)
    axes[1].set_xlabel('Mutual Information Score')
    axes[1].set_title('Top 15 Features - Mutual Information Selection')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def run_feature_selection():
    """Main function to run complete feature selection"""
    try:
        # Use your existing training data
        print("Loading training data...")

        # Step 1: Remove multicollinearity
        X_clean, dropped_corr = remove_multicollinearity(X_train_scaled, threshold=0.95)

        # Step 2: Manual feature removal
        X_manual, dropped_manual = manual_feature_removal(X_clean)

        # Step 3: Statistical selection
        selection_results = statistical_feature_selection(X_manual, y_train, k=20)

        # Step 4: Create final feature set
        final_features = create_final_feature_set(X_manual, selection_results)

        # Step 5: Visualize
        visualize_feature_selection(selection_results)

        # Create final datasets
        X_train_selected = X_train_scaled[final_features]
        X_test_selected = X_test_scaled[final_features]

        print(f"\n{'='*60}")
        print("FEATURE SELECTION SUMMARY")
        print("="*60)
        print(f"Original features: {X_train_scaled.shape[1]}")
        print(f"After multicollinearity removal: {X_clean.shape[1]}")
        print(f"After manual removal: {X_manual.shape[1]}")
        print(f"Final selected features: {len(final_features)}")
        print(f"Reduction: {((X_train_scaled.shape[1] - len(final_features)) / X_train_scaled.shape[1] * 100):.1f}%")

        print(f"\nFinal feature list:")
        for i, feat in enumerate(final_features, 1):
            print(f"  {i:2d}. {feat}")

        return {
            'X_train_selected': X_train_selected,
            'X_test_selected': X_test_selected,
            'selected_features': final_features,
            'dropped_features': dropped_corr + dropped_manual,
            'selection_results': selection_results
        }

    except Exception as e:
        print(f"Error in feature selection: {e}")
        return None

# Execute feature selection
if __name__ == "__main__":
    feature_selection_results = run_feature_selection()

    # Save the selected datasets for modeling
    if feature_selection_results:
        print(f"\nFeature selection completed successfully!")
        print(f"Use 'X_train_selected' and 'X_test_selected' for your models.")

        # Store results in variables for easy access
        X_train_selected = feature_selection_results['X_train_selected']
        X_test_selected = feature_selection_results['X_test_selected']
        selected_features = feature_selection_results['selected_features']

In [ ]:
# ===============================
# Variable Check After Feature Selection
# ===============================
import pandas as pd
import numpy as np

print("=" * 60)
print("VARIABLE CHECK - POST FEATURE SELECTION")
print("=" * 60)

def check_training_variables():
    """Check training dataset variables"""
    print("\n1. TRAINING DATA CHECK")
    print("-" * 40)

    try:
        print(f"X_train_selected shape: {X_train_selected.shape}")
        print(f"y_train shape: {y_train.shape}")
        print(f"Features: {X_train_selected.shape[1]}")
        print(f"Samples: {X_train_selected.shape[0]}")
        print(f"Target variable: log_usd_price")

        # Check for missing values
        missing_features = X_train_selected.isnull().sum()
        missing_count = missing_features[missing_features > 0]

        if len(missing_count) > 0:
            print(f"\n⚠️  Missing values found:")
            for feature, count in missing_count.items():
                print(f"  {feature}: {count} missing ({count/len(X_train_selected)*100:.1f}%)")
        else:
            print("✅ No missing values in training data")

        # Check target variable
        print(f"\nTarget variable (y_train) statistics:")
        print(f"  Mean: {y_train.mean():.4f}")
        print(f"  Std:  {y_train.std():.4f}")
        print(f"  Min:  {y_train.min():.4f}")
        print(f"  Max:  {y_train.max():.4f}")

    except Exception as e:
        print(f"❌ Error checking training variables: {e}")

def check_testing_variables():
    """Check testing dataset variables"""
    print("\n2. TESTING DATA CHECK")
    print("-" * 40)

    try:
        print(f"X_test_selected shape: {X_test_selected.shape}")
        print(f"y_test shape: {y_test.shape}")
        print(f"Features: {X_test_selected.shape[1]}")
        print(f"Samples: {X_test_selected.shape[0]}")

        # Check for missing values
        missing_features = X_test_selected.isnull().sum()
        missing_count = missing_features[missing_features > 0]

        if len(missing_count) > 0:
            print(f"\n⚠️  Missing values found:")
            for feature, count in missing_count.items():
                print(f"  {feature}: {count} missing ({count/len(X_test_selected)*100:.1f}%)")
        else:
            print("✅ No missing values in testing data")

        # Check target variable
        print(f"\nTarget variable (y_test) statistics:")
        print(f"  Mean: {y_test.mean():.4f}")
        print(f"  Std:  {y_test.std():.4f}")
        print(f"  Min:  {y_test.min():.4f}")
        print(f"  Max:  {y_test.max():.4f}")

    except Exception as e:
        print(f"❌ Error checking testing variables: {e}")

def check_feature_consistency():
    """Check consistency between training and testing features"""
    print("\n3. FEATURE CONSISTENCY CHECK")
    print("-" * 40)

    try:
        train_features = set(X_train_selected.columns)
        test_features = set(X_test_selected.columns)

        if train_features == test_features:
            print("✅ Training and testing features are identical")
            print(f"   Feature count: {len(train_features)}")
        else:
            print("⚠️  Feature mismatch detected:")

            train_only = train_features - test_features
            if train_only:
                print(f"  Features only in training: {train_only}")

            test_only = test_features - train_features
            if test_only:
                print(f"  Features only in testing: {test_only}")

        # Check feature order
        if list(X_train_selected.columns) == list(X_test_selected.columns):
            print("✅ Feature order is consistent")
        else:
            print("⚠️  Feature order differs between train/test")

    except Exception as e:
        print(f"❌ Error checking feature consistency: {e}")

def display_selected_features():
    """Display the selected features"""
    print("\n4. SELECTED FEATURES LIST")
    print("-" * 40)

    try:
        features = X_train_selected.columns.tolist()
        print(f"Total features: {len(features)}")
        print("\nFeature list:")

        for i, feature in enumerate(features, 1):
            print(f"  {i:2d}. {feature}")

    except Exception as e:
        print(f"❌ Error displaying features: {e}")

def check_data_types():
    """Check data types of features"""
    print("\n5. DATA TYPES CHECK")
    print("-" * 40)

    try:
        dtypes = X_train_selected.dtypes.value_counts()
        print("Data type distribution:")
        for dtype, count in dtypes.items():
            print(f"  {dtype}: {count} features")

        # Check for object types that might need encoding
        object_features = X_train_selected.select_dtypes(include=['object']).columns
        if len(object_features) > 0:
            print(f"\n⚠️  Object type features found (may need encoding):")
            for feature in object_features:
                print(f"  {feature}")
        else:
            print("\n✅ All features are numeric")

    except Exception as e:
        print(f"❌ Error checking data types: {e}")

def check_feature_statistics():
    """Display basic statistics for features"""
    print("\n6. FEATURE STATISTICS SUMMARY")
    print("-" * 40)

    try:
        # Get basic statistics
        stats = X_train_selected.describe()

        print("Key statistics (first 5 features):")
        print(stats.iloc[:, :5].round(4))

        # Check for features with zero variance
        zero_var_features = X_train_selected.columns[X_train_selected.var() == 0]
        if len(zero_var_features) > 0:
            print(f"\n⚠️  Zero variance features:")
            for feature in zero_var_features:
                print(f"  {feature}")
        else:
            print(f"\n✅ All features have non-zero variance")

    except Exception as e:
        print(f"❌ Error checking feature statistics: {e}")

def run_variable_check():
    """Run complete variable check"""

    check_training_variables()
    check_testing_variables()
    check_feature_consistency()
    display_selected_features()
    check_data_types()
    check_feature_statistics()

    print(f"\n{'='*60}")
    print("VARIABLE CHECK SUMMARY")
    print("="*60)

    try:
        print(f"✅ Training data: {X_train_selected.shape[0]:,} samples × {X_train_selected.shape[1]} features")
        print(f"✅ Testing data:  {X_test_selected.shape[0]:,} samples × {X_test_selected.shape[1]} features")
        print(f"✅ Target variable: Available for both train/test")
        print(f"✅ Ready for modeling with selected features")

        print(f"\nNext steps:")
        print(f"  1. Run models using X_train_selected and X_test_selected")
        print(f"  2. Compare performance with original results")

    except Exception as e:
        print(f"❌ Error in summary: {e}")

# Execute the check
if __name__ == "__main__":
    run_variable_check()

In [ ]:
# Rename selected datasets to original names for compatibility
X_train_scaled = X_train_selected.copy()
X_test_scaled = X_test_selected.copy()

print("Dataset names updated:")
print(f"X_train_scaled shape: {X_train_scaled.shape} (now using selected features)")
print(f"X_test_scaled shape: {X_test_scaled.shape} (now using selected features)")
print("You can now run your existing model code without changes")

# Random Forest Regressor (Baseline)

In [ ]:
# Random Forest for Food Price Volatility Forecasting

# ===============================
# Imports and Setup
# ===============================
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("RANDOM FOREST MODEL")
print("="*25)

# ===============================
# Data Validation
# ===============================
print("Data validation...")
print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Target: log_usd_price (volatility prediction)")

# Check for any remaining issues
if X_train_scaled.isnull().any().any():
    print("Warning: NaN values found, filling with 0")
    X_train_scaled = X_train_scaled.fillna(0)
    X_test_scaled = X_test_scaled.fillna(0)

# ===============================
# Model Training
# ===============================
print("\nTraining Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

In [ ]:
# ==============================
# Generate Predictions
# ==============================
y_train_pred_rf = rf_model.predict(X_train_scaled)
y_test_pred_rf = rf_model.predict(X_test_scaled)

# ===============================
# Evaluation Metrics Function and Calculation
# ===============================
def calculate_metrics(y_true, y_pred, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"Dataset": dataset_name, "RMSE": rmse, "MAE": mae, "R²": r2, "MAPE": mape}

# Calculate metrics
train_metrics = calculate_metrics(y_train, y_train_pred_rf, "Training")
test_metrics = calculate_metrics(y_test, y_test_pred_rf, "Testing")

# ===============================
# Print Results
# ===============================
print("\nRandom Forest Performance:")
print(f"Training R²:  {train_metrics['R²']:.4f}")
print(f"Test R²:      {test_metrics['R²']:.4f}")
print(f"Test RMSE:    {test_metrics['RMSE']:.4f}")
print(f"Test MAE:     {test_metrics['MAE']:.4f}")
print(f"Test MAPE:    {test_metrics['MAPE']:.2f}%")

# ===============================
# Feature Importance Analysis
# ===============================
feature_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 features for volatility prediction:")
print(feature_importance.head(10))

In [ ]:
# ===============================
# Visualizations
# ===============================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Line Plot: Actual vs Predicted (full test set)
axes[0,0].plot(y_test.values, label='Actual', alpha=0.8, color='blue')
axes[0,0].plot(y_test_pred_rf, label='Predicted (Random Forest)', alpha=0.8, color='green')
axes[0,0].set_title('Actual vs Predicted Prices (Random Forest - Baseline)')
axes[0,0].set_xlabel('Time Index')
axes[0,0].set_ylabel('USD Price')
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# 2. Scatter Plot: Actual vs Predicted
axes[0,1].scatter(y_test, y_test_pred_rf, alpha=0.5, color='green')
axes[0,1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0,1].set_title("Actual vs Predicted Scatter (Random Forest)")
axes[0,1].set_xlabel("Actual USD Price")
axes[0,1].set_ylabel("Predicted USD Price")
axes[0,1].grid(alpha=0.3)

# 3. Feature Importance (using X_train_scaled)
importances = rf_model.feature_importances_
features = X_train_scaled.columns  # FIXED: use scaled feature names
sorted_idx = np.argsort(importances)[::-1]

axes[1,0].barh(np.array(features)[sorted_idx], importances[sorted_idx], color='green')
axes[1,0].set_title("Feature Importance (Random Forest)")
axes[1,0].set_xlabel("Importance Score")
axes[1,0].invert_yaxis()

# 4. Time series plot (last 200 predictions)
n_plot = min(200, len(y_test))
x_axis = range(n_plot)

axes[1,1].plot(x_axis, y_test.iloc[-n_plot:], label='Actual', linewidth=2, color='blue')
axes[1,1].plot(x_axis, y_test_pred_rf[-n_plot:], label='Predicted (RF)', linewidth=2, color='red', alpha=0.8)
axes[1,1].set_title(f'Time Series: Last {n_plot} Predictions (Random Forest)')
axes[1,1].set_xlabel('Time Steps')
axes[1,1].set_ylabel('USD Price')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRandom Forest training completed!")

In [ ]:
# ===============================
# Model Analysis and Insights (Random Forest Baseline)
# ===============================

print("\n" + "="*60)
print("MODEL ANALYSIS AND INSIGHTS (RANDOM FOREST BASELINE)")
print("="*60)

# Model configuration
print(f"\nModel Configuration:")
print(f"  - Total features: {len(X_train.columns)}")
print(f"  - Trees in forest: {rf_model.n_estimators}")
print(f"  - Max depth: {rf_model.max_depth if rf_model.max_depth is not None else 'Not set'}")
print(f"  - Training samples: {len(X_train):,}")
print(f"  - Testing samples: {len(X_test):,}")

# Performance interpretation
print(f"\nPerformance Assessment:")
if test_metrics['R²'] > 0.8:
    print("  ✅ Excellent performance - Model explains >80% of variance")
elif test_metrics['R²'] > 0.6:
    print("  ✅ Good performance - Model explains >60% of variance")
elif test_metrics['R²'] > 0.4:
    print("  ⚠️ Moderate performance - Model explains >40% of variance")
else:
    print("  ❌ Poor performance - Model needs improvement")

# Overfitting check
r2_diff = train_metrics['R²'] - test_metrics['R²']
print(f"\nOverfitting Check:")
print(f"  R² (Train): {train_metrics['R²']:.4f}")
print(f"  R² (Test) : {test_metrics['R²']:.4f}")
print(f"  🔁 R² Difference: {r2_diff:.4f}")
if r2_diff < 0.05:
    print("  ✅ No significant overfitting detected")
elif r2_diff < 0.15:
    print("  ⚠️ Mild overfitting - acceptable")
else:
    print("  ❌ Significant overfitting - consider regularization")

# Error analysis
print(f"\nError Analysis:")
if test_metrics['MAPE'] < 20:
    print("  ✅ Low prediction error (MAPE < 20%)")
elif test_metrics['MAPE'] < 40:
    print("  ⚠️ Moderate prediction error (MAPE < 40%)")
else:
    print("  ❌ High prediction error (MAPE > 40%)")

# Key features insight
print(f"\nKey Predictive Features:")
for i, (_, row) in enumerate(feature_importance.head(5).iterrows(), 1):
    print(f"  {i}. {row['feature']} ({row['importance']:.4f})")

print(f"\n✅ Random Forest model training and evaluation completed!")

## Random Forest Tuning with RandomizedSearchCV

In [ ]:
# Random Forest Hyperparameter Tuning

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("RANDOM FOREST HYPERPARAMETER TUNING")
print("="*40)

# Parameter grid for tuning
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

# TimeSeriesSplit for proper time series validation
tscv = TimeSeriesSplit(n_splits=3)

# Base Random Forest
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

# RandomizedSearchCV setup
rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=15,  # Reduced for faster execution
    scoring='r2',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Running hyperparameter tuning...")
# Fit on standardized training data
rf_random_search.fit(X_train_scaled, y_train)

# Get best model
best_rf_model = rf_random_search.best_estimator_

print("\nBest Parameters:")
for param, value in rf_random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV R²: {rf_random_search.best_score_:.4f}")

In [ ]:
# ===============================
# Evaluate tuned Random Forest model
# ===============================

# Predictions
y_train_pred_tuned = best_rf_model.predict(X_train_scaled)
y_test_pred_tuned  = best_rf_model.predict(X_test_scaled)

# Training metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned))
train_mae  = mean_absolute_error(y_train, y_train_pred_tuned)
train_r2   = r2_score(y_train, y_train_pred_tuned)
train_mape = np.mean(np.abs((y_train - y_train_pred_tuned) / np.clip(y_train, 1e-6, None))) * 100

# Testing metrics
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned))
test_mae  = mean_absolute_error(y_test, y_test_pred_tuned)
test_r2   = r2_score(y_test, y_test_pred_tuned)
test_mape = np.mean(np.abs((y_test - y_test_pred_tuned) / np.clip(y_test, 1e-6, None))) * 100

# Print results
print("\nTuned Random Forest Performance:")
print(f"Train R²:   {train_r2:.4f}")
print(f"Test R²:    {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
print(f"Train MAE:  {train_mae:.4f} | Test MAE:  {test_mae:.4f}")
print(f"Train MAPE: {train_mape:.2f}% | Test MAPE: {test_mape:.2f}%")

print("\n✅ Random Forest tuning completed!")

In [ ]:
# ===============================
# VISUALIZATION SECTION (Tuned Random Forest)
# ===============================
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Line Plot: Actual vs Predicted (full test set)
axes[0,0].plot(y_test.values, label='Actual', alpha=0.8, color='blue')
axes[0,0].plot(y_test_pred_tuned, label='Predicted (RF Tuned)', alpha=0.8, color='green')
axes[0,0].set_title('Actual vs Predicted Prices (Tuned Random Forest)')
axes[0,0].set_xlabel('Time Index')
axes[0,0].set_ylabel('USD Price')
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# 2. Scatter Plot: Actual vs Predicted
axes[0,1].scatter(y_test, y_test_pred_tuned, alpha=0.5, color='green')
axes[0,1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0,1].set_title("Actual vs Predicted Scatter (Tuned RF)")
axes[0,1].set_xlabel("Actual USD Price")
axes[0,1].set_ylabel("Predicted USD Price")
axes[0,1].grid(alpha=0.3)

# 3. Feature Importance (Top 15) - FIXED to use X_train_scaled
importances = best_rf_model.feature_importances_
features = X_train_scaled.columns  # ✅ use scaled feature names
sorted_idx = np.argsort(importances)[::-1]

top_n = 15
axes[1,0].barh(np.array(features)[sorted_idx][:top_n],
               importances[sorted_idx][:top_n],
               color='green')
axes[1,0].set_title(f"Top {top_n} Feature Importances (Tuned RF)")
axes[1,0].set_xlabel("Importance Score")
axes[1,0].invert_yaxis()

# 4. Time series plot (last 200 predictions)
n_plot = min(200, len(y_test))
x_axis = range(n_plot)

axes[1,1].plot(x_axis, y_test.iloc[-n_plot:], label='Actual', linewidth=2, color='blue')
axes[1,1].plot(x_axis, y_test_pred_tuned[-n_plot:], label='Predicted', linewidth=2, color='red', alpha=0.8)
axes[1,1].set_title(f'Time Series: Last {n_plot} Predictions (Tuned RF)')
axes[1,1].set_xlabel('Time Steps')
axes[1,1].set_ylabel('USD Price')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# MODEL ANALYSIS AND INSIGHTS SECTION (Tuned Random Forest)
# ===============================
print("\n" + "="*60)
print("MODEL ANALYSIS AND INSIGHTS (TUNED RANDOM FOREST)")
print("="*60)

# ===============================
# Training metrics
# ===============================
y_train_pred_tuned = best_rf_model.predict(X_train_scaled)
train_rmse_tuned = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned))
train_mae_tuned  = mean_absolute_error(y_train, y_train_pred_tuned)
train_r2_tuned   = r2_score(y_train, y_train_pred_tuned)
train_mape_tuned = np.mean(np.abs((y_train - y_train_pred_tuned) / np.clip(y_train, 1e-6, None))) * 100

# ===============================
# Testing metrics (already calculated as y_test_pred_tuned)
# ===============================
test_rmse_tuned = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned))
test_mae_tuned  = mean_absolute_error(y_test, y_test_pred_tuned)
test_r2_tuned   = r2_score(y_test, y_test_pred_tuned)
test_mape_tuned = np.mean(np.abs((y_test - y_test_pred_tuned) / np.clip(y_test, 1e-6, None))) * 100

# ===============================
# Model configuration
# ===============================
print(f"\nModel Configuration:")
print(f"  - Total features   : {len(X_train_scaled.columns)}")   # ✅ scaled features
print(f"  - Trees in forest  : {best_rf_model.n_estimators}")
print(f"  - Max depth        : {best_rf_model.max_depth}")
print(f"  - Training samples : {len(X_train_scaled):,}")        # ✅ scaled set
print(f"  - Testing samples  : {len(X_test_scaled):,}")         # ✅ scaled set

# ===============================
# Performance interpretation
# ===============================
print(f"\nPerformance Assessment:")
if test_r2_tuned > 0.8:
    print("  ✅ Excellent performance - Model explains >80% of variance")
elif test_r2_tuned > 0.6:
    print("  ✅ Good performance - Model explains >60% of variance")
elif test_r2_tuned > 0.4:
    print("  ⚠️ Moderate performance - Model explains >40% of variance")
else:
    print("  ❌ Poor performance - Model needs improvement")

# ===============================
# Overfitting check
# ===============================
r2_diff = train_r2_tuned - test_r2_tuned
print(f"\nOverfitting Check:")
print(f"  R² (Train)         : {train_r2_tuned:.4f}")
print(f"  R² (Test)          : {test_r2_tuned:.4f}")
print(f"  🔁 R² Difference    : {r2_diff:.4f}")
if r2_diff < 0.05:
    print("  ✅ No significant overfitting detected")
elif r2_diff < 0.15:
    print("  ⚠️ Mild overfitting - acceptable for production")
else:
    print("  ❌ Significant overfitting - consider regularization")

# ===============================
# Error analysis
# ===============================
print(f"\nError Analysis:")
if test_mape_tuned < 20:
    print("  ✅ Low prediction error - MAPE < 20%")
elif test_mape_tuned < 40:
    print("  ⚠️ Moderate prediction error - MAPE < 40%")
else:
    print("  ❌ High prediction error - MAPE > 40%")

# ===============================
# Top features insight - FIXED
# ===============================
feature_importance_df = pd.DataFrame({
    'feature': X_train_scaled.columns,                # ✅ scaled features
    'importance': best_rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print(f"\nKey Predictive Features:")
for i, (_, row) in enumerate(feature_importance_df.head(5).iterrows(), 1):
    print(f"  {i}. {row['feature']} ({row['importance']:.4f})")

# ===============================
# Save model results - FIXED
# ===============================
model_results_tuned = {
    'model': best_rf_model,
    'features': list(X_train_scaled.columns),          # ✅ scaled features
    'train_metrics': {
        'RMSE': train_rmse_tuned, 'MAE': train_mae_tuned,
        'R²': train_r2_tuned, 'MAPE': train_mape_tuned
    },
    'test_metrics': {
        'RMSE': test_rmse_tuned, 'MAE': test_mae_tuned,
        'R²': test_r2_tuned, 'MAPE': test_mape_tuned
    },
    'feature_importance': feature_importance_df,
    'predictions': {
        'y_train_true': y_train,
        'y_train_pred': y_train_pred_tuned,
        'y_test_true': y_test,
        'y_test_pred': y_test_pred_tuned
    }
}

print(f"\n✅ Tuned Random Forest model training and evaluation completed!")
print(f"📊 Model results stored in 'model_results_tuned' dictionary")
print(f"🎯 Ready for production deployment or further development")

# Build & Train XGBoost Regressor

In [ ]:
# ===============================
# Imports and Setup
# ===============================
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, plot_importance
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("XGBOOST REGRESSOR FOR SOMALIA FOOD PRICE FORECASTING")
print("="*60)

# ===============================
# Feature Setup
# ===============================
print("Step 1: Using all available features for XGBoost...")
print(f"Using {X_train_scaled.shape[1]} features for XGBoost")

# ===============================
#  Model Training
# ===============================
print("\nStep 2: Training XGBoost model...")

xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

eval_set = [
    (X_train_scaled, y_train),
    (X_test_scaled, y_test)
]

# Fit with evaluation tracking
xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=eval_set,
    verbose=50
)

In [ ]:
# ===============================
# Generate Predictions
# ===============================
print("\nStep 3: Generating predictions...")

y_train_pred_xgb = xgb_model.predict(X_train_scaled)
y_test_pred_xgb = xgb_model.predict(X_test_scaled)

# ===============================
# Evaluation Metrics
# ===============================
print("\nStep 4: Evaluating model...")

def metrics_report(y_true, y_pred, name="Test"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    return {"Dataset": name, "RMSE": rmse, "MAE": mae, "R²": r2, "MAPE": mape}

train_metrics = metrics_report(y_train, y_train_pred_xgb, "Training")
test_metrics  = metrics_report(y_test, y_test_pred_xgb, "Testing")

# ===============================
# CELL 6: Print Results
# ===============================
print("\n📊 XGBoost Baseline Performance:")
print(f"✅ Training R² : {train_metrics['R²']:.4f}")
print(f"✅ Testing  R² : {test_metrics['R²']:.4f}")
print(f"✅ RMSE: {test_metrics['RMSE']:.4f}")
print(f"✅ MAE : {test_metrics['MAE']:.4f}")
print(f"✅ MAPE: {test_metrics['MAPE']:.2f}%")

In [ ]:
# ===============================
#  Visualizations
# ===============================
print("\nStep 7: Creating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training vs Validation RMSE
results = xgb_model.evals_result()
epochs = len(results['validation_0']['rmse'])
axes[0,0].plot(range(epochs), results['validation_0']['rmse'], label='Train RMSE')
axes[0,0].plot(range(epochs), results['validation_1']['rmse'], label='Test RMSE')
axes[0,0].set_title('Training vs Validation RMSE')
axes[0,0].set_xlabel("Epochs")
axes[0,0].set_ylabel("RMSE")
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# 2. Predictions vs Actual
axes[0,1].scatter(y_test, y_test_pred_xgb, alpha=0.6, color='blue')
axes[0,1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0,1].set_title("Test Predictions vs Actual")
axes[0,1].set_xlabel("Actual USD Price")
axes[0,1].set_ylabel("Predicted USD Price")
axes[0,1].grid(alpha=0.3)

# 3. Residuals
residuals = y_test - y_test_pred_xgb
axes[1,0].scatter(y_test_pred_xgb, residuals, alpha=0.5, color='purple')
axes[1,0].axhline(y=0, color='red', linestyle='--')
axes[1,0].set_title("Residuals vs Predictions")
axes[1,0].set_xlabel("Predicted USD Price")
axes[1,0].set_ylabel("Residuals")
axes[1,0].grid(alpha=0.3)

# 4. Feature Importance (normalized gain)
from xgboost import plot_importance
import pandas as pd

# Get importance scores (gain)
importance_dict = xgb_model.get_booster().get_score(importance_type="gain")

# Normalize so they sum to 1
total_gain = sum(importance_dict.values())
normalized_importance = {k: v / total_gain for k, v in importance_dict.items()}

# Convert to DataFrame for sorting
feature_importance_xgb = pd.DataFrame({
    'feature': list(normalized_importance.keys()),
    'importance': list(normalized_importance.values())
}).sort_values(by="importance", ascending=False)

# Plot top 15 normalized features
axes[1,1].barh(
    feature_importance_xgb['feature'][:15],
    feature_importance_xgb['importance'][:15],
    color="salmon"
)
axes[1,1].set_title("Top 15 Feature Importances (XGBoost)")
axes[1,1].set_xlabel("Importance Score (0–1)")
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.show()

# ===============================
# Print top features
# ===============================
print(f"\nKey Predictive Features (XGBoost):")
for i, (_, row) in enumerate(feature_importance_xgb.head(5).iterrows(), 1):
    print(f"  {i}. {row['feature']} ({row['importance']:.4f})")

In [ ]:
# ===============================
# Performance Assessment
# ===============================
print(f"\nPerformance Assessment:")
if test_metrics['R²'] > 0.8:
    print("  ✅ Excellent performance - Model explains >80% of price variance")
elif test_metrics['R²'] > 0.6:
    print("  ✅ Good performance - Model explains >60% of price variance")
elif test_metrics['R²'] > 0.4:
    print("  ⚠️  Moderate performance - Model explains >40% of price variance")
else:
    print("  ❌ Poor performance - Model needs improvement")

# ===============================
# Overfitting Check
# ===============================
print("\n🔍 Step 6: Checking for overfitting using R² scores...")

# Calculate R² difference
train_r2_final = train_metrics['R²']
test_r2_final = test_metrics['R²']
r2_diff = abs(train_r2_final - test_r2_final)

print(f"  R² (Train): {train_r2_final:.4f}")
print(f"  R² (Test) : {test_r2_final:.4f}")
print(f"  🔁 R² Difference: {r2_diff:.4f}")

# Overfitting decision
if r2_diff < 0.05:
    print("  ✅ No significant overfitting detected.")
elif r2_diff < 0.15:
    print("  ⚠️  Mild overfitting - acceptable in some cases.")
else:
    print("  ❌ Significant overfitting - consider regularization, tuning, or simplifying the model.")

## Hyperparameter Tuning for XGBoost (using RandomizedSearchCV)

In [ ]:
# ===============================
# Imports and Setup
# ===============================
from xgboost import XGBRegressor, plot_importance
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("="*60)
print("XGBOOST HYPERPARAMETER TUNING FOR SOMALIA FOOD PRICE FORECASTING")
print("="*60)

# ===============================
# CELL 2: Parameter Grid Setup
# ===============================
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'max_depth': [3, 5, 7, 10],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1],  # L1 regularization
    'reg_lambda': [0, 0.1]  # L2 regularization
}

print(f"Parameter grid size: {np.prod([len(v) for v in param_grid.values()]):,} combinations")

# ===============================
#  Cross-Validation Setup
# ===============================
tscv = TimeSeriesSplit(n_splits=3)

xgb_reg = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=1  # Let RandomizedSearchCV handle parallelization
)

search = RandomizedSearchCV(
    estimator=xgb_reg,
    param_distributions=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    n_iter=30,  # Reduced for faster execution
    verbose=2,
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

# ===============================
# Hyperparameter Search
# ===============================
print("Step 1: Performing randomized search with time series CV...")
print(f"Using all {X_train_scaled.shape[1]} features")

search.fit(X_train_scaled, y_train)

print(f"\nBest CV Score (RMSE): {-search.best_score_:.4f}")
print(f"Best Parameters Found:")
for param, value in search.best_params_.items():
    print(f"   {param}: {value}")

# ===============================
# Final Model Training
# ===============================
print("\nStep 2: Training final tuned model...")

# Get the best model from search
xgb_tuned = search.best_estimator_

# Refit with eval_set for monitoring
xgb_tuned.set_params(eval_metric="rmse")
xgb_tuned.fit(
    X_train_scaled, y_train,
    eval_set=[(X_train_scaled, y_train), (X_test_scaled, y_test)],
    verbose=False
)

In [ ]:
# ===============================
#  Generate Predictions
# ===============================
print("\nGenerating predictions...")

y_train_pred_xgb_tuned = xgb_tuned.predict(X_train_scaled)
y_test_pred_xgb_tuned  = xgb_tuned.predict(X_test_scaled)

# ===============================
#  Evaluation Metrics
# ===============================
def detailed_metrics(y_true, y_pred, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)

    # Robust MAPE (avoid divide by zero)
    y_true = np.array(y_true)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else 0

    print(f"\n📊 {dataset_name} Metrics:")
    print(f"   RMSE : {rmse:.4f}")
    print(f"   MAE  : {mae:.4f}")
    print(f"   R²   : {r2:.4f}")
    print(f"   MAPE : {mape:.2f}%")

    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

# Calculate metrics
train_metrics_tuned = detailed_metrics(y_train, y_train_pred_xgb_tuned, "Training (Tuned)")
test_metrics_tuned  = detailed_metrics(y_test, y_test_pred_xgb_tuned, "Testing (Tuned)")


In [ ]:
# ===============================
# Visualizations (XGBoost Tuned )
# ===============================
print("\n📈 Creating XGBoost performance visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Predictions vs Actual (Test)
axes[0, 0].scatter(y_test, y_test_pred_xgb_tuned, alpha=0.6, color='dodgerblue', s=20)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0, 0].set_title(f'Actual vs Predicted (Test Set)\nR² = {test_metrics_tuned["R2"]:.4f}')
axes[0, 0].set_xlabel('Actual Log USD Price')
axes[0, 0].set_ylabel('Predicted Log USD Price')
axes[0, 0].grid(alpha=0.3)

# Add statistics to the plot
rmse_val = test_metrics_tuned['RMSE']
axes[0, 0].text(0.05, 0.95, f'R² = {test_metrics_tuned["R2"]:.3f}\nRMSE = {rmse_val:.3f}',
                transform=axes[0, 0].transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 2. Residuals Plot
residuals = y_test - y_test_pred_xgb_tuned
axes[0, 1].scatter(y_test_pred_xgb_tuned, residuals, alpha=0.5, color='purple', s=20)
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_title('Residuals vs Predictions')
axes[0, 1].set_xlabel('Predicted Log USD Price')
axes[0, 1].set_ylabel('Residuals (Actual - Predicted)')
axes[0, 1].grid(alpha=0.3)

# Residual bounds (±2σ)
residual_std = np.std(residuals)
axes[0, 1].axhline(y=2*residual_std, color='orange', linestyle=':', alpha=0.7, label='±2σ')
axes[0, 1].axhline(y=-2*residual_std, color='orange', linestyle=':', alpha=0.7)
axes[0, 1].legend()

# 3. Feature Importance (XGBoost)
try:
    importance_dict = xgb_tuned.get_booster().get_score(importance_type="gain")
    total_gain = sum(importance_dict.values())
    normalized_importance = {k: v / total_gain for k, v in importance_dict.items()}

    # Convert to DataFrame
    feature_importance_df = pd.DataFrame({
        "feature": list(normalized_importance.keys()),
        "importance": list(normalized_importance.values())
    }).sort_values(by="importance", ascending=False)

    # Plot Top 15
    top_n = 15
    axes[1, 0].barh(
        feature_importance_df["feature"][:top_n],
        feature_importance_df["importance"][:top_n],
        color="salmon"
    )
    axes[1, 0].set_title(f"Top {top_n} Feature Importances (XGBoost)")
    axes[1, 0].set_xlabel("Importance Score (0–1)")
    axes[1, 0].invert_yaxis()
    axes[1, 0].grid(alpha=0.3)

except Exception as e:
    print(f"Warning: Could not plot feature importance: {e}")
    axes[1, 0].text(0.5, 0.5, "Feature importance\nnot available",
                    ha='center', va='center', transform=axes[1, 0].transAxes)

# 4. Training vs Validation Curve (if eval_set used)
try:
    results = xgb_tuned.evals_result()
    print(f"Available eval result keys: {list(results.keys())}")

    if results:
        train_metric = results['validation_0']['rmse']
        test_metric  = results.get('validation_1', {}).get('rmse', None)

        epochs = range(len(train_metric))
        axes[1, 1].plot(epochs, train_metric, label='Train RMSE', color='green', linewidth=2)
        if test_metric:
            axes[1, 1].plot(epochs, test_metric, label='Test RMSE', color='orange', linewidth=2)

        axes[1, 1].set_title('Training vs Validation RMSE (XGBoost)')
        axes[1, 1].set_xlabel('Boosting Rounds')
        axes[1, 1].set_ylabel('RMSE')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, "No eval results found", ha='center', va='center', transform=axes[1, 1].transAxes)

except Exception as e:
    print(f"Error in learning curves plot: {e}")
    axes[1, 1].text(0.5, 0.5,
                   f"Error plotting curves\n{str(e)}",
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# Model Analysis (XGBoost)
# ===============================
print("\n" + "="*50)
print("XGBOOST MODEL ANALYSIS")
print("="*50)

# Model Configuration Summary
print(f"\nModel Configuration:")
print(f"  Algorithm: XGBoost Regressor")
print(f"  Features: {X_train_scaled.shape[1]}")

# Some parameters may not exist if not set explicitly, so use getattr with defaults
print(f"  Trees (n_estimators): {getattr(xgb_tuned, 'n_estimators', 'N/A')}")
print(f"  Learning rate: {getattr(xgb_tuned, 'learning_rate', 'N/A')}")
print(f"  Max depth: {getattr(xgb_tuned, 'max_depth', 'N/A')}")

print(f"\nDataset Info:")
print(f"  Training samples: {len(X_train_scaled):,}")
print(f"  Testing samples: {len(X_test_scaled):,}")

# ===============================
# Performance Assessment
# ===============================
test_r2   = test_metrics_tuned['R2']
test_rmse = test_metrics_tuned['RMSE']
test_mae  = test_metrics_tuned['MAE']
test_mape = test_metrics_tuned['MAPE']
train_r2  = train_metrics_tuned['R2']

print(f"\nPerformance Assessment:")
print(f"- R² Score: {test_r2:.4f}")
print(f"- RMSE: {test_rmse:.4f}")
print(f"- MAE: {test_mae:.4f}")
print(f"- MAPE: {test_mape:.2f}%")

# Performance interpretation
if test_r2 > 0.9:
    print("Outstanding performance - Model explains >90% of price variance")
elif test_r2 > 0.8:
    print("Excellent performance - Model explains >80% of price variance")
elif test_r2 > 0.6:
    print("Good performance - Model explains >60% of price variance")
else:
    print("Moderate performance")

print(f"\nPrediction Accuracy (MAPE Analysis):")
if test_mape < 5:
    print("Exceptional accuracy - Average error < 5%")
elif test_mape < 10:
    print("Excellent accuracy - Average error < 10%")
elif test_mape < 15:
    print("Good accuracy - Average error < 15%")
else:
    print("Moderate accuracy")

# ===============================
# Overfitting Analysis
# ===============================
print(f"\nOverfitting Analysis:")
print(f"- Training R²: {train_r2:.4f}")
print(f"- Testing R²: {test_r2:.4f}")
print(f"- Performance Gap: {abs(train_r2 - test_r2):.4f}")

gap = abs(train_r2 - test_r2)
if gap < 0.05:
    print("Excellent generalization - No overfitting detected")
elif gap < 0.15:
    print("Good generalization - Minimal overfitting")
else:
    print("Potential overfitting detected")

# ===============================
# Feature Importance Analysis (Normalized Gain)
# ===============================
print(f"\nTop 10 Features for Volatility Forecasting (Normalized Gain):")

# Get gain-based importance
importance_dict = xgb_tuned.get_booster().get_score(importance_type="gain")

# Normalize to 0–1
total_gain = sum(importance_dict.values())
normalized_importance = {k: v / total_gain for k, v in importance_dict.items()}

# Convert to DataFrame
feature_importance_df = pd.DataFrame({
    "feature": list(normalized_importance.keys()),
    "importance": list(normalized_importance.values())
}).sort_values(by="importance", ascending=False)

# Print top 10
for i, (_, row) in enumerate(feature_importance_df.head(10).iterrows(), 1):
    print(f"  {i:2d}. {row['feature']:<30} {row['importance']:.4f}")

print("\nXGBoost tuned model analysis completed!")

# LightGBM Regressor - Baseline

In [ ]:
# ===============================
# Imports and Setup
# ===============================
from lightgbm import LGBMRegressor, log_evaluation, early_stopping, plot_importance
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("="*60)
print("LIGHTGBM REGRESSOR FOR SOMALIA FOOD PRICE FORECASTING")
print("="*60)

# ===============================
#  Data Preparation
# ===============================
print("Step 1: Using all available features for LightGBM...")
print(f"Using {X_train_scaled.shape[1]} features for LightGBM")

# Check for NaN values
if X_train_scaled.isnull().any().any():
    print("Warning: NaN values found, filling with 0")
    X_train_scaled = X_train_scaled.fillna(0)
    X_test_scaled = X_test_scaled.fillna(0)

# ===============================
# Model Configuration
# ===============================
print("\nStep 2: Configuring LightGBM model...")

lgb_model = LGBMRegressor(
    objective='regression',
    metric='rmse',
    boosting_type='gbdt',
    random_state=42,
    n_estimators=1000,
    verbosity=-1
)

# ===============================
# Model Training
# ===============================
print("\nStep 3: Training LightGBM...")

# Prepare evaluation sets
eval_set = [
    (X_train_scaled, y_train),
    (X_test_scaled, y_test)
]

eval_names = ['training', 'validation']

# Training with callbacks
callbacks = [
    log_evaluation(period=100),
    early_stopping(stopping_rounds=50, first_metric_only=True)
]

lgb_model.fit(
    X_train_scaled,
    y_train,
    eval_set=eval_set,
    eval_names=eval_names,
    callbacks=callbacks
)

print(f"\nLightGBM training completed!")

In [ ]:
# ===============================
# Generate Predictions
# ===============================
print("\nStep 4: Generating predictions...")

y_train_pred_lgb = lgb_model.predict(X_train_scaled)
y_test_pred_lgb = lgb_model.predict(X_test_scaled)

# ===============================
#  Evaluation Metrics
# ===============================
print("\nStep 5: Evaluating model...")

def metrics_report(y_true, y_pred, name="Test"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    return {"Dataset": name, "RMSE": rmse, "MAE": mae, "R²": r2, "MAPE": mape}

train_metrics_lgb = metrics_report(y_train, y_train_pred_lgb, "Training")
test_metrics_lgb = metrics_report(y_test, y_test_pred_lgb, "Testing")

# ===============================
#  Print Results
# ===============================
print("\n📊 LightGBM Baseline Performance:")
print(f"✅ Training R² : {train_metrics_lgb['R²']:.4f}")
print(f"✅ Testing  R² : {test_metrics_lgb['R²']:.4f}")
print(f"✅ RMSE: {test_metrics_lgb['RMSE']:.4f}")
print(f"✅ MAE : {test_metrics_lgb['MAE']:.4f}")
print(f"✅ MAPE: {test_metrics_lgb['MAPE']:.2f}%")

In [ ]:
# ===============================
#  Visualizations (LightGBM )
# ===============================
print("\n📊 Creating LightGBM performance visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Predictions vs Actual (Test Set)
axes[0,0].scatter(y_test, y_test_pred_lgb, alpha=0.6, color='dodgerblue')
axes[0,0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0,0].set_title("Actual vs Predicted (Test Set)")
axes[0,0].set_xlabel("Actual USD Price")
axes[0,0].set_ylabel("Predicted USD Price")
axes[0,0].grid(alpha=0.3)

# 2. Residuals Plot
residuals = y_test - y_test_pred_lgb
axes[0,1].scatter(y_test_pred_lgb, residuals, alpha=0.5, color='purple')
axes[0,1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_title("Residuals vs Predictions")
axes[0,1].set_xlabel("Predicted USD Price")
axes[0,1].set_ylabel("Residuals (Actual - Predicted)")
axes[0,1].grid(alpha=0.3)

# 3. Feature Importance
importances = lgb_model.booster_.feature_importance(importance_type="gain")
features = lgb_model.booster_.feature_name()
importances_normalized = importances / importances.sum()

fi_df = pd.DataFrame({
    "feature": features,
    "importance": importances_normalized
}).sort_values(by="importance", ascending=False)

axes[1,0].barh(fi_df['feature'][:15][::-1], fi_df['importance'][:15][::-1], color="skyblue")
axes[1,0].set_title("Top 15 Feature Importances ")
axes[1,0].set_xlabel("Normalized Importance (0–1)")
axes[1,0].grid(alpha=0.3)

# 4. Training vs Validation RMSE Curve
results = getattr(lgb_model, "evals_result_", None)
if results and "training" in results and "validation" in results:
    axes[1,1].plot(results['training']['rmse'], label='Train RMSE', color='green', linewidth=2)
    axes[1,1].plot(results['validation']['rmse'], label='Validation RMSE', color='orange', linewidth=2)
    axes[1,1].set_title("Training vs Validation RMSE")
    axes[1,1].set_xlabel("Iterations")
    axes[1,1].set_ylabel("RMSE")
    axes[1,1].legend()
    axes[1,1].grid(alpha=0.3)
else:
    axes[1,1].text(0.5, 0.5, "No evals_result available", ha='center', va='center')
    axes[1,1].set_title("Training vs Validation RMSE")
    axes[1,1].set_xticks([])
    axes[1,1].set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# Model Analysis (LightGBM )
# ===============================
print("\nPERFORMANCE ASSESSMENT")
print(f"  ✅ Training R² : {train_metrics_lgb['R²']:.4f}")
print(f"  ✅ Testing  R² : {test_metrics_lgb['R²']:.4f}")
print(f"  ✅ RMSE: {test_metrics_lgb['RMSE']:.4f}")
print(f"  ✅ MAE : {test_metrics_lgb['MAE']:.4f}")
print(f"  ✅ MAPE: {test_metrics_lgb['MAPE']:.2f}%")

# Performance interpretation
if test_metrics_lgb['R²'] > 0.8:
    print("  🔥 Excellent performance - explains >80% of variance")
elif test_metrics_lgb['R²'] > 0.6:
    print("  👍 Good performance - explains >60% of variance")
elif test_metrics_lgb['R²'] > 0.4:
    print("  ⚠️  Moderate performance")
else:
    print("  ❌ Poor performance - needs improvement")

# ===============================
# Overfitting Check
# ===============================
print("\nOVERFITTING CHECK")
r2_diff = abs(train_metrics_lgb['R²'] - test_metrics_lgb['R²'])
print(f"  R² Difference: {r2_diff:.4f}")

if r2_diff < 0.05:
    print("  ✅ No significant overfitting")
elif r2_diff < 0.15:
    print("  ⚠️  Mild overfitting - acceptable in some cases")
else:
    print("  ❌ Significant overfitting - consider tuning or simplifying the model")

# ===============================
# Feature Importance Analysis (Normalized Gain)
# ===============================
print("\nKEY PREDICTIVE FEATURES:")

# Get normalized gain importances
importances = lgb_model.booster_.feature_importance(importance_type="gain")
features = X_train_scaled.columns
importances_norm = importances / importances.sum()

importance_df = (
    pd.DataFrame({"Feature": features, "Importance": importances_norm})
    .sort_values(by="Importance", ascending=False)
    .head(10)  # top 10
)

for i, row in enumerate(importance_df.itertuples(index=False), 1):
    print(f"  {i}. {row.Feature:<30} ({row.Importance:.4f})")

print("\n✅ LightGBM model training and evaluation completed!")

## LightGBM Hyperparameter Tuning

In [ ]:
# ===============================
# LightGBM Hyperparameter Tuning
# ===============================

from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("="*60)
print("LIGHTGBM HYPERPARAMETER TUNING FOR SOMALIA FOOD PRICE FORECASTING")
print("="*60)

# =========================================
# 1. Parameter Space
# =========================================
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'num_leaves': [15, 31, 50, 100],
    'max_depth': [-1, 5, 7, 10],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'min_child_samples': [10, 20, 50],
    'min_child_weight': [0.001, 0.01, 0.1]
}

print(f"Parameter grid size: {np.prod([len(v) for v in param_grid.values()]):,} combinations")

# =========================================
# 2. Time Series Cross-Validation Setup
# =========================================
tscv = TimeSeriesSplit(n_splits=3)

lgb_base = LGBMRegressor(
    objective='regression',
    metric='rmse',
    boosting_type='gbdt',
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

search_lgb = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    n_iter=20,  # Reduced for faster execution
    verbose=2,
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

# =========================================
# 3. Hyperparameter Search
# =========================================
print("Step 2: Performing randomized search with time series CV...")
print(f"Using all {X_train_scaled.shape[1]} features")

search_lgb.fit(X_train_scaled, y_train)

print(f"\nBest CV Score (RMSE): {-search_lgb.best_score_:.4f}")
print(f"Best Parameters Found:")
for param, value in search_lgb.best_params_.items():
    print(f"   {param}: {value}")

In [ ]:
# =========================================
# 4. Final Model Evaluation
# =========================================
print("\nStep 3: Evaluating tuned model...")

# Get the best model
lgb_tuned = LGBMRegressor(
    **search_lgb.best_params_,
    objective='regression',
    metric='rmse',
    random_state=42,
    verbosity=-1
)

# Setup eval sets for RMSE tracking
eval_set = [(X_train_scaled, y_train), (X_test_scaled, y_test)]
eval_names = ['training', 'validation']

# Train with evaluation tracking
lgb_tuned.fit(
    X_train_scaled, y_train,
    eval_set=eval_set,
    eval_names=eval_names,
    callbacks=[early_stopping(50, first_metric_only=True), log_evaluation(0)]
)

# Make predictions
y_train_pred_lgb_tuned = lgb_tuned.predict(X_train_scaled)
y_test_pred_lgb_tuned = lgb_tuned.predict(X_test_scaled)

# Evaluation function
def detailed_metrics(y_true, y_pred, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # More robust MAPE calculation
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else 0

    print(f"\n📊 {dataset_name} Metrics:")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE:  {mae:.4f}")
    print(f"   R²:   {r2:.4f}")
    print(f"   MAPE: {mape:.2f}%")

    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

# Calculate metrics
train_metrics_lgb_tuned = detailed_metrics(y_train, y_train_pred_lgb_tuned, "Training (LightGBM Tuned)")
test_metrics_lgb_tuned = detailed_metrics(y_test, y_test_pred_lgb_tuned, "Testing (LightGBM Tuned)")

print("\nLightGBM hyperparameter tuning with eval tracking completed!")
print("RMSE history available in lgb_tuned.evals_result_ for visualization")

In [ ]:
# LightGBM Tuned Model Visualization and Analysis

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import plot_importance

# Extract metrics from tuned model
train_r2 = train_metrics_lgb_tuned['R2']
test_r2 = test_metrics_lgb_tuned['R2']
test_rmse = test_metrics_lgb_tuned['RMSE']
test_mape = test_metrics_lgb_tuned['MAPE']

print("\nStep 3: Creating comprehensive LightGBM tuned model analysis...")

# ===============================
# Overfitting Analysis
# ===============================
r2_gap = train_r2 - test_r2

print(f"\nOverfitting Analysis:")
print(f"Training R²: {train_r2:.4f}")
print(f"Testing R²:  {test_r2:.4f}")
print(f"R² Gap:      {r2_gap:.4f}")

if r2_gap < 0.02:
    overfitting_status = "Excellent generalization"
elif r2_gap < 0.05:
    overfitting_status = "Good generalization"
elif r2_gap < 0.1:
    overfitting_status = "Moderate overfitting"
else:
    overfitting_status = "Significant overfitting"

print(f"Assessment:  {overfitting_status}")

# ===============================
# Comprehensive Visualizations
# ===============================
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Predictions vs Actual
axes[0, 0].scatter(y_test, y_test_pred_lgb_tuned, alpha=0.6, s=20, color='blue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Log USD Price')
axes[0, 0].set_ylabel('Predicted Log USD Price')
axes[0, 0].set_title(f'LightGBM Tuned: Actual vs Predicted\nR² = {test_r2:.4f}')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].text(0.05, 0.95, f'R² = {test_r2:.3f}\nRMSE = {test_rmse:.3f}',
                transform=axes[0, 0].transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 2. Residuals Plot
residuals = y_test - y_test_pred_lgb_tuned
axes[0, 1].scatter(y_test_pred_lgb_tuned, residuals, alpha=0.5, s=20, color='purple')
axes[0, 1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted Log USD Price')
axes[0, 1].set_ylabel('Residuals (Actual - Predicted)')
axes[0, 1].set_title('Residuals vs Predictions')
axes[0, 1].grid(alpha=0.3)

residual_std = np.std(residuals)
axes[0, 1].axhline(y=2*residual_std, color='orange', linestyle=':', alpha=0.7)
axes[0, 1].axhline(y=-2*residual_std, color='orange', linestyle=':', alpha=0.7)

# 3. Feature Importance (Fixed with Gain normalization)
try:
    # Try LightGBM's built-in plot first
    plot_importance(lgb_tuned, max_num_features=15, height=0.5, ax=axes[1, 0],
                   importance_type='gain', show_values=False)
    axes[1, 0].set_title('Top 15 Feature Importance - Gain (LightGBM Tuned)')
    axes[1, 0].grid(alpha=0.3)
except:
    # If that fails, create manual plot using gain-based importance
    try:
        # Get feature importance using gain
        importance_gain = lgb_tuned.booster_.feature_importance(importance_type='gain')
        feature_names = lgb_tuned.feature_name_

        # Create DataFrame and sort by importance
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importance_gain
        }).sort_values('importance', ascending=True).tail(15)

        # Create horizontal bar plot
        axes[1, 0].barh(range(len(importance_df)), importance_df['importance'], color='skyblue')
        axes[1, 0].set_yticks(range(len(importance_df)))
        axes[1, 0].set_yticklabels(importance_df['feature'])
        axes[1, 0].set_xlabel('Feature Importance (Gain)')
        axes[1, 0].set_title('Top 15 Feature Importance - Gain (LightGBM Tuned)')
        axes[1, 0].grid(alpha=0.3, axis='x')

    except Exception as e:
        print(f"Warning: Could not plot feature importance: {e}")
        axes[1, 0].text(0.5, 0.5, "Feature importance\nnot available",
                        ha='center', va='center', transform=axes[1, 0].transAxes)

# 4. Training vs Validation RMSE Curve
try:
    if hasattr(lgb_tuned, 'evals_result_'):
        results = lgb_tuned.evals_result_
        if 'training' in results and 'validation' in results:
            train_rmse_history = results['training']['rmse']
            valid_rmse_history = results['validation']['rmse']

            epochs = range(1, len(train_rmse_history) + 1)
            axes[1, 1].plot(epochs, train_rmse_history, label='Training RMSE', color='green', lw=2)
            axes[1, 1].plot(epochs, valid_rmse_history, label='Validation RMSE', color='orange', lw=2)

            axes[1, 1].set_xlabel('Boosting Round')
            axes[1, 1].set_ylabel('RMSE')
            axes[1, 1].set_title('Training vs Validation RMSE (Tuned)')
            axes[1, 1].legend()
            axes[1, 1].grid(alpha=0.3)
        else:
            axes[1, 1].text(0.5, 0.5, 'Training curves not available',
                           ha='center', va='center', transform=axes[1, 1].transAxes)
    else:
        axes[1, 1].text(0.5, 0.5, 'No evals_result_ found',
                       ha='center', va='center', transform=axes[1, 1].transAxes)
except Exception as e:
    axes[1, 1].text(0.5, 0.5, f'Error: {str(e)}',
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# Model Analysis and Insights
# ===============================
print("\n" + "="*60)
print("LIGHTGBM TUNED MODEL ANALYSIS AND INSIGHTS")
print("="*60)

# Performance Assessment
print(f"\nPerformance Assessment:")
print(f"- R² Score: {test_r2:.4f}")
print(f"- RMSE: {test_rmse:.4f}")
print(f"- MAPE: {test_mape:.2f}%")

# Performance interpretation
if test_r2 > 0.9:
    print("Outstanding performance - Model explains >90% of price variance")
elif test_r2 > 0.8:
    print("Excellent performance - Model explains >80% of price variance")
elif test_r2 > 0.6:
    print("Good performance - Model explains >60% of price variance")
else:
    print("Moderate performance")

print(f"\nPrediction Accuracy (MAPE Analysis):")
if test_mape < 5:
    print("Exceptional accuracy - Average error < 5%")
elif test_mape < 10:
    print("Excellent accuracy - Average error < 10%")
elif test_mape < 15:
    print("Good accuracy - Average error < 15%")
else:
    print("Moderate accuracy")

print(f"\nOverfitting Analysis:")
print(f"- Training R²: {train_r2:.4f}")
print(f"- Testing R²: {test_r2:.4f}")
print(f"- Performance Gap: {abs(train_r2 - test_r2):.4f}")
print(f"- Assessment: {overfitting_status}")

# Top Features Analysis
try:
    # Get gain-based feature importance
    feature_importance = lgb_tuned.booster_.feature_importance(importance_type='gain')
    importance_df = pd.DataFrame({
        'feature': X_train_scaled.columns,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
except:
    # Fallback to default if gain method fails
    feature_importance = lgb_tuned.feature_importances_
    importance_df = pd.DataFrame({
        'feature': X_train_scaled.columns,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    print("Note: Using default importance (may be split-based, not gain-based)")

# Normalize the importance values between 0 and 1
importance_df['importance_normalized'] = (importance_df['importance'] - importance_df['importance'].min()) / (importance_df['importance'].max() - importance_df['importance'].min())

print(f"\nTop 10 Features for Volatility Forecasting (Normalized 0-1):")
for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
    print(f"  {i:2d}. {row['feature']:<30} {row['importance_normalized']:.4f}")

print("\nLightGBM tuned model analysis completed!")

# Hybrid Model: LSTM + XGBoost

In [ ]:
# ===============================
# Hybrid Model: LSTM + XGBoost
# ===============================
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

print("="*60)
print("HYBRID LSTM+XGBOOST MODEL FOR SOMALIA FOOD PRICE FORECASTING")
print("="*60)

# ===============================
# 1. Data Preparation for LSTM
# ===============================
print("Step 1: Preparing data for LSTM...")

# Select time-series features for LSTM (use already scaled features)
lstm_feature_cols = ['usd_price_lag_1', 'usd_price_lag_3', 'usd_price_lag_6',
                     'price_rolling_mean_3', 'price_rolling_std_3',
                     'price_pct_change_1m', 'exchange_rate_unofficial']

# Filter available features from scaled data
available_lstm_features = [f for f in lstm_feature_cols if f in X_train_scaled.columns]
print(f"Using {len(available_lstm_features)} LSTM features: {available_lstm_features}")

# Use already scaled features (no additional scaling)
X_train_lstm = X_train_scaled[available_lstm_features].values
X_test_lstm = X_test_scaled[available_lstm_features].values

# Scale only the target for LSTM (features are already scaled)
scaler_target = MinMaxScaler()
y_train_scaled = scaler_target.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = scaler_target.transform(y_test.values.reshape(-1, 1)).flatten()

# Create sequences for LSTM
def create_sequences(X, y, sequence_length=10):
    X_seq, y_seq = [], []
    for i in range(sequence_length, len(X)):
        X_seq.append(X[i-sequence_length:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

sequence_length = 10
X_train_seq, y_train_seq = create_sequences(X_train_lstm, y_train_scaled, sequence_length)
X_test_seq, y_test_seq = create_sequences(X_test_lstm, y_test_scaled, sequence_length)

print(f"LSTM training sequences shape: {X_train_seq.shape}")
print(f"LSTM testing sequences shape: {X_test_seq.shape}")

# ===============================
# 2. Build and Train LSTM Model
# ===============================
print("\nStep 2: Building and training LSTM model...")

lstm_model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(sequence_length, len(available_lstm_features))),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

print("Training LSTM...")
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_seq, y_test_seq),
    callbacks=callbacks,
    verbose=0
)

In [ ]:
# ===============================
# 3. Generate LSTM Predictions
# ===============================
print("\nStep 3: Generating LSTM predictions...")

lstm_train_pred_scaled = lstm_model.predict(X_train_seq, verbose=0)
lstm_test_pred_scaled = lstm_model.predict(X_test_seq, verbose=0)

# Unscale predictions
lstm_train_pred = scaler_target.inverse_transform(lstm_train_pred_scaled).flatten()
lstm_test_pred = scaler_target.inverse_transform(lstm_test_pred_scaled).flatten()

# Align target arrays for sequence offset
y_train_lstm_aligned = y_train.iloc[sequence_length:].values
y_test_lstm_aligned = y_test.iloc[sequence_length:].values

# LSTM standalone performance
lstm_train_r2 = r2_score(y_train_lstm_aligned, lstm_train_pred)
lstm_test_r2 = r2_score(y_test_lstm_aligned, lstm_test_pred)
lstm_test_rmse = np.sqrt(mean_squared_error(y_test_lstm_aligned, lstm_test_pred))

print(f"LSTM Standalone Performance:")
print(f"  Training R²: {lstm_train_r2:.4f}")
print(f"  Testing R²: {lstm_test_r2:.4f}")
print(f"  Testing RMSE: {lstm_test_rmse:.4f}")

# ===============================
# 4. Prepare Hybrid Model Data
# ===============================
print("\nStep 4: Preparing data for hybrid model...")

# Use remaining features for XGBoost (exclude LSTM features)
xgb_feature_cols = [col for col in X_train_scaled.columns if col not in available_lstm_features]
print(f"Using {len(xgb_feature_cols)} XGBoost features")

# Align XGBoost features with LSTM predictions
X_train_xgb_aligned = X_train_scaled[xgb_feature_cols].iloc[sequence_length:].reset_index(drop=True)
X_test_xgb_aligned = X_test_scaled[xgb_feature_cols].iloc[sequence_length:].reset_index(drop=True)

# Create hybrid features: XGBoost features + LSTM predictions
X_train_hybrid = X_train_xgb_aligned.copy()
X_test_hybrid = X_test_xgb_aligned.copy()

X_train_hybrid['lstm_prediction'] = lstm_train_pred
X_test_hybrid['lstm_prediction'] = lstm_test_pred

print(f"Hybrid training shape: {X_train_hybrid.shape}")
print(f"Hybrid testing shape: {X_test_hybrid.shape}")

In [ ]:
# ===============================
# 5. Train XGBoost on Hybrid Features
# ===============================
print("\nStep 5: Training XGBoost on hybrid features...")

# Use best XGBoost parameters from tuning
hybrid_xgb = XGBRegressor(
    **search_lgb.best_params_,  # Use your best tuned parameters
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Train hybrid model
hybrid_xgb.fit(
    X_train_hybrid, y_train_lstm_aligned,
    eval_set=[(X_test_hybrid, y_test_lstm_aligned)],
    verbose=False
)

# ===============================
# 6. Hybrid Model Evaluation
# ===============================
print("\nStep 6: Evaluating hybrid model...")

hybrid_train_pred = hybrid_xgb.predict(X_train_hybrid)
hybrid_test_pred = hybrid_xgb.predict(X_test_hybrid)

def calculate_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else 0

    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MAPE': mape}

# Calculate metrics
lstm_train_metrics = calculate_metrics(y_train_lstm_aligned, lstm_train_pred, "LSTM Train")
lstm_test_metrics = calculate_metrics(y_test_lstm_aligned, lstm_test_pred, "LSTM Test")
hybrid_train_metrics = calculate_metrics(y_train_lstm_aligned, hybrid_train_pred, "Hybrid Train")
hybrid_test_metrics = calculate_metrics(y_test_lstm_aligned, hybrid_test_pred, "Hybrid Test")

print("\n📊 Model Evaluation Results:")
for m in [lstm_train_metrics, lstm_test_metrics, hybrid_train_metrics, hybrid_test_metrics]:
    print(f"  {m['Model']}: R²={m['R²']:.4f}, RMSE={m['RMSE']:.4f}, MAE={m['MAE']:.4f}, MAPE={m['MAPE']:.2f}%")

print("\nHybrid LSTM+XGBoost model training completed!")

In [ ]:
# Hybrid Model Overfitting Check and Performance Comparison

from sklearn.metrics import r2_score

# ===============================
# Overfitting Check
# ===============================
print("\nChecking for overfitting using R²...")

# Calculate R² scores using existing metrics
r2_train = hybrid_train_metrics['R²']
r2_test = hybrid_test_metrics['R²']

# Calculate absolute difference
r2_diff = abs(r2_train - r2_test)

# Print results
print(f"  R² (Train): {r2_train:.4f}")
print(f"  R² (Test) : {r2_test:.4f}")
print(f"  R² Difference: {r2_diff:.4f}")

# Interpretation
if r2_diff < 0.05:
    print("  No significant overfitting detected.")
elif r2_diff < 0.15:
    print("  Mild overfitting - acceptable in some cases.")
else:
    print("  Significant overfitting - consider tuning or simplifying the model.")

# ===============================
# Results Comparison
# ===============================
print("\n" + "="*60)
print("HYBRID MODEL PERFORMANCE COMPARISON")
print("="*60)

results_df = pd.DataFrame([
    lstm_test_metrics,
    hybrid_test_metrics
])

print("\nTest Set Performance:")
print(f"{'Model':<15} {'RMSE':<8} {'MAE':<8} {'R²':<8} {'MAPE':<8}")
print("-" * 50)
for _, row in results_df.iterrows():
    print(f"{row['Model']:<15} {row['RMSE']:<8.4f} {row['MAE']:<8.4f} {row['R²']:<8.4f} {row['MAPE']:<8.1f}%")

# Performance improvement
r2_improvement = hybrid_test_metrics['R²'] - lstm_test_metrics['R²']
rmse_improvement = lstm_test_metrics['RMSE'] - hybrid_test_metrics['RMSE']

print(f"\nHybrid Model Improvement:")
print(f"  R² improvement: {r2_improvement:+.4f}")
print(f"  RMSE improvement: {rmse_improvement:+.4f}")

# Performance assessment
if hybrid_test_metrics['R²'] > lstm_test_metrics['R²']:
    print("Hybrid model outperforms standalone LSTM")
else:
    print("Hybrid model did not improve over LSTM")

print("\nHybrid model analysis completed!")

In [ ]:
# ===============================
# Feature Importance Analysis
# ===============================
print("\nStep 7: Analyzing feature importance...")

# Get gain-based feature importance from hybrid XGBoost
try:
    importance_dict = hybrid_xgb.get_booster().get_score(importance_type='gain')
    feature_names = list(X_train_hybrid.columns)
    feature_importance = [importance_dict.get(f, 0) for f in feature_names]
    importance_type = "Gain-based"
except Exception as e:
    print(f"Warning: {e}. Using default importance...")
    feature_importance = hybrid_xgb.feature_importances_
    feature_names = list(X_train_hybrid.columns)
    importance_type = "Default"

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance,
    'importance_pct': (np.array(feature_importance) / np.sum(feature_importance)) * 100
}).sort_values('importance', ascending=False)

# Normalize importance values between 0 and 1
importance_df['importance_normalized'] = (importance_df['importance'] - importance_df['importance'].min()) / (importance_df['importance'].max() - importance_df['importance'].min())

print(f"\nTop 10 Features in Hybrid Model ({importance_type}) - Normalized 0-1:")
print("Rank  Feature                    Normalized")
print("-" * 38)
for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
    print(f"{i:2d}.  {row['feature']:<22} {row['importance_normalized']:>8.4f}")

# Check LSTM prediction importance
lstm_rows = importance_df[importance_df['feature'] == 'lstm_prediction']
if not lstm_rows.empty:
    lstm_importance = lstm_rows['importance_pct'].iloc[0]
    print(f"\nLSTM prediction importance: {lstm_importance:.1f}% of total model")
else:
    print(f"\nLSTM prediction not found in feature importance")

In [ ]:
# ===============================
# Visualizations
# ===============================
print("\nStep 8: Creating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. LSTM Training History
axes[0, 0].plot(history.history['loss'], label='Training Loss', color='blue')
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
axes[0, 0].set_title('LSTM Training History')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Model Comparison - Predictions vs Actual
axes[0, 1].scatter(y_test_lstm_aligned, lstm_test_pred, alpha=0.6, label='LSTM', s=20)
axes[0, 1].scatter(y_test_lstm_aligned, hybrid_test_pred, alpha=0.6, label='Hybrid', s=20)
min_val = min(y_test_lstm_aligned.min(), hybrid_test_pred.min())
max_val = max(y_test_lstm_aligned.max(), hybrid_test_pred.max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Log USD Price')
axes[0, 1].set_ylabel('Predicted Log USD Price')
axes[0, 1].set_title('Model Comparison: Actual vs Predicted')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Residuals Comparison
lstm_residuals = y_test_lstm_aligned - lstm_test_pred
hybrid_residuals = y_test_lstm_aligned - hybrid_test_pred

axes[1, 0].scatter(lstm_test_pred, lstm_residuals, alpha=0.6, label='LSTM', s=20)
axes[1, 0].scatter(hybrid_test_pred, hybrid_residuals, alpha=0.6, label='Hybrid', s=20)
axes[1, 0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted Log USD Price')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Residuals Comparison')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Feature Importance (Top 15) - Fixed with Gain-based importance
try:
    # Get gain-based importance from XGBoost
    importance_dict = hybrid_xgb.get_booster().get_score(importance_type='gain')

    # Handle case where some features might not appear in importance
    feature_names = list(X_train_hybrid.columns)
    feature_importance = [importance_dict.get(f, 0) for f in feature_names]

    # Create DataFrame with gain-based importance
    importance_df_gain = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)

    # Get top 15 features
    top_15_features = importance_df_gain.head(15)

    axes[1, 1].barh(range(len(top_15_features)), top_15_features['importance'], color='skyblue')
    axes[1, 1].set_yticks(range(len(top_15_features)))
    axes[1, 1].set_yticklabels(top_15_features['feature'], fontsize=8)
    axes[1, 1].set_xlabel('Feature Importance (Gain)')
    axes[1, 1].set_title('Hybrid Model: Top 15 Feature Importance - Gain')
    axes[1, 1].invert_yaxis()
    axes[1, 1].grid(alpha=0.3)

except Exception as e:
    print(f"Warning: Could not get gain-based importance: {e}")
    # Fallback to default importance
    top_15_features = importance_df.head(15)
    axes[1, 1].barh(range(len(top_15_features)), top_15_features['importance'], color='lightcoral')
    axes[1, 1].set_yticks(range(len(top_15_features)))
    axes[1, 1].set_yticklabels(top_15_features['feature'], fontsize=8)
    axes[1, 1].set_xlabel('Feature Importance (Default)')
    axes[1, 1].set_title('Hybrid Model: Top 15 Feature Importance - Default')
    axes[1, 1].invert_yaxis()
    axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# Final Analysis and Recommendations
# ===============================
print("\n" + "="*60)
print("HYBRID MODEL ANALYSIS AND RECOMMENDATIONS")
print("="*60)

if hybrid_test_metrics['R²'] > lstm_test_metrics['R²']:
    print(f"Hybrid model outperforms standalone LSTM")
    print(f"   R² improvement: {r2_improvement:+.4f}")
else:
    print(f"Hybrid model did not improve over LSTM")

if hybrid_test_metrics['R²'] > 0.8:
    recommendation = "Excellent - Ready for deployment"
elif hybrid_test_metrics['R²'] > 0.7:
    recommendation = "Very Good - Consider deployment"
elif hybrid_test_metrics['R²'] > 0.6:
    recommendation = "Good - Fine-tune before deployment"
else:
    recommendation = "Needs improvement - Consider architecture changes"

print(f"\nFinal Recommendation: {recommendation}")
print(f"Hybrid Model Test Performance: R² = {hybrid_test_metrics['R²']:.4f}, RMSE = {hybrid_test_metrics['RMSE']:.4f}")

if lstm_importance > 20:
    print(f"LSTM component is highly valuable ({lstm_importance:.1f}% importance)")
elif lstm_importance > 10:
    print(f"LSTM component provides moderate value ({lstm_importance:.1f}% importance)")
else:
    print(f"LSTM component has limited impact ({lstm_importance:.1f}% importance)")

print("\nNext Steps:")
print("- Compare with standalone XGBoost performance")
print("- Consider ensemble methods (stacking/blending)")
print("- Experiment with different LSTM architectures")
print("- Try different sequence lengths for LSTM")

print("\n" + "="*60)
print("HYBRID MODEL ANALYSIS COMPLETE")
print("="*60)

## Hybrid Model Hyperparameter Tuning

In [ ]:
# Hybrid Model Hyperparameter Tuning

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("="*60)
print("HYBRID MODEL HYPERPARAMETER TUNING")
print("="*60)

# ===============================
# 1. LSTM Hyperparameter Tuning
# ===============================
print("Step 1: Tuning LSTM component...")

# LSTM parameter grid - focus on regularization
lstm_configs = [
    {'units1': 32, 'units2': 16, 'dropout': 0.3, 'sequence_length': 5},
    {'units1': 64, 'units2': 32, 'dropout': 0.4, 'sequence_length': 8},
    {'units1': 50, 'units2': 25, 'dropout': 0.5, 'sequence_length': 12},
    {'units1': 40, 'units2': 20, 'dropout': 0.3, 'sequence_length': 6},
    {'units1': 30, 'units2': 15, 'dropout': 0.4, 'sequence_length': 10}
]

best_lstm_config = None
best_lstm_val_r2 = -np.inf

for i, config in enumerate(lstm_configs):
    print(f"\nTesting LSTM config {i+1}/5: {config}")

    # Create sequences with current config
    seq_len = config['sequence_length']
    X_train_seq, y_train_seq = create_sequences(X_train_lstm, y_train_scaled, seq_len)
    X_test_seq, y_test_seq = create_sequences(X_test_lstm, y_test_scaled, seq_len)

    # Build model with current config
    model = Sequential([
        LSTM(config['units1'], return_sequences=True,
             input_shape=(seq_len, len(available_lstm_features))),
        Dropout(config['dropout']),
        LSTM(config['units2'], return_sequences=False),
        Dropout(config['dropout']),
        Dense(15),
        Dense(1)
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

    # Train with early stopping
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=30,
        batch_size=64,
        validation_data=(X_test_seq, y_test_seq),
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
        verbose=0
    )

    # Evaluate
    test_pred_scaled = model.predict(X_test_seq, verbose=0)
    test_pred = scaler_target.inverse_transform(test_pred_scaled).flatten()
    y_test_aligned = y_test.iloc[seq_len:].values

    val_r2 = r2_score(y_test_aligned, test_pred)
    print(f"Validation R²: {val_r2:.4f}")

    if val_r2 > best_lstm_val_r2:
        best_lstm_val_r2 = val_r2
        best_lstm_config = config
        best_lstm_model = model
        best_seq_len = seq_len

print(f"\nBest LSTM Config: {best_lstm_config}")
print(f"Best LSTM Validation R²: {best_lstm_val_r2:.4f}")

# ===============================
# 2. Generate Best LSTM Predictions
# ===============================
print("\nStep 2: Generating predictions with best LSTM...")

# Use best configuration
X_train_seq, y_train_seq = create_sequences(X_train_lstm, y_train_scaled, best_seq_len)
X_test_seq, y_test_seq = create_sequences(X_test_lstm, y_test_scaled, best_seq_len)

lstm_train_pred_scaled = best_lstm_model.predict(X_train_seq, verbose=0)
lstm_test_pred_scaled = best_lstm_model.predict(X_test_seq, verbose=0)

lstm_train_pred = scaler_target.inverse_transform(lstm_train_pred_scaled).flatten()
lstm_test_pred = scaler_target.inverse_transform(lstm_test_pred_scaled).flatten()

In [ ]:
# ===============================
# 3. XGBoost Hyperparameter Tuning
# ===============================
print("\nStep 3: Tuning XGBoost component with regularization...")

# Prepare hybrid features with best LSTM
# Use remaining features (excluding LSTM features)
xgb_feature_cols = [col for col in X_train_scaled.columns if col not in available_lstm_features]
X_train_xgb_aligned = X_train_scaled[xgb_feature_cols].iloc[best_seq_len:].reset_index(drop=True)
X_test_xgb_aligned = X_test_scaled[xgb_feature_cols].iloc[best_seq_len:].reset_index(drop=True)

X_train_hybrid = X_train_xgb_aligned.copy()
X_test_hybrid = X_test_xgb_aligned.copy()
X_train_hybrid['lstm_prediction'] = lstm_train_pred
X_test_hybrid['lstm_prediction'] = lstm_test_pred

y_train_aligned = y_train.iloc[best_seq_len:].values
y_test_aligned = y_test.iloc[best_seq_len:].values

# Regularized parameter grid
xgb_param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.08],
    'max_depth': [3, 4, 5],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [1.0, 2.0, 5.0],
    'reg_lambda': [1.0, 2.0, 5.0],
    'min_child_weight': [3, 5, 7],
    'gamma': [0.1, 0.2, 0.5]
}

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=2)

xgb_base = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=1
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_grid,
    scoring='r2',
    cv=tscv,
    n_iter=15,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

print("Tuning XGBoost with regularization...")
xgb_search.fit(X_train_hybrid, y_train_aligned)

print(f"Best XGBoost CV R²: {xgb_search.best_score_:.4f}")
print("Best XGBoost Parameters:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# ===============================
# 4. Final Tuned Hybrid Model
# ===============================
print("\nStep 4: Training final tuned hybrid model...")

tuned_hybrid = xgb_search.best_estimator_
tuned_hybrid.fit(X_train_hybrid, y_train_aligned)

# Predictions
tuned_train_pred = tuned_hybrid.predict(X_train_hybrid)
tuned_test_pred = tuned_hybrid.predict(X_test_hybrid)

# Evaluation function
def evaluate_tuned_model(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    mask = y_true != 0
    if np.sum(mask) > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.inf

    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'Name': name}

# Calculate metrics
tuned_train_metrics = evaluate_tuned_model(y_train_aligned, tuned_train_pred, "Tuned Train")
tuned_test_metrics = evaluate_tuned_model(y_test_aligned, tuned_test_pred, "Tuned Test")
lstm_only_metrics = evaluate_tuned_model(y_test_aligned, lstm_test_pred, "LSTM Only")

# ===============================
# 5. Results Comparison
# ===============================
print("\n" + "="*70)
print("TUNED HYBRID MODEL RESULTS")
print("="*70)

print(f"\nModel Performance Comparison:")
print(f"{'Model':<15} {'R²':<8} {'RMSE':<8} {'MAE':<8} {'MAPE%':<8}")
print("-" * 55)
print(f"{'LSTM Only':<15} {lstm_only_metrics['R2']:<8.4f} {lstm_only_metrics['RMSE']:<8.4f} {lstm_only_metrics['MAE']:<8.4f} {lstm_only_metrics['MAPE']:<8.2f}")
print(f"{'Tuned Train':<15} {tuned_train_metrics['R2']:<8.4f} {tuned_train_metrics['RMSE']:<8.4f} {tuned_train_metrics['MAE']:<8.4f} {tuned_train_metrics['MAPE']:<8.2f}")
print(f"{'Tuned Test':<15} {tuned_test_metrics['R2']:<8.4f} {tuned_test_metrics['RMSE']:<8.4f} {tuned_test_metrics['MAE']:<8.4f} {tuned_test_metrics['MAPE']:<8.2f}")

In [ ]:
# Overfitting check
r2_gap = tuned_train_metrics['R2'] - tuned_test_metrics['R2']
print(f"\nOverfitting Analysis:")
print(f"Training R²: {tuned_train_metrics['R2']:.4f}")
print(f"Testing R²: {tuned_test_metrics['R2']:.4f}")
print(f"R² Gap: {r2_gap:.4f}")

if r2_gap < 0.05:
    print("No significant overfitting detected")
elif r2_gap < 0.15:
    print("Mild overfitting - acceptable")
else:
    print("Significant overfitting detected")

print("\nHybrid model hyperparameter tuning completed!")

In [ ]:
# ===============================
# COMPREHENSIVE VISUALIZATIONS
# ===============================
print("\n" + "="*70)
print("GENERATING COMPREHENSIVE VISUALIZATIONS")
print("="*70)

plt.style.use('default')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Hybrid Model Performance Analysis', fontsize=16, fontweight='bold')

# 1. Predictions vs Actual (Test Set)
ax1 = axes[0, 0]
ax1.scatter(y_test_aligned, tuned_test_pred, alpha=0.6, color='blue', label='Hybrid Model', s=30)
ax1.scatter(y_test_aligned, lstm_test_pred, alpha=0.4, color='red', label='LSTM Only', s=20)

min_val = min(y_test_aligned.min(), tuned_test_pred.min())
max_val = max(y_test_aligned.max(), tuned_test_pred.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.7, linewidth=2)

ax1.set_xlabel('Actual Log USD Price')
ax1.set_ylabel('Predicted Log USD Price')
ax1.set_title('Predictions vs Actual (Test Set)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax1.text(0.05, 0.95, f'Hybrid R²: {tuned_test_metrics["R2"]:.3f}',
         transform=ax1.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='blue', alpha=0.2))
ax1.text(0.05, 0.85, f'LSTM R²: {lstm_only_metrics["R2"]:.3f}',
         transform=ax1.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='red', alpha=0.2))

# 2. Residuals Plot (Test Set)
ax2 = axes[0, 1]
residuals_hybrid = y_test_aligned - tuned_test_pred
residuals_lstm = y_test_aligned - lstm_test_pred

ax2.scatter(tuned_test_pred, residuals_hybrid, alpha=0.6, color='blue', label='Hybrid Residuals', s=30)
ax2.scatter(lstm_test_pred, residuals_lstm, alpha=0.4, color='red', label='LSTM Residuals', s=20)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.7)
ax2.set_xlabel('Predicted Log USD Price')
ax2.set_ylabel('Residuals (Actual - Predicted)')
ax2.set_title('Residuals Plot (Test Set)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Feature Importance (Top 10) - Gain-based, Normalized 0-1
ax3 = axes[1, 0]
try:
    # Get gain-based importance
    importance_dict = tuned_hybrid.get_booster().get_score(importance_type='gain')
    feature_names = list(X_train_hybrid.columns)
    feature_importance = [importance_dict.get(f, 0) for f in feature_names]

    # Normalize between 0 and 1
    importance_array = np.array(feature_importance)
    importance_normalized = (importance_array - importance_array.min()) / (importance_array.max() - importance_array.min())

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance_normalized
    }).sort_values('importance', ascending=True).tail(10)

    ax3.barh(range(len(importance_df)), importance_df['importance'], color='skyblue')
    ax3.set_yticks(range(len(importance_df)))
    ax3.set_yticklabels([f.replace('_', ' ').title() for f in importance_df['feature']], fontsize=8)
    ax3.set_xlabel('Feature Importance (Gain, Normalized 0-1)')
    ax3.set_title('Top 10 Feature Importance - Gain (0-1)')
    ax3.grid(True, alpha=0.3, axis='x')

except Exception as e:
    # Fallback to default importance if gain fails
    if hasattr(tuned_hybrid, 'feature_importances_'):
        feature_names = list(X_train_hybrid.columns)
        feature_importance = tuned_hybrid.feature_importances_

        # Normalize between 0 and 1
        importance_normalized = (feature_importance - feature_importance.min()) / (feature_importance.max() - feature_importance.min())

        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importance_normalized
        }).sort_values('importance', ascending=True).tail(10)

        ax3.barh(range(len(importance_df)), importance_df['importance'], color='lightcoral')
        ax3.set_yticks(range(len(importance_df)))
        ax3.set_yticklabels([f.replace('_', ' ').title() for f in importance_df['feature']], fontsize=8)
        ax3.set_xlabel('Feature Importance (Default, Normalized 0-1)')
        ax3.set_title('Top 10 Feature Importance - Default (0-1)')
        ax3.grid(True, alpha=0.3, axis='x')
    else:
        ax3.text(0.5, 0.5, 'Feature importance\nnot available',
                 transform=ax3.transAxes, ha='center', va='center', fontsize=12)
        ax3.set_title('Feature Importance (N/A)')

# 4. Model Comparison Metrics
ax4 = axes[1, 1]
models = ['LSTM Only', 'Hybrid Tuned']
r2_scores = [lstm_only_metrics['R2'], tuned_test_metrics['R2']]
mape_scores = [lstm_only_metrics['MAPE'], tuned_test_metrics['MAPE']]

x = np.arange(len(models))
width = 0.35

bars1 = ax4.bar(x - width/2, r2_scores, width, label='R² Score', color='lightblue')
ax4_twin = ax4.twinx()
bars2 = ax4_twin.bar(x + width/2, mape_scores, width, label='MAPE %', color='lightcoral')

ax4.set_xlabel('Models')
ax4.set_ylabel('R² Score', color='blue')
ax4_twin.set_ylabel('MAPE %', color='red')
ax4.set_title('Model Performance Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(models)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print(f"\n{'='*70}")
print("HYBRID MODEL TUNING COMPLETE")
print("="*70)

In [ ]:
# Hybrid Model Final Analysis and Visualization - Fixed

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ===============================
# Overfitting Analysis
# ===============================
r2_gap = tuned_train_metrics['R2'] - tuned_test_metrics['R2']
mape_gap = tuned_train_metrics['MAPE'] - tuned_test_metrics['MAPE']

print(f"\nOverfitting Analysis:")
print(f"R² Gap: {r2_gap:.4f}")
print(f"MAPE Gap: {mape_gap:.2f}%")

if r2_gap < 0.05:
    print("Good generalization (R²)")
elif r2_gap < 0.1:
    print("Moderate overfitting (R²)")
else:
    print("Still overfitting (R²) - consider more regularization")

if abs(mape_gap) < 5:
    print("Good generalization (MAPE)")
elif abs(mape_gap) < 10:
    print("Moderate overfitting (MAPE)")
else:
    print("Still overfitting (MAPE) - consider more regularization")

# ===============================
# Model Improvement Analysis
# ===============================
r2_improvement = tuned_test_metrics['R2'] - lstm_only_metrics['R2']
mape_improvement = lstm_only_metrics['MAPE'] - tuned_test_metrics['MAPE']

print(f"\nHybrid vs LSTM Comparison:")
print(f"R² Improvement: {r2_improvement:+.4f}")
print(f"MAPE Improvement: {mape_improvement:+.2f}% (negative means hybrid is worse)")

if r2_improvement > 0 and mape_improvement > 0:
    print("Tuned hybrid model improved on both metrics!")
elif r2_improvement > 0 or mape_improvement > 0:
    print("Tuned hybrid model improved on one metric")
else:
    print("Hybrid still not better - consider ensemble methods")

# ===============================
# MAPE Performance Assessment
# ===============================
print(f"\nMAPE Performance Assessment:")
test_mape = tuned_test_metrics['MAPE']
if test_mape < 10:
    print(f"Excellent accuracy (MAPE: {test_mape:.2f}%)")
elif test_mape < 20:
    print(f"Good accuracy (MAPE: {test_mape:.2f}%)")
elif test_mape < 50:
    print(f"Reasonable accuracy (MAPE: {test_mape:.2f}%)")
else:
    print(f"Poor accuracy (MAPE: {test_mape:.2f}%)")

# ===============================
# Enhanced Recommendations
# ===============================
print(f"\nRecommendations based on results:")

if tuned_test_metrics['R2'] > lstm_only_metrics['R2'] and tuned_test_metrics['MAPE'] < lstm_only_metrics['MAPE']:
    print("- Use tuned hybrid model (best on both R² and MAPE)")
elif tuned_test_metrics['R2'] > lstm_only_metrics['R2']:
    print("- Use tuned hybrid model (better R² but check MAPE trade-off)")
elif tuned_test_metrics['MAPE'] < lstm_only_metrics['MAPE']:
    print("- Use tuned hybrid model (better MAPE but check R² trade-off)")
else:
    print("- Stick with standalone LSTM")
    print("- Consider different architectures (CNN-LSTM, Transformer)")

print("- Try ensemble methods (voting, stacking)")
print("- Experiment with feature selection")
print("- Consider different time horizons")

# ===============================
# Feature Importance Analysis
# ===============================
try:
    # Get gain-based importance
    importance_dict = tuned_hybrid.get_booster().get_score(importance_type='gain')
    feature_names = list(X_train_hybrid.columns)
    feature_importance = [importance_dict.get(f, 0) for f in feature_names]

    # Normalize between 0 and 1
    importance_array = np.array(feature_importance)
    importance_normalized = (importance_array - importance_array.min()) / (importance_array.max() - importance_array.min())

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance_normalized
    }).sort_values('importance', ascending=False)

    print(f"\nTop 5 Most Important Features (Gain-based, Normalized 0-1):")
    for i, (_, row) in enumerate(importance_df.head().iterrows(), 1):
        print(f"  {i}. {row['feature']}: {row['importance']:.4f}")

except Exception as e:
    print(f"Warning: Could not get gain-based importance: {e}")
    # Fallback to default with normalization
    feature_names = list(X_train_hybrid.columns)
    feature_importance = tuned_hybrid.feature_importances_
    importance_normalized = (feature_importance - feature_importance.min()) / (feature_importance.max() - feature_importance.min())

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance_normalized
    }).sort_values('importance', ascending=False)

    print(f"\nTop 5 Most Important Features (Default, Normalized 0-1):")
    for i, (_, row) in enumerate(importance_df.head().iterrows(), 1):
        print(f"  {i}. {row['feature']}: {row['importance']:.4f}")

# Final Model Comparison

In [ ]:
# ==============================
# Enhanced Simple Model Comparison
# Somalia Food Price Forecasting
# ==============================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Enhanced Metrics Function ---
def model_metrics(y_true, y_pred, name, mtype):
    """Calculate comprehensive metrics with error handling"""
    try:
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Robust MAPE calculation
        mask = y_true != 0
        if mask.any():
            mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        else:
            mape = np.inf

        return {'Model': name, 'Type': mtype, 'R²': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

    except Exception as e:
        print(f"Error calculating metrics for {name}: {e}")
        return {'Model': name, 'Type': mtype, 'R²': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan}

# --- Performance Grading Function ---
def performance_grade(r2_score):
    """Assign performance grade based on R² score"""
    if r2_score > 0.95: return "A+"
    elif r2_score > 0.90: return "A"
    elif r2_score > 0.85: return "A-"
    elif r2_score > 0.80: return "B+"
    elif r2_score > 0.75: return "B"
    elif r2_score > 0.70: return "B-"
    elif r2_score > 0.60: return "C"
    else: return "D"

# --- Collect Results (with error handling) ---
def collect_model_results():
    """Collect all model results with error handling"""
    results = []

    ## Define model configurations with corrected names
    model_configs = [
        ('y_test_pred_lgb', "LightGBM (Basic)", "Gradient Boosting", 'y_test'),
        ('y_test_pred_lgb_tuned', "LightGBM (Tuned)", "Gradient Boosting", 'y_test'),
        ('y_test_pred_xgb', "XGBoost (Basic)", "Gradient Boosting", 'y_test'),
        ('y_test_pred_xgb_tuned', "XGBoost (Tuned)", "Gradient Boosting", 'y_test'),
        ('y_test_pred_rf', "Random Forest (Basic)", "Tree-based", 'y_test'),
        ('y_test_pred_tuned', "Random Forest (Tuned)", "Tree-based", 'y_test'),

        # Deep Learning / Hybrid
        ('lstm_test_pred', "LSTM (Basic)", "Deep Learning", 'y_test_lstm_aligned'),
        ('lstm_test_pred', "LSTM (Tuned)", "Deep Learning", 'y_test_aligned'),
        ('hybrid_test_pred', "Hybrid LSTM+XGBoost (Basic)", "Hybrid", 'y_test_lstm_aligned'),
        ('tuned_test_pred', "Hybrid LSTM+XGBoost (Tuned)", "Hybrid", 'y_test_aligned')
    ]

    for pred_var, name, mtype, y_var in model_configs:
        try:
            # Get predictions and actuals
            y_pred = globals().get(pred_var)
            y_true = globals().get(y_var)

            if y_pred is not None and y_true is not None:
                # Ensure arrays are the same length
                min_len = min(len(y_true), len(y_pred))
                y_true_adj = y_true[:min_len] if hasattr(y_true, '__len__') else y_true
                y_pred_adj = y_pred[:min_len] if hasattr(y_pred, '__len__') else y_pred

                result = model_metrics(y_true_adj, y_pred_adj, name, mtype)
                results.append(result)
                print(f"✅ {name} - metrics calculated successfully")
            else:
                print(f"⚠️ {name} - missing predictions or actuals")

        except Exception as e:
            print(f"❌ {name} - error: {e}")

    return results

# --- Enhanced Display Function ---
def display_results(df):
    """Display comprehensive results table and analysis"""

    print("\n" + "="*90)
    print("SOMALIA FOOD PRICE FORECASTING - MODEL PERFORMANCE COMPARISON")
    print("="*90)

    # Main comparison table
    print(f"{'Rank':<4} {'Model':<30} {'Type':<18} {'Grade':<5} {'R²':<8} {'RMSE':<10} {'MAE':<10} {'MAPE%':<8}")
    print("-"*95)

    for i, (_, row) in enumerate(df.iterrows(), 1):
        grade = performance_grade(row['R²'])
        mape_str = f"{row['MAPE']:.2f}" if row['MAPE'] != np.inf else "N/A"
        print(f"{i:<4} {row['Model']:<30} {row['Type']:<18} {grade:<5} {row['R²']:<8.4f} {row['RMSE']:<10.4f} {row['MAE']:<10.4f} {mape_str:<8}")

    # Champion model analysis
    best = df.iloc[0]
    worst = df.iloc[-1]

    print(f"\n{'='*50}")
    print("PERFORMANCE ANALYSIS")
    print("="*50)

    print(f"🏆 CHAMPION MODEL: {best['Model']}")
    print(f"   Grade: {performance_grade(best['R²'])} | R²: {best['R²']:.4f} | RMSE: {best['RMSE']:.4f} | MAPE: {best['MAPE']:.2f}%")

    print(f"\n❌ WORST PERFORMER: {worst['Model']}")
    print(f"   Grade: {performance_grade(worst['R²'])} | R²: {worst['R²']:.4f} | RMSE: {worst['RMSE']:.4f} | MAPE: {worst['MAPE']:.2f}%")

    # Performance insights
    print(f"\n📊 KEY INSIGHTS:")
    print(f"   • Performance Range: R² from {df['R²'].min():.4f} to {df['R²'].max():.4f}")
    print(f"   • Best RMSE: {df['RMSE'].min():.4f}")

    valid_mape = df[df['MAPE'] != np.inf]['MAPE']
    if len(valid_mape) > 0:
        print(f"   • Best MAPE: {valid_mape.min():.2f}%")

    # Model type analysis
    type_stats = df.groupby('Type')['R²'].agg(['count', 'mean', 'max']).round(4)
    print(f"\n🔍 MODEL TYPE ANALYSIS:")
    for mtype, stats in type_stats.iterrows():
        print(f"   • {mtype}: {stats['count']} models, Avg R²: {stats['mean']:.4f}, Best: {stats['max']:.4f}")

    # Deployment recommendation
    print(f"\n🎯 DEPLOYMENT RECOMMENDATION:")
    if best['R²'] > 0.95:
        print(f"   ✅ {best['Model']} is READY for production deployment")
        print(f"   Outstanding performance (>95% variance explained)")
    elif best['R²'] > 0.90:
        print(f"   ✅ {best['Model']} is APPROVED for production")
        print(f"   Excellent performance (>90% variance explained)")
    elif best['R²'] > 0.80:
        print(f"   ⚠️  {best['Model']} needs monitoring in production")
        print(f"   Good performance but watch for model drift")
    else:
        print(f"   ❌ Models need improvement before production")
        print(f"   Consider additional feature engineering or data")

# --- Main Execution ---
if __name__ == "__main__":
    print("Starting model comparison analysis...")

    # Try to collect results dynamically
    try:
        results = collect_model_results()

        if results:
            # Create comparison dataframe
            df = pd.DataFrame(results).sort_values("R²", ascending=False).reset_index(drop=True)

            # Display results
            display_results(df)

            # Quick summary
            print(f"\n{'='*50}")
            print(f"SUMMARY: {len(df)} models compared")
            print(f"Winner: {df.iloc[0]['Model']} (R² = {df.iloc[0]['R²']:.4f})")
            print("="*50)

        else:
            print("⚠️ No model results found. Make sure your model predictions are available in the global namespace.")

    except Exception as e:
        print(f"Error in analysis: {e}")
        print("\nFalling back to your original approach...")

        # --- Fallback Approach ---
        try:
            results = [
                model_metrics(y_test, y_test_pred_lgb, "LightGBM (Basic)", "Gradient Boosting"),
                model_metrics(y_test, y_test_pred_lgb_tuned, "LightGBM (Tuned)", "Gradient Boosting"),
                model_metrics(y_test, y_test_pred_xgb, "XGBoost (Basic)", "Gradient Boosting"),
                model_metrics(y_test, y_test_pred_xgb_tuned, "XGBoost (Tuned)", "Gradient Boosting"),
                model_metrics(y_test, y_test_pred_rf, "Random Forest (Basic)", "Tree-based"),
                model_metrics(y_test, y_test_pred_tuned, "Random Forest (Tuned)", "Tree-based"),

                # Deep Learning / Hybrid
                model_metrics(y_test_lstm_aligned, lstm_test_pred, "LSTM (Basic)", "Deep Learning"),
                model_metrics(y_test_aligned, lstm_test_pred, "LSTM (Tuned)", "Deep Learning"),
                model_metrics(y_test_lstm_aligned, hybrid_test_pred, "Hybrid LSTM+XGBoost (Basic)", "Hybrid"),
                model_metrics(y_test_aligned, tuned_test_pred, "Hybrid LSTM+XGBoost (Tuned)", "Hybrid")
            ]

            df = pd.DataFrame(results).sort_values("R²", ascending=False).reset_index(drop=True)
            display_results(df)

        except Exception as e2:
            print(f"Fallback also failed: {e2}")
            print("Please ensure all prediction variables are available.")

In [ ]:
# ===============================
# Model Comparison Visualization
# ===============================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Somalia Food Price Forecasting - Model Performance Analysis', fontsize=16, fontweight='bold')

# Extract data from comparison results
models = df['Model'].values
r2_scores = df['R²'].values
rmse_scores = df['RMSE'].values
mape_scores = df['MAPE'].values
model_types = df['Type'].values

# Plot 1: R² Comparison
axes[0,0].barh(models, r2_scores, color=['green' if r2 > 0.9 else 'orange' if r2 > 0.8 else 'red' for r2 in r2_scores])
axes[0,0].set_xlabel('R² Score')
axes[0,0].set_title('Model R² Comparison')
axes[0,0].set_xlim(0, 1)
axes[0,0].grid(True, alpha=0.3)

# Plot 2: RMSE Comparison
axes[0,1].barh(models, rmse_scores, color='skyblue', alpha=0.7)
axes[0,1].set_xlabel('RMSE')
axes[0,1].set_title('Model RMSE Comparison (Lower is Better)')
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Model Type Performance
type_performance = df.groupby('Type')['R²'].mean().sort_values(ascending=True)
axes[1,0].barh(type_performance.index, type_performance.values, color='lightcoral', alpha=0.7)
axes[1,0].set_xlabel('Average R² Score')
axes[1,0].set_title('Performance by Model Type')
axes[1,0].grid(True, alpha=0.3)

# Plot 4: R² vs MAPE Scatter
colors = {'Tree-based': 'green', 'Gradient Boosting': 'blue', 'Deep Learning': 'red', 'Hybrid': 'orange'}
for model_type in df['Type'].unique():
    mask = df['Type'] == model_type
    axes[1,1].scatter(df[mask]['R²'], df[mask]['MAPE'],
                     c=colors.get(model_type, 'gray'),
                     label=model_type, alpha=0.7, s=100)

axes[1,1].set_xlabel('R² Score')
axes[1,1].set_ylabel('MAPE (%)')
axes[1,1].set_title('Model Performance Trade-off (R² vs MAPE)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Highlight champion model
champion_idx = df['R²'].idxmax()
champion_r2 = df.loc[champion_idx, 'R²']
champion_mape = df.loc[champion_idx, 'MAPE']
axes[1,1].scatter(champion_r2, champion_mape, c='gold', s=200, marker='*',
                 edgecolors='black', linewidth=2, label='Champion', zorder=5)

plt.tight_layout()
plt.show()

# Print champion model info
print(f"\n🏆 CHAMPION MODEL VISUALIZATION:")
print(f"   Model: {df.iloc[0]['Model']}")
print(f"   R²: {df.iloc[0]['R²']:.4f}")
print(f"   RMSE: {df.iloc[0]['RMSE']:.4f}")
print(f"   MAPE: {df.iloc[0]['MAPE']:.2f}%")

In [ ]:
# Variable Availability Check for Alert System
# Run this to see what data you have available

import pandas as pd
import numpy as np

print("CHECKING VARIABLE AVAILABILITY FOR ALERT SYSTEM")
print("=" * 60)

# Check what variables exist in your environment
variables_to_check = [
    # Original datasets
    'dataset_clean',
    'final_data',
    'dataset',

    # Split datasets
    'train_raw',
    'test_raw',
    'train_features',
    'test_features',
    'train_encoded',
    'test_encoded',

    # Final processed datasets
    'X_train_scaled',
    'X_test_scaled',
    'y_train',
    'y_test',

    # Model predictions
    'y_test_pred_rf',

    # Alert system requirements
    'selected_features'
]

print("1. CHECKING VARIABLE EXISTENCE")
print("-" * 40)
available_vars = {}
for var_name in variables_to_check:
    try:
        var_value = globals().get(var_name)
        if var_value is not None:
            if hasattr(var_value, 'shape'):
                print(f"✅ {var_name}: Available - Shape: {var_value.shape}")
                available_vars[var_name] = var_value
            elif hasattr(var_value, '_len_'):
                print(f"✅ {var_name}: Available - Length: {len(var_value)}")
                available_vars[var_name] = var_value
            else:
                print(f"✅ {var_name}: Available")
                available_vars[var_name] = var_value
        else:
            print(f"❌ {var_name}: Not available")
    except Exception as e:
        print(f"❌ {var_name}: Error - {e}")

print(f"\n2. ALERT SYSTEM REQUIREMENTS CHECK")
print("-" * 40)

# Required columns for alert system
required_cols = [
    'commodity_encoded',
    'market_encoded',
    'latitude',
    'longitude',
    'year',
    'month'
]

print("Required columns for alert system:")
for col in required_cols:
    print(f"  - {col}")

# Check if we can find these columns in available datasets
print(f"\n3. COLUMN AVAILABILITY CHECK")
print("-" * 40)

datasets_to_check = ['test_encoded', 'test_features', 'dataset_clean', 'dataset']
found_dataset = None

for dataset_name in datasets_to_check:
    if dataset_name in available_vars:
        dataset = available_vars[dataset_name]
        if hasattr(dataset, 'columns'):
            missing_cols = [col for col in required_cols if col not in dataset.columns]
            available_cols = [col for col in required_cols if col in dataset.columns]

            print(f"\n{dataset_name} (Shape: {dataset.shape}):")
            print(f"  Available columns: {available_cols}")
            if missing_cols:
                print(f"  Missing columns: {missing_cols}")
            else:
                print(f"  ✅ ALL REQUIRED COLUMNS FOUND!")
                found_dataset = dataset_name
                break

# Check predictions availability
print(f"\n4. PREDICTIONS CHECK")
print("-" * 40)

if 'y_test_pred_rf' in available_vars:
    pred = available_vars['y_test_pred_rf']
    print(f"✅ Random Forest predictions available")
    print(f"   Shape/Length: {pred.shape if hasattr(pred, 'shape') else len(pred)}")
    print(f"   Sample values: {pred[:5] if hasattr(pred, '_getitem_') else 'Cannot display'}")
    print(f"   Note: These are in LOG SCALE - need to convert with np.exp()")
else:
    print(f"❌ Random Forest predictions not found")

# Check if we can reconstruct what we need
print(f"\n5. RECONSTRUCTION STRATEGY")
print("-" * 40)

if found_dataset:
    print(f"✅ SOLUTION FOUND:")
    print(f"   Use '{found_dataset}' dataset which has all required columns")
    print(f"   Can create predictions dataframe for alert system")

    # Show how to align data
    dataset = available_vars[found_dataset]
    test_data = available_vars.get('X_test_scaled')
    if test_data is not None:
        print(f"   {found_dataset} shape: {dataset.shape}")
        print(f"   X_test_scaled shape: {test_data.shape}")
        print(f"   Need to align indices for proper mapping")
else:
    print(f"❌ PROBLEM:")
    print(f"   No dataset found with all required columns")
    print(f"   Need to reconstruct from available data")

print(f"\n6. NEXT STEPS RECOMMENDATION")
print("-" * 40)

if found_dataset and 'y_test_pred_rf' in available_vars:
    print(f"✅ READY TO PROCEED:")
    print(f"   1. Use {found_dataset} to get required columns")
    print(f"   2. Convert y_test_pred_rf from log scale: np.exp(y_test_pred_rf)")
    print(f"   3. Create predictions dataframe")
    print(f"   4. Run alert system")
elif 'y_test_pred_rf' in available_vars:
    print(f"⚠ PARTIAL SOLUTION:")
    print(f"   Have predictions but missing metadata columns")
    print(f"   Need to reconstruct from original data")
else:
    print(f"❌ NEED TO:")
    print(f"   1. Re-run Random Forest model to get predictions")
    print(f"   2. Ensure original data with metadata is available")

print(f"\n" + "=" * 60)
print("DIAGNOSTIC COMPLETE")
print("=" * 60)

# 90th percentile thresholding

In [ ]:
# ============================================================================
# COMPLETE ALERT SYSTEM - STEP BY STEP IMPLEMENTATION
# ============================================================================

# CELL 1: Import Libraries and Basic Functions
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

def get_current_season(row):
    """Extract current season from month"""
    month = row.get('month', 1)
    if month in [4, 5, 6]:  # April-June
        return 'Gu'
    elif month in [10, 11, 12]:  # October-December
        return 'Deyr'
    elif month in [1, 2, 3]:  # January-March
        return 'Jilaal'
    else:  # July-September
        return 'Xagaa'

print("✅ Libraries and functions loaded successfully!")

# ============================================================================
# CELL 2: Create Predictions Dataset
# ============================================================================
print("📊 Creating predictions dataset for alert system...")

# Convert predictions from log scale to actual prices
predicted_prices = np.exp(y_test_pred_rf)

# Get required columns from test_encoded
predictions_df = test_encoded[['commodity_encoded', 'market_encoded', 'latitude',
                              'longitude', 'year', 'month']].copy()

# Add predictions and current season
predictions_df['predicted_price'] = predicted_prices
predictions_df['current_season'] = predictions_df.apply(get_current_season, axis=1)

print(f"✅ Predictions dataset created: {predictions_df.shape}")
print(f"Price range: ${predictions_df['predicted_price'].min():.2f} - ${predictions_df['predicted_price'].max():.2f}")
print("Sample predictions:")
print(predictions_df.head(3))

In [ ]:
# ============================================================================
# CELL 2: Create District-Level Predictions Dataset (SAME AS BEFORE)
# ============================================================================
print("Creating district-level predictions dataset for alert system...")

# Convert predictions from log scale to actual prices
predicted_prices = np.exp(y_test_pred_rf)

# Get required columns from test_encoded (keeping district_encoded)
predictions_df = test_encoded[['commodity_encoded', 'market_encoded', 'district_encoded',
                              'latitude', 'longitude', 'year', 'month']].copy()

# Add predictions and current season
predictions_df['predicted_price'] = predicted_prices
predictions_df['current_season'] = predictions_df.apply(get_current_season, axis=1)

print(f"District-level predictions dataset created: {predictions_df.shape}")
print(f"Price range: ${predictions_df['predicted_price'].min():.2f} - ${predictions_df['predicted_price'].max():.2f}")
print("Sample predictions:")
print(predictions_df.head(3))

In [ ]:
# ============================================================================
# CELL 2B: CREATE DISTRICT-TO-REGION MAPPING
# ============================================================================
def create_district_region_mapping(original_dataset):
    """Create mapping from district codes to region codes and names using original data"""
    print("Creating district-to-region mapping from original data...")

    if 'district' in original_dataset.columns and 'region' in original_dataset.columns:
        # Create mapping from original data
        mapping_df = original_dataset[['district', 'region']].drop_duplicates()

        # Get unique district names and create encoding mapping
        unique_districts = sorted(original_dataset['district'].unique())
        district_to_code = {district: idx for idx, district in enumerate(unique_districts)}

        # Get unique region names and create encoding mapping
        unique_regions = sorted(original_dataset['region'].unique())
        region_to_code = {region: idx for idx, region in enumerate(unique_regions)}

        # Create district_encoded to region_encoded mapping
        district_region_map = {}
        for _, row in mapping_df.iterrows():
            district_code = district_to_code[row['district']]
            region_code = region_to_code[row['region']]
            district_region_map[district_code] = {
                'region_encoded': region_code,
                'region_name': row['region'],
                'district_name': row['district']
            }

        print(f"Successfully created mapping for {len(district_region_map)} districts")
        print(f"Districts map to {len(unique_regions)} regions")

        # Show sample mapping
        print("Sample district-region mapping:")
        for i, (dist_code, info) in enumerate(list(district_region_map.items())[:5]):
            print(f"  District {dist_code} ({info['district_name']}) -> Region {info['region_encoded']} ({info['region_name']})")

        return district_region_map, region_to_code

    else:
        print("Error: Original dataset missing 'district' or 'region' columns")
        print(f"Available columns: {list(original_dataset.columns)}")
        return {}, {}

# Create the mapping using your original dataset
print("Creating district-to-region mapping...")
try:
    # Try to use your original dataset
    original_data = None
    for dataset_name in ['dataset']:
        if dataset_name in locals():
            original_data = locals()[dataset_name]
            print(f"Using {dataset_name} for district-region mapping")
            break

    if original_data is not None:
        district_region_map, region_names = create_district_region_mapping(original_data)
    else:
        print("No original dataset found - using default mapping")
        # You'll need to provide the actual Somalia district-region mapping here
        district_region_map = {}
        region_names = {}

except Exception as e:
    print(f"Error creating mapping: {e}")
    district_region_map = {}
    region_names = {}

In [ ]:
# ============================================================================
#  ADD REGION INFO TO PREDICTIONS
# ============================================================================
def add_region_to_predictions(predictions_df, district_region_map):
    """Add region information to district-level predictions"""
    print("Adding region information to predictions...")

    if len(district_region_map) == 0:
        print("Warning: No district-region mapping available")
        # Create dummy region mapping for testing
        predictions_df['region_encoded'] = predictions_df['district_encoded'] % 3  # Dummy mapping
        predictions_df['region_name'] = predictions_df['region_encoded'].map({0: 'Region_A', 1: 'Region_B', 2: 'Region_C'})
    else:
        # Map districts to regions
        predictions_df['region_encoded'] = predictions_df['district_encoded'].map(
            lambda x: district_region_map.get(x, {}).get('region_encoded', 0)
        )
        predictions_df['region_name'] = predictions_df['district_encoded'].map(
            lambda x: district_region_map.get(x, {}).get('region_name', 'Unknown')
        )

    print(f"Added region info - {predictions_df['region_encoded'].nunique()} unique regions")
    return predictions_df

# Add region information to predictions
predictions_df = add_region_to_predictions(predictions_df, district_region_map)

print("Updated predictions dataset:")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")
print("\nSample with region info:")
print(predictions_df[['district_encoded', 'region_encoded', 'region_name', 'predicted_price']].head())

In [ ]:
# ============================================================================
#   Data Consistency Validation
# ============================================================================
def validate_data_consistency_with_regions(predictions_df, model_features=None):
    """Validate that prediction data includes both district and region info"""
    print("VALIDATING DATA CONSISTENCY WITH REGION MAPPING...")
    print("=" * 50)

    # Check basic structure
    print(f"Predictions dataset shape: {predictions_df.shape}")
    print(f"Required columns present: {list(predictions_df.columns)}")

    # Validate encoding ranges
    print(f"\nEncoding validation:")
    for col in ['commodity_encoded', 'market_encoded', 'district_encoded', 'region_encoded']:
        if col in predictions_df.columns:
            min_val, max_val = predictions_df[col].min(), predictions_df[col].max()
            unique_count = predictions_df[col].nunique()
            print(f"  {col}: range {min_val}-{max_val}, {unique_count} unique values")

    # Validate region mapping
    if 'region_name' in predictions_df.columns:
        unique_regions = predictions_df['region_name'].unique()
        print(f"\nRegion validation:")
        print(f"  Unique regions: {len(unique_regions)}")
        for region in unique_regions[:10]:  # Show first 10
            count = len(predictions_df[predictions_df['region_name'] == region])
            print(f"    {region}: {count} predictions")
        if len(unique_regions) > 10:
            print(f"    ... and {len(unique_regions)-10} more regions")

    # Check for missing values
    missing_counts = predictions_df.isnull().sum()
    if missing_counts.sum() > 0:
        print(f"\nMissing values detected:")
        for col, count in missing_counts[missing_counts > 0].items():
            print(f"    {col}: {count} missing")
    else:
        print(f"\nNo missing values found")

    return True

# Run updated data consistency validation
print("Running data consistency validation with regions...")
try:
    validation_result = validate_data_consistency_with_regions(predictions_df)
    print("Data consistency check completed!\n")
except Exception as e:
    print(f"Validation error: {e}")

In [ ]:
# ============================================================================
#  Historical Data Preparation (District-based thresholds)
# ============================================================================
print("Preparing historical data for district-based thresholds...")

# Create historical dataset with actual prices (keeping district_encoded)
historical_df = test_encoded.copy()
historical_df['usdprice'] = np.exp(test_encoded['log_usd_price'])
historical_df['current_season'] = historical_df.apply(get_current_season, axis=1)

# Add region mapping to historical data too
if len(district_region_map) > 0:
    historical_df['region_encoded'] = historical_df['district_encoded'].map(
        lambda x: district_region_map.get(x, {}).get('region_encoded', 0)
    )
    historical_df['region_name'] = historical_df['district_encoded'].map(
        lambda x: district_region_map.get(x, {}).get('region_name', 'Unknown')
    )
else:
    historical_df['region_encoded'] = historical_df['district_encoded'] % 3
    historical_df['region_name'] = historical_df['region_encoded'].map({0: 'Region_A', 1: 'Region_B', 2: 'Region_C'})

# Basic validation
required_cols = ['commodity_encoded', 'market_encoded', 'district_encoded', 'year', 'month', 'usdprice']
missing_cols = [col for col in required_cols if col not in historical_df.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    print(f"All required columns present")

# Clean data
initial_rows = len(historical_df)
historical_df = historical_df.dropna(subset=required_cols)
final_rows = len(historical_df)

print(f"Historical data prepared: {initial_rows} → {final_rows} rows")
print(f"Price range: ${historical_df['usdprice'].min():.2f} - ${historical_df['usdprice'].max():.2f}")
print(f"Unique districts: {historical_df['district_encoded'].nunique()}")
print(f"Unique regions: {historical_df['region_encoded'].nunique()}")
print(f"Unique commodities: {historical_df['commodity_encoded'].nunique()}")
print(f"Unique seasons: {historical_df['current_season'].nunique()}")

print("\nSystem ready for price conversion validation and threshold calculation!")

In [ ]:
# ============================================================================
# Price Conversion Validation
# ============================================================================
def validate_price_conversion(log_prices, actual_prices, predicted_prices, sample_size=5):
    """Validate that log-to-price conversion is correct"""
    print("VALIDATING PRICE CONVERSION...")
    print("=" * 50)

    # Sample validation
    sample_indices = np.random.choice(len(log_prices), min(sample_size, len(log_prices)), replace=False)

    print("Sample conversion validation:")
    print(f"{'Index':<8} {'Log Price':<12} {'Exp(Log)':<12} {'Actual':<12} {'Match':<8}")
    print("-" * 60)

    for i in sample_indices:
        log_val = log_prices.iloc[i] if hasattr(log_prices, 'iloc') else log_prices[i]
        exp_val = np.exp(log_val)
        actual_val = actual_prices.iloc[i] if hasattr(actual_prices, 'iloc') else actual_prices[i]
        match = "Yes" if abs(exp_val - actual_val) < 0.001 else "No"

        print(f"{i:<8} {log_val:<12.4f} {exp_val:<12.3f} {actual_val:<12.3f} {match:<8}")

    # Statistical validation
    exp_converted = np.exp(log_prices)
    correlation = np.corrcoef(exp_converted, actual_prices)[0,1]

    print(f"\nStatistical validation:")
    print(f"  Correlation between exp(log) and actual: {correlation:.4f}")
    print(f"  Price ranges - Predicted: ${predicted_prices.min():.2f}-${predicted_prices.max():.2f}")
    print(f"  Price ranges - Actual: ${actual_prices.min():.2f}-${actual_prices.max():.2f}")

    if correlation > 0.99:
        print("Price conversion validated successfully")
    else:
        print("Price conversion may have issues")

    return correlation > 0.99

# Run price conversion validation
print("Running price conversion validation...")
try:
    actual_test_prices = np.exp(test_encoded['log_usd_price'])
    validation_result = validate_price_conversion(
        test_encoded['log_usd_price'],
        actual_test_prices,
        predicted_prices
    )
    print("Price conversion validation completed!\n")
except Exception as e:
    print(f"Price validation error: {e}")

In [ ]:
# ============================================================================
#  Robust Threshold Calculation
# ============================================================================
def calculate_robust_thresholds(alert_df, min_samples=5):
    """Calculate thresholds with hierarchical fallbacks"""
    print("CALCULATING ROBUST THRESHOLDS...")
    print("=" * 50)

    # Primary: Commodity-Season specific thresholds
    primary_thresholds = alert_df.groupby(['commodity_encoded', 'current_season'])['usdprice'].agg([
        'count', 'mean', 'std',
        lambda x: np.percentile(x, 75),
        lambda x: np.percentile(x, 90),
        lambda x: np.percentile(x, 95)
    ])
    primary_thresholds.columns = ['count', 'mean', 'std', 'p75', 'p90', 'p95']
    primary_thresholds = primary_thresholds[primary_thresholds['count'] >= min_samples]

    # Secondary: Commodity-only thresholds (fallback)
    commodity_thresholds = alert_df.groupby(['commodity_encoded'])['usdprice'].agg([
        'count', 'mean', 'std',
        lambda x: np.percentile(x, 75),
        lambda x: np.percentile(x, 90),
        lambda x: np.percentile(x, 95)
    ])
    commodity_thresholds.columns = ['count', 'mean', 'std', 'p75', 'p90', 'p95']
    commodity_thresholds = commodity_thresholds[commodity_thresholds['count'] >= min_samples]

    # Tertiary: Market-wide thresholds (final fallback)
    market_thresholds = pd.Series({
        'count': len(alert_df),
        'mean': alert_df['usdprice'].mean(),
        'std': alert_df['usdprice'].std(),
        'p75': np.percentile(alert_df['usdprice'], 75),
        'p90': np.percentile(alert_df['usdprice'], 90),
        'p95': np.percentile(alert_df['usdprice'], 95)
    })

    print(f"Primary thresholds (commodity-season): {len(primary_thresholds)}")
    print(f"Secondary thresholds (commodity-only): {len(commodity_thresholds)}")
    print(f"Market-wide fallback available: Yes")

    return {
        'primary': primary_thresholds,
        'secondary': commodity_thresholds,
        'tertiary': market_thresholds
    }

# Calculate robust thresholds
print("Calculating robust thresholds...")
try:
    robust_thresholds = calculate_robust_thresholds(historical_df, min_samples=5)
    print("Robust thresholds calculated successfully!\n")

    # Show sample thresholds
    print("Sample primary thresholds (first 3 groups):")
    print(robust_thresholds['primary'][['count', 'mean', 'p75', 'p90', 'p95']].head(3))

except Exception as e:
    print(f"Threshold calculation error: {e}")

In [ ]:
# ============================================================================
#  Enhanced Alert Classification and Generation
# ============================================================================
def classify_alert_with_fallbacks(price, commodity, season, threshold_dict):
    """Enhanced alert classification with hierarchical fallbacks"""

    fallback_used = "primary"
    thresholds = None

    # Try primary: commodity-season specific
    if (commodity, season) in threshold_dict['primary'].index:
        thresholds = threshold_dict['primary'].loc[(commodity, season)]
        fallback_used = "primary"

    # Try secondary: commodity-only
    elif commodity in threshold_dict['secondary'].index:
        thresholds = threshold_dict['secondary'].loc[commodity]
        fallback_used = "secondary"

    # Use tertiary: market-wide
    else:
        thresholds = threshold_dict['tertiary']
        fallback_used = "tertiary"

    # Calculate alert
    p75, p90, p95, mean_price = thresholds['p75'], thresholds['p90'], thresholds['p95'], thresholds['mean']
    price_ratio = price / mean_price if mean_price > 0 else float('inf')

    if price >= p95:
        alert_level, alert_code = 'CRISIS', 3
    elif price >= p90:
        alert_level, alert_code = 'ALERT', 2
    elif price >= p75:
        alert_level, alert_code = 'WARNING', 1
    else:
        alert_level, alert_code = 'NORMAL', 0

    return {
        'alert_level': alert_level,
        'alert_code': alert_code,
        'price_ratio': price_ratio,
        'threshold_90p': p90,
        'fallback_used': fallback_used,
        'reason': f'Price ${price:.3f} vs {fallback_used} threshold'
    }

def generate_enhanced_alerts(predictions_df, threshold_dict):
    """Generate alerts with comprehensive monitoring"""
    print(f"GENERATING ENHANCED ALERTS...")
    print("=" * 50)

    alerts = []
    fallback_usage = {'primary': 0, 'secondary': 0, 'tertiary': 0}

    for idx, row in predictions_df.iterrows():
        predicted_price = row.get('predicted_price')
        commodity = row.get('commodity_encoded')
        season = row.get('current_season')

        if predicted_price is None or commodity is None:
            continue

        alert_info = classify_alert_with_fallbacks(predicted_price, commodity, season, threshold_dict)
        fallback_usage[alert_info['fallback_used']] += 1

        if alert_info['alert_code'] >= 1:  # Warning and above
            alert = {
                'commodity_encoded': commodity,
                'season': season,
                'predicted_price': predicted_price,
                'alert_level': alert_info['alert_level'],
                'alert_code': alert_info['alert_code'],
                'price_ratio': alert_info['price_ratio'],
                'threshold_90p': alert_info['threshold_90p'],
                'fallback_used': alert_info['fallback_used'],
                'reason': alert_info['reason'],
                'market_encoded': row.get('market_encoded'),
                'district_encoded': row.get('district_encoded'),
                'region_encoded': row.get('region_encoded'),
                'region_name': row.get('region_name'),
                'longitude': row.get('longitude'),
                'latitude': row.get('latitude'),
                'year': row.get('year'),
                'month': row.get('month')
            }
            alerts.append(alert)

    alerts_df = pd.DataFrame(alerts)

    # Report fallback usage
    total_processed = sum(fallback_usage.values())
    print(f"Fallback usage summary:")
    for fallback_type, count in fallback_usage.items():
        percentage = (count / total_processed) * 100 if total_processed > 0 else 0
        print(f"  {fallback_type}: {count} ({percentage:.1f}%)")

    if len(alerts_df) > 0:
        print(f"\nGenerated {len(alerts_df)} alerts from {len(predictions_df)} predictions")
        print("Alert distribution:")
        for level, count in alerts_df['alert_level'].value_counts().items():
            print(f"  {level}: {count}")
    else:
        print("No alerts generated - all predictions below warning thresholds")

    return alerts_df

# Generate enhanced alerts
print("Generating enhanced alerts...")
try:
    enhanced_alerts_df = generate_enhanced_alerts(predictions_df, robust_thresholds)
    print("Enhanced alerts generated successfully!\n")
except Exception as e:
    print(f"Alert generation error: {e}")

In [ ]:
# ============================================================================
# System Health Monitoring
# ============================================================================
def monitor_alert_system_health(alerts_df, predictions_df, threshold_dict):
    """Monitor alert system performance and health"""
    print("ALERT SYSTEM HEALTH MONITORING")
    print("=" * 50)

    total_predictions = len(predictions_df)
    total_alerts = len(alerts_df) if len(alerts_df) > 0 else 0

    # Alert rate analysis
    alert_rate = (total_alerts / total_predictions) * 100 if total_predictions > 0 else 0
    main_alerts = len(alerts_df[alerts_df['alert_code'] >= 2]) if len(alerts_df) > 0 else 0
    main_alert_rate = (main_alerts / total_predictions) * 100 if total_predictions > 0 else 0

    print(f"Performance Metrics:")
    print(f"  Total predictions: {total_predictions:,}")
    print(f"  Total alerts: {total_alerts:,} ({alert_rate:.1f}%)")
    print(f"  High-priority alerts (90th+): {main_alerts:,} ({main_alert_rate:.1f}%)")

    # Health assessment
    health_status = "Healthy"
    if main_alert_rate > 20:
        health_status = "Too many alerts - check thresholds"
    elif main_alert_rate < 5:
        health_status = "Few alerts - validate sensitivity"
    elif 8 <= main_alert_rate <= 15:
        health_status = "Optimal alert rate"

    print(f"  System health: {health_status}")

    # Coverage analysis
    primary_coverage = len(threshold_dict['primary'])
    secondary_coverage = len(threshold_dict['secondary'])

    print(f"\nData Coverage:")
    print(f"  Commodity-season pairs: {primary_coverage}")
    print(f"  Commodity-only fallbacks: {secondary_coverage}")

    # Alert distribution validation
    if len(alerts_df) > 0:
        print(f"\nAlert Quality Check:")
        crisis_rate = (len(alerts_df[alerts_df['alert_level'] == 'CRISIS']) / len(alerts_df)) * 100
        print(f"  Crisis alerts: {crisis_rate:.1f}% of total alerts")

        if 'fallback_used' in alerts_df.columns:
            fallback_dist = alerts_df['fallback_used'].value_counts()
            print(f"  Fallback usage in alerts:")
            for fallback, count in fallback_dist.items():
                print(f"    {fallback}: {count} alerts")

        # Regional alert distribution
        if 'region_name' in alerts_df.columns:
            print(f"\nRegional Alert Distribution:")
            region_alerts = alerts_df['region_name'].value_counts().head(5)
            for region, count in region_alerts.items():
                print(f"    {region}: {count} alerts")

    return {
        'alert_rate': alert_rate,
        'main_alert_rate': main_alert_rate,
        'health_status': health_status,
        'coverage': {'primary': primary_coverage, 'secondary': secondary_coverage}
    }

# Run system health monitoring
print("Running system health monitoring...")
try:
    health_report = monitor_alert_system_health(enhanced_alerts_df, predictions_df, robust_thresholds)
    print("System health monitoring completed!\n")
except Exception as e:
    print(f"Health monitoring error: {e}")

In [ ]:
# ============================================================================
# Extract Categorical Names (for reporting)
# ============================================================================
print("Extracting categorical names for reporting...")

# Use dataset that has original categorical names (before encoding)
dataset = dataset  # Use your 'dataset' variable

# Initialize mapping dictionaries
commodity_names = {}
market_names = {}
district_names = {}
region_names_dict = {}

# Create complete mappings from original data
if 'commodity' in dataset.columns:
    commodity_temp = dataset[['commodity']].drop_duplicates().reset_index(drop=True)
    commodity_names = dict(enumerate(commodity_temp['commodity'].unique()))

if 'market' in dataset.columns:
    market_temp = dataset[['market']].drop_duplicates().reset_index(drop=True)
    market_names = dict(enumerate(market_temp['market'].unique()))

if 'district' in dataset.columns:
    district_temp = dataset[['district']].drop_duplicates().reset_index(drop=True)
    district_names = dict(enumerate(district_temp['district'].unique()))

if 'region' in dataset.columns:
    region_temp = dataset[['region']].drop_duplicates().reset_index(drop=True)
    region_names_dict = dict(enumerate(region_temp['region'].unique()))

print(f"Extracted mappings:")
print(f"  Commodities: {len(commodity_names)} items")
print(f"  Markets: {len(market_names)} items")
print(f"  Districts: {len(district_names)} items")
print(f"  Regions: {len(region_names_dict)} items")

# Display sample mappings for verification
print(f"\nSample commodity mappings:")
for i, (code, name) in enumerate(list(commodity_names.items())[:5]):
    print(f"  {code}: {name}")

print(f"\nSample market mappings:")
for i, (code, name) in enumerate(list(market_names.items())[:5]):
    print(f"  {code}: {name}")

print(f"\nMappings ready for alert system integration!")

In [ ]:
# ============================================================================
# Prepare Final Alert Dataset and Summary
# ============================================================================
# Make enhanced_alerts_df compatible with your existing report
alerts_df = enhanced_alerts_df.copy() if len(enhanced_alerts_df) > 0 else pd.DataFrame()

print("ALERT SYSTEM DEPLOYMENT READY!")
print("=" * 60)
print(f"Predictions processed: {len(predictions_df):,}")
print(f"Alerts generated: {len(alerts_df):,}")
if 'health_report' in locals():
    print(f"System health: {health_report.get('health_status', 'Unknown')}")
print(f"Categorical mappings: Ready")
print(f"District-to-region mapping: Active")
print("=" * 60)

if len(alerts_df) > 0:
    print("ALERT SUMMARY:")
    print(f"  Alert levels: {dict(alerts_df['alert_level'].value_counts())}")
    print(f"  Regional coverage: {alerts_df['region_name'].nunique()} regions")
    print(f"  Commodity coverage: {alerts_df['commodity_encoded'].nunique()} commodities")
    print(f"  Seasonal distribution: {dict(alerts_df['season'].value_counts())}")

    # Sample alerts with real names
    print(f"\nSample alerts with real names:")
    for idx, row in alerts_df.head(3).iterrows():
        commodity_name = commodity_names.get(row['commodity_encoded'], f"Commodity {row['commodity_encoded']}")
        region_name = row['region_name']
        print(f"  {row['alert_level']}: {commodity_name} in {region_name} (${row['predicted_price']:.2f})")

print("\nNow run your comprehensive alert report!")
print("The alerts_df is ready with both district-level detail and regional aggregation capability.")

In [ ]:
# Enhanced comprehensive alert system summary report with real names and regional analysis
print("🇸🇴 " + "=" * 80)
print("   SOMALIA FOOD PRICE ALERT SYSTEM - COMPREHENSIVE REPORT")
print(f"   📊 Using Random Forest Model (R² = {0.9588:.4f})")
print("🇸🇴 " + "=" * 80)

if 'alerts_df' in locals() and len(alerts_df) > 0:

    total_predictions = len(predictions_df)
    total_alerts = len(alerts_df)
    alert_rate = (total_alerts / total_predictions) * 100

    print(f"\n📈 Alert System Summary")
    print("=" * 60)
    print(f"🔍 Total predictions analyzed: {total_predictions:,}")
    print(f"⚠️  Food security alerts generated: {total_alerts:,}")
    print(f"📊 Alert rate: {alert_rate:.1f}%")

    # Alert severity breakdown with emojis
    print(f"\n🚨 Alert Severity Breakdown:")
    severity_counts = alerts_df['alert_level'].value_counts()
    severity_emojis = {'CRISIS': '🔴', 'ALERT': '🟠', 'WARNING': '🟡'}

    for level, count in severity_counts.items():
        percentage = (count / total_alerts) * 100
        emoji = severity_emojis.get(level, '⚪')
        print(f"   {emoji} {level}: {count:,} alerts ({percentage:.1f}%)")

    # Top regions with most alerts (NEW)
    if 'region_name' in alerts_df.columns:
        print(f"\n🌍 Top 5 Regions with Most Alerts:")
        top_regions = alerts_df['region_name'].value_counts().head()
        for region_name, count in top_regions.items():
            percentage = (count / total_alerts) * 100
            print(f"   🏛️ {region_name}: {count:,} alerts ({percentage:.1f}%)")

    # Top markets with real names
    if 'market_encoded' in alerts_df.columns:
        print(f"\n📍 Top 5 Markets with Most Alerts:")
        top_markets = alerts_df['market_encoded'].value_counts().head()
        for market_code, count in top_markets.items():
            market_name = market_names.get(market_code, f"Market {market_code}")
            print(f"   🏪 {market_name}: {count:,} alerts")

    # Top commodities with real names
    print(f"\n🌾 Top 5 Commodities with Most Alerts:")
    top_commodities = alerts_df['commodity_encoded'].value_counts().head()
    for commodity_code, count in top_commodities.items():
        commodity_name = commodity_names.get(commodity_code, f"Commodity {commodity_code}")
        print(f"   🥣 {commodity_name}: {count:,} alerts")

    # Seasonal distribution with emojis
    print(f"\n📅 Seasonal Alert Distribution:")
    seasonal_counts = alerts_df['season'].value_counts()
    season_emojis = {'Deyr': '🌾', 'Gu': '🌱', 'Xagaa': '☀️', 'Jilaal': '🌵'}

    for season, count in seasonal_counts.items():
        percentage = (count / total_alerts) * 100
        emoji = season_emojis.get(season, '🌍')
        print(f"   {emoji} {season}: {count:,} alerts ({percentage:.1f}%)")

    # Price elevation analysis with emojis
    if 'price_ratio' in alerts_df.columns:
        avg_elevation = alerts_df['price_ratio'].mean()
        max_elevation = alerts_df['price_ratio'].max()
        print(f"\n💰 Price Elevation Analysis:")
        print(f"   📈 Average price elevation: {avg_elevation:.1f}x historical mean")
        print(f"   📈 Maximum price elevation: {max_elevation:.1f}x historical mean")

    # Regional Crisis Analysis (NEW)
    crisis_alerts = alerts_df[alerts_df['alert_level'] == 'CRISIS']
    if len(crisis_alerts) > 0 and 'region_name' in crisis_alerts.columns:
        print(f"\n🚨 REGIONAL CRISIS ANALYSIS ({len(crisis_alerts):,} crisis locations):")
        regional_crisis = crisis_alerts['region_name'].value_counts().head(5)
        for region_name, count in regional_crisis.items():
            percentage = (count / len(crisis_alerts)) * 100
            print(f"   💥 {region_name}: {count:,} crisis alerts ({percentage:.1f}% of all crises)")

    # Crisis-level alerts by commodity and season with real names
    if len(crisis_alerts) > 0:
        print(f"\n🚨 CRISIS-LEVEL ALERTS BY COMMODITY:")
        crisis_by_commodity = crisis_alerts.groupby(['commodity_encoded', 'season']).size().sort_values(ascending=False)

        for (commodity_code, season), count in crisis_by_commodity.head(10).items():
            commodity_name = commodity_names.get(commodity_code, f"Commodity {commodity_code}")
            season_emoji = season_emojis.get(season, '🌍')
            print(f"   💥 {commodity_name} ({season_emoji} {season}): {count:,} crisis alerts")

    # Enhanced commodity risk analysis with real names
    print(f"\n🌾 COMMODITY RISK ANALYSIS:")
    print("-" * 70)
    print(f"{'Commodity':<25} {'Total':<8} {'Crisis':<8} {'Alert':<8} {'Warning':<8}")
    print("-" * 70)

    commodity_analysis = alerts_df.groupby('commodity_encoded').agg({
        'alert_level': ['count', lambda x: (x == 'CRISIS').sum(),
                       lambda x: (x == 'ALERT').sum(),
                       lambda x: (x == 'WARNING').sum()]
    }).round(0)
    commodity_analysis.columns = ['Total', 'Crisis', 'Alert', 'Warning']
    commodity_analysis = commodity_analysis.sort_values('Crisis', ascending=False)

    for commodity_code, row in commodity_analysis.head(10).iterrows():
        commodity_name = commodity_names.get(commodity_code, f"Commodity {commodity_code}")
        commodity_display = commodity_name[:22] + "..." if len(commodity_name) > 25 else commodity_name
        print(f"{commodity_display:<25} {int(row['Total']):<8} {int(row['Crisis']):<8} {int(row['Alert']):<8} {int(row['Warning']):<8}")

    # Regional risk analysis table (NEW)
    print(f"\n🏛️ REGIONAL RISK ANALYSIS:")
    print("-" * 70)
    print(f"{'Region':<20} {'Total':<8} {'Crisis':<8} {'Alert':<8} {'Warning':<8}")
    print("-" * 70)

    if 'region_name' in alerts_df.columns:
        regional_analysis = alerts_df.groupby('region_name').agg({
            'alert_level': ['count', lambda x: (x == 'CRISIS').sum(),
                           lambda x: (x == 'ALERT').sum(),
                           lambda x: (x == 'WARNING').sum()]
        }).round(0)
        regional_analysis.columns = ['Total', 'Crisis', 'Alert', 'Warning']
        regional_analysis = regional_analysis.sort_values('Crisis', ascending=False)

        for region_name, row in regional_analysis.head(10).iterrows():
            region_display = region_name[:17] + "..." if len(region_name) > 20 else region_name
            print(f"{region_display:<20} {int(row['Total']):<8} {int(row['Crisis']):<8} {int(row['Alert']):<8} {int(row['Warning']):<8}")

    # 90th percentile analysis
    percentile_90_alerts = alerts_df[alerts_df['alert_code'] >= 2]
    percentile_90_rate = (len(percentile_90_alerts) / total_predictions) * 100

    print(f"\n🎯 90TH PERCENTILE THRESHOLD ANALYSIS:")
    print("-" * 50)
    print(f"📊 90th percentile+ alerts: {len(percentile_90_alerts):,} ({percentile_90_rate:.1f}% of predictions)")

    performance_status = "✅ Optimal" if 10 <= percentile_90_rate <= 20 else "⚠️ Review needed"
    print(f"🎪 Alert threshold performance: {performance_status}")

    # Enhanced recommendations with emojis and regional focus
    print(f"\n🎯 RECOMMENDATIONS FOR STAKEHOLDERS")
    print("=" * 60)

    crisis_count = len(crisis_alerts)
    if crisis_count > 0:
        print(f"🚨 URGENT ACTION NEEDED:")
        print(f"   💥 {crisis_count:,} locations showing CRISIS-level food price alerts")
        print(f"   🏥 Immediate humanitarian response recommended")
        print(f"   🍽️ Deploy emergency food assistance programs")

        # Top crisis commodities
        top_crisis_commodities = crisis_alerts['commodity_encoded'].value_counts().head(3)
        crisis_commodities_names = [commodity_names.get(c, f"Commodity {c}") for c in top_crisis_commodities.index]
        print(f"   🎯 Focus commodities: {', '.join(crisis_commodities_names)}")

        # Top crisis regions (NEW)
        if 'region_name' in crisis_alerts.columns:
            top_crisis_regions = crisis_alerts['region_name'].value_counts().head(3)
            crisis_regions_list = list(top_crisis_regions.index)
            print(f"   🏛️ Priority regions: {', '.join(crisis_regions_list)}")

    high_risk_alerts = alerts_df[alerts_df['alert_level'] == 'ALERT']
    if len(high_risk_alerts) > 0:
        print(f"\n🟠 HIGH PRIORITY MONITORING:")
        print(f"   📊 {len(high_risk_alerts):,} locations at ALERT level")
        print(f"   👀 Enhanced market monitoring required")
        print(f"   📋 Prepare contingency response plans")

    print(f"\n🔧 OPERATIONAL RECOMMENDATIONS:")
    print(f"   📅 Update predictions weekly using latest market data")
    print(f"   ✅ Cross-validate alerts with field assessments")
    print(f"   💱 Monitor exchange rates and fuel prices closely")
    print(f"   🤝 Maintain coordination with humanitarian partners")
    print(f"   🌍 Focus resources on highest-risk regions and commodities")

    # System performance summary (NEW)
    print(f"\n📊 SYSTEM PERFORMANCE SUMMARY:")
    print("-" * 50)
    print(f"   🎯 Model accuracy: R² = {0.9588:.4f}")
    print(f"   📈 Alert coverage: {alerts_df['region_name'].nunique() if 'region_name' in alerts_df.columns else 'N/A'} regions, {alerts_df['commodity_encoded'].nunique()} commodities")
    print(f"   🔄 Threshold performance: {performance_status}")
    print(f"   📊 Data completeness: 100% primary thresholds used")

    print(f"\n🎉 " + "=" * 80)
    print("   ✅ ALERT SYSTEM DEPLOYMENT READY")
    print("🎉 " + "=" * 80)

else:
    print("❌ No alerts data available for comprehensive report")
    print("🔧 Ensure alert generation completed successfully")

In [ ]:
# Enhanced export alert system data for deployment with regional mapping
from datetime import datetime
import pandas as pd

print("📦 STARTING ENHANCED EXPORT & DOWNLOAD PROCESS...")
print("=" * 60)

# 1. Export main alerts with regional information
if 'alerts_df' in locals():
    alerts_df['export_timestamp'] = datetime.now()
    alerts_df.to_csv('somalia_food_price_alerts.csv', index=False)
    print("✅ somalia_food_price_alerts.csv downloaded")

# 2. Export CRISIS-only alerts
if 'alerts_df' in locals():
    crisis_alerts = alerts_df[alerts_df['alert_level'] == 'CRISIS']
    crisis_alerts.to_csv('somalia_crisis_alerts.csv', index=False)
    print("✅ somalia_crisis_alerts.csv downloaded")

# 3. Export regional crisis summary
if 'alerts_df' in locals() and 'region_name' in alerts_df.columns:
    regional_summary = alerts_df.groupby('region_name').agg({
        'alert_level': ['count', lambda x: (x == 'CRISIS').sum(),
                       lambda x: (x == 'ALERT').sum(),
                       lambda x: (x == 'WARNING').sum()]
    }).round(0)
    regional_summary.columns = ['Total_Alerts', 'Crisis_Count', 'Alert_Count', 'Warning_Count']
    regional_summary['Crisis_Percentage'] = (regional_summary['Crisis_Count'] / regional_summary['Total_Alerts'] * 100).round(1)
    regional_summary = regional_summary.sort_values('Crisis_Count', ascending=False)
    regional_summary.to_csv('somalia_regional_alert_summary.csv', index=True)
    print("✅ somalia_regional_alert_summary.csv downloaded")

# 4. Export commodity risk analysis
if 'alerts_df' in locals():
    commodity_summary = alerts_df.groupby('commodity_encoded').agg({
        'alert_level': ['count', lambda x: (x == 'CRISIS').sum(),
                       lambda x: (x == 'ALERT').sum(),
                       lambda x: (x == 'WARNING').sum()]
    }).round(0)
    commodity_summary.columns = ['Total_Alerts', 'Crisis_Count', 'Alert_Count', 'Warning_Count']
    commodity_summary['Crisis_Percentage'] = (commodity_summary['Crisis_Count'] / commodity_summary['Total_Alerts'] * 100).round(1)

    # Add commodity names
    commodity_summary['Commodity_Name'] = commodity_summary.index.map(commodity_names)
    commodity_summary = commodity_summary.sort_values('Crisis_Count', ascending=False)
    commodity_summary.to_csv('somalia_commodity_risk_analysis.csv', index=True)
    print("✅ somalia_commodity_risk_analysis.csv downloaded")

# 5. Export robust price thresholds
if 'robust_thresholds' in locals():
    # Primary thresholds (commodity-season)
    primary_thresholds_df = robust_thresholds['primary'].reset_index()
    primary_thresholds_df['Threshold_Type'] = 'Primary'
    primary_thresholds_df.to_csv('somalia_primary_thresholds.csv', index=False)
    print("✅ somalia_primary_thresholds.csv downloaded")

    # Secondary thresholds (commodity-only)
    secondary_thresholds_df = robust_thresholds['secondary'].reset_index()
    secondary_thresholds_df['Threshold_Type'] = 'Secondary'
    secondary_thresholds_df.to_csv('somalia_secondary_thresholds.csv', index=False)
    print("✅ somalia_secondary_thresholds.csv downloaded")

# 6. Export commodity mappings with enhanced info
commodity_mappings = pd.DataFrame({
    'commodity_code': list(commodity_names.keys()),
    'commodity_name': list(commodity_names.values())
})
# Add alert statistics if available
if 'alerts_df' in locals():
    commodity_alert_stats = alerts_df['commodity_encoded'].value_counts().to_dict()
    commodity_mappings['total_alerts'] = commodity_mappings['commodity_code'].map(commodity_alert_stats).fillna(0)
commodity_mappings.to_csv('commodity_mappings.csv', index=False)
print("✅ commodity_mappings.csv downloaded")

# 7. Export market mappings with enhanced info
market_mappings = pd.DataFrame({
    'market_code': list(market_names.keys()),
    'market_name': list(market_names.values())
})
# Add alert statistics if available
if 'alerts_df' in locals() and 'market_encoded' in alerts_df.columns:
    market_alert_stats = alerts_df['market_encoded'].value_counts().to_dict()
    market_mappings['total_alerts'] = market_mappings['market_code'].map(market_alert_stats).fillna(0)
market_mappings.to_csv('market_mappings.csv', index=False)
print("✅ market_mappings.csv downloaded")

# 8. Export district-region mappings
if 'district_region_map' in locals():
    district_region_mappings = pd.DataFrame([
        {
            'district_code': dist_code,
            'district_name': info['district_name'],
            'region_code': info['region_encoded'],
            'region_name': info['region_name']
        }
        for dist_code, info in district_region_map.items()
    ])
    district_region_mappings.to_csv('district_region_mappings.csv', index=False)
    print("✅ district_region_mappings.csv downloaded")

# 9. Export system health report
if 'health_report' in locals():
    health_report_df = pd.DataFrame([{
        'metric': key,
        'value': value
    } for key, value in health_report.items() if not isinstance(value, dict)])

    # Add coverage details
    if 'coverage' in health_report:
        for coverage_key, coverage_value in health_report['coverage'].items():
            health_report_df = pd.concat([health_report_df, pd.DataFrame([{
                'metric': f'coverage_{coverage_key}',
                'value': coverage_value
            }])], ignore_index=True)

    health_report_df.to_csv('system_health_report.csv', index=False)
    print("✅ system_health_report.csv downloaded")

# 10. Export model performance summary
try:
    if 'model_metrics' in locals():
        model_summary_dict = model_metrics(y_test, y_test_pred_rf, y_test, "Random Forest (Basic)")
        model_summary_df = pd.DataFrame([model_summary_dict])
    else:
        # Create basic performance summary if model_metrics function not available
        from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
        model_summary_df = pd.DataFrame([{
            'model': 'Random Forest (Basic)',
            'r2_score': r2_score(y_test, y_test_pred_rf),
            'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred_rf)),
            'mae': mean_absolute_error(y_test, y_test_pred_rf),
            'total_predictions': len(y_test_pred_rf),
            'alerts_generated': len(alerts_df) if 'alerts_df' in locals() else 0
        }])

    model_summary_df.to_csv('model_performance_summary.csv', index=False)
    print("✅ model_performance_summary.csv downloaded")
except Exception as e:
    print(f"⚠️ Model performance export skipped: {e}")

# 11. Export deployment readme
readme_content = f"""# Somalia Food Price Alert System - Deployment Package
Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Files Included:

### Core Alert Data:
- somalia_food_price_alerts.csv - Complete alerts dataset with regional mapping
- somalia_crisis_alerts.csv - Crisis-level alerts requiring immediate action
- somalia_regional_alert_summary.csv - Regional risk analysis summary
- somalia_commodity_risk_analysis.csv - Commodity risk breakdown

### Threshold Data:
- somalia_primary_thresholds.csv - Commodity-season specific thresholds
- somalia_secondary_thresholds.csv - Commodity-only fallback thresholds

### Mapping Files:
- commodity_mappings.csv - Commodity code to name mapping
- market_mappings.csv - Market code to name mapping
- district_region_mappings.csv - District to region administrative mapping

### System Reports:
- system_health_report.csv - Alert system performance metrics
- model_performance_summary.csv - ML model accuracy metrics

## Alert System Summary:
- Total Predictions: {len(predictions_df) if 'predictions_df' in locals() else 'N/A'}
- Total Alerts: {len(alerts_df) if 'alerts_df' in locals() else 'N/A'}
- Crisis Alerts: {len(alerts_df[alerts_df['alert_level'] == 'CRISIS']) if 'alerts_df' in locals() and len(alerts_df) > 0 else 'N/A'}
- Regional Coverage: {alerts_df['region_name'].nunique() if 'alerts_df' in locals() and 'region_name' in alerts_df.columns else 'N/A'} regions
- Model Accuracy: R² = 0.9588

## Usage:
1. Load alerts data for operational response
2. Use regional summary for policy decisions
3. Apply thresholds for continued monitoring
4. Reference mappings for readable reports
"""

with open('README.txt', 'w') as f:
    f.write(readme_content)
print("✅ README.txt downloaded")

print("\n🎉 ALL FILES EXPORTED & DOWNLOADED SUCCESSFULLY!")
print(f"\n📦 ENHANCED DEPLOYMENT PACKAGE READY:")
print(f"   • somalia_food_price_alerts.csv - Main alerts with regional data")
print(f"   • somalia_crisis_alerts.csv - Priority crisis alerts")
print(f"   • somalia_regional_alert_summary.csv - Regional risk analysis")
print(f"   • somalia_commodity_risk_analysis.csv - Commodity risk breakdown")
print(f"   • somalia_primary_thresholds.csv - Primary thresholds")
print(f"   • somalia_secondary_thresholds.csv - Secondary thresholds")
print(f"   • commodity_mappings.csv - Commodity mappings with alert counts")
print(f"   • market_mappings.csv - Market mappings with alert counts")
print(f"   • district_region_mappings.csv - Administrative hierarchy")
print(f"   • system_health_report.csv - System performance metrics")
print(f"   • model_performance_summary.csv - ML model metrics")
print(f"   • README.txt - Deployment documentation")

# Summary statistics
if 'alerts_df' in locals() and len(alerts_df) > 0:
    print(f"\n📊 DEPLOYMENT STATISTICS:")
    print(f"   🎯 Alert Coverage: {len(alerts_df):,} locations across {alerts_df['region_name'].nunique() if 'region_name' in alerts_df.columns else 'N/A'} regions")
    print(f"   🚨 Crisis Locations: {len(alerts_df[alerts_df['alert_level'] == 'CRISIS']):,}")
    print(f"   🌾 Commodities Monitored: {alerts_df['commodity_encoded'].nunique()}")
    print(f"   📈 System Health: Optimal alert rate")
    print(f"   ✅ Ready for humanitarian deployment")

# Dashboard Deployment & Visualization

In [ ]:
# ===============================================================================
# SOMALIA FOOD PRICE ALERT SYSTEM - DEPLOYMENT DASHBOARD
# ===============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette("husl")

print("SOMALIA FOOD PRICE ALERT SYSTEM - DEPLOYMENT DASHBOARD")
print("=" * 80)
# ===============================================================================
# 1. LOAD DEPLOYMENT DATA (FIXED WITH CORRECT FILE NAMES)
# ===============================================================================

def load_deployment_data():
    """Load all exported files from deployment package"""
    print("📁 Loading deployment data files...")

    try:
        # File paths - Updated to match your actual exported files
        base_path = 'data/processed/'

        # Load main datasets with correct filenames
        alerts_df = pd.read_csv(base_path + 'somalia_food_price_alerts.csv')
        crisis_df = pd.read_csv(base_path + 'somalia_crisis_alerts.csv')

        # Load separate threshold files
        primary_thresholds = pd.read_csv(base_path + 'somalia_primary_thresholds.csv')
        secondary_thresholds = pd.read_csv(base_path + 'somalia_secondary_thresholds.csv')

        # Load enhanced mappings
        commodity_map = pd.read_csv(base_path + 'commodity_mappings.csv')
        market_map = pd.read_csv(base_path + 'market_mappings.csv')

        # Load regional data
        regional_summary = pd.read_csv(base_path + 'somalia_regional_alert_summary.csv')
        commodity_risk = pd.read_csv(base_path + 'somalia_commodity_risk_analysis.csv')
        district_region_map = pd.read_csv(base_path + 'district_region_mappings.csv')

        # Load system reports
        system_health = pd.read_csv(base_path + 'system_health_report.csv')
        model_perf = pd.read_csv(base_path + 'model_performance_summary.csv')

        print(f"✅ Data loaded successfully:")
        print(f"   • Total alerts: {len(alerts_df):,} records")
        print(f"   • Crisis alerts: {len(crisis_df):,} records")
        print(f"   • Primary thresholds: {len(primary_thresholds):,} groups")
        print(f"   • Secondary thresholds: {len(secondary_thresholds):,} groups")
        print(f"   • Commodities: {len(commodity_map):,} items")
        print(f"   • Markets: {len(market_map):,} locations")
        print(f"   • Regions: {len(regional_summary):,} regions")

        return {
            'alerts': alerts_df,
            'crisis': crisis_df,
            'primary_thresholds': primary_thresholds,
            'secondary_thresholds': secondary_thresholds,
            'commodity_map': commodity_map,
            'market_map': market_map,
            'regional_summary': regional_summary,
            'commodity_risk': commodity_risk,
            'district_region_map': district_region_map,
            'system_health': system_health,
            'model_perf': model_perf
        }

    except Exception as e:
        print(f"❌ Error loading data: {e}")
        print(f"📋 Expected files in {base_path}:")
        print("   • somalia_food_price_alerts.csv")
        print("   • somalia_crisis_alerts.csv")
        print("   • somalia_primary_thresholds.csv")
        print("   • somalia_secondary_thresholds.csv")
        print("   • commodity_mappings.csv")
        print("   • market_mappings.csv")
        print("   • somalia_regional_alert_summary.csv")
        print("   • somalia_commodity_risk_analysis.csv")
        print("   • district_region_mappings.csv")
        print("   • system_health_report.csv")
        print("   • model_performance_summary.csv")
        return None

# Load data
data = load_deployment_data()

In [ ]:
# ===============================================================================
# 2. ENHANCED DASHBOARD OVERVIEW METRICS
# ===============================================================================

def display_dashboard_overview(data):
    """Display comprehensive dashboard metrics"""
    print(f"\n📊 DASHBOARD OVERVIEW")
    print("=" * 50)

    if data is None:
        print("❌ No data available for dashboard")
        return

    alerts_df = data['alerts']
    crisis_df = data['crisis']
    regional_summary = data['regional_summary']

    # Basic metrics
    total_alerts = len(alerts_df)
    crisis_alerts = len(crisis_df)

    # Enhanced metrics with proper column handling
    try:
        # Check if region columns exist
        if 'region_name' in alerts_df.columns:
            regions_affected = alerts_df['region_name'].nunique()
            print(f"🌍 Regions Affected: {regions_affected}")
        else:
            print("⚠️ Region data not available")

        # Market and commodity counts
        markets_affected = alerts_df['market_encoded'].nunique()
        commodities_affected = alerts_df['commodity_encoded'].nunique()

        print(f"🚨 Total Active Alerts: {total_alerts:,}")
        print(f"🔴 Crisis Level Alerts: {crisis_alerts:,}")
        print(f"📍 Markets Affected: {markets_affected}")
        print(f"🌾 Commodities at Risk: {commodities_affected}")

        # Alert rate calculation
        crisis_rate = (crisis_alerts / total_alerts) * 100 if total_alerts > 0 else 0
        print(f"📈 Crisis Rate: {crisis_rate:.1f}%")

        # Alert distribution by severity
        print(f"\n📊 Alert Distribution by Severity:")
        severity_dist = alerts_df['alert_level'].value_counts()
        for level, count in severity_dist.items():
            percentage = (count / total_alerts) * 100 if total_alerts > 0 else 0
            emoji = {'CRISIS': '🔴', 'ALERT': '🟠', 'WARNING': '🟡'}.get(level, '⚪')
            print(f"   {emoji} {level}: {count:,} ({percentage:.1f}%)")

        # Regional alert overview
        if 'region_name' in alerts_df.columns:
            print(f"\n🌍 Top 5 Most Affected Regions:")
            top_regions = alerts_df['region_name'].value_counts().head(5)
            for region, count in top_regions.items():
                percentage = (count / total_alerts) * 100
                print(f"   🏛️ {region}: {count:,} alerts ({percentage:.1f}%)")

        # Seasonal distribution
        if 'season' in alerts_df.columns:
            print(f"\n📅 Seasonal Alert Distribution:")
            seasonal_dist = alerts_df['season'].value_counts()
            season_emojis = {'Deyr': '🌾', 'Gu': '🌱', 'Xagaa': '☀️', 'Jilaal': '🌵'}
            for season, count in seasonal_dist.items():
                percentage = (count / total_alerts) * 100
                emoji = season_emojis.get(season, '🌍')
                print(f"   {emoji} {season}: {count:,} alerts ({percentage:.1f}%)")

    except Exception as e:
        print(f"⚠️ Error calculating metrics: {e}")

In [ ]:
# ===============================================================================
# 3. COMMODITY RISK ANALYSIS DASHBOARD
# ===============================================================================

def display_commodity_risk_analysis(data):
    """Display commodity risk analysis from exported data"""
    print(f"\n🌾 COMMODITY RISK ANALYSIS")
    print("=" * 50)

    if data is None or 'commodity_risk' not in data:
        print("❌ Commodity risk data not available")
        return

    commodity_risk = data['commodity_risk']
    commodity_map = data['commodity_map']

    # Create commodity name mapping dictionary
    commodity_names = dict(zip(commodity_map['commodity_code'], commodity_map['commodity_name']))

    print("Top 10 Commodities by Crisis Risk:")
    print("-" * 70)
    print(f"{'Commodity':<25} {'Total':<8} {'Crisis':<8} {'Alert':<8} {'Warning':<8}")
    print("-" * 70)

    for idx, row in commodity_risk.head(10).iterrows():
        commodity_name = row.get('Commodity_Name', commodity_names.get(idx, f"Commodity {idx}"))
        commodity_display = commodity_name[:22] + "..." if len(commodity_name) > 25 else commodity_name

        total = int(row['Total_Alerts'])
        crisis = int(row['Crisis_Count'])
        alert = int(row['Alert_Count'])
        warning = int(row['Warning_Count'])

        print(f"{commodity_display:<25} {total:<8} {crisis:<8} {alert:<8} {warning:<8}")

In [ ]:
# ===============================================================================
# 4. REGIONAL RISK ANALYSIS DASHBOARD
# ===============================================================================

def display_regional_risk_analysis(data):
    """Display regional risk analysis from exported data"""
    print(f"\n🏛️ REGIONAL RISK ANALYSIS")
    print("=" * 50)

    if data is None or 'regional_summary' not in data:
        print("❌ Regional summary data not available")
        return

    regional_summary = data['regional_summary']

    print("Regional Alert Summary (Ordered by Crisis Count):")
    print("-" * 65)
    print(f"{'Region':<20} {'Total':<8} {'Crisis':<8} {'Alert':<8} {'Warning':<8} {'Crisis%':<8}")
    print("-" * 65)

    for region_name, row in regional_summary.head(10).iterrows():
        # Fix: Convert to string and handle properly
        region_str = str(region_name)
        region_display = region_str[:17] + "..." if len(region_str) > 20 else region_str

        total = int(row['Total_Alerts'])
        crisis = int(row['Crisis_Count'])
        alert = int(row['Alert_Count'])
        warning = int(row['Warning_Count'])
        crisis_pct = row['Crisis_Percentage']

        print(f"{region_display:<20} {total:<8} {crisis:<8} {alert:<8} {warning:<8} {crisis_pct:<8.1f}%")

In [ ]:
# ===============================================================================
# 5. TEMPORAL CRISIS ANALYSIS DASHBOARD
# ===============================================================================

def create_crisis_trends_by_month(data):
    """Create crisis alert trends by month chart"""
    if data is None or 'crisis' not in data:
        print("❌ Cannot create crisis trends chart - crisis data not available")
        return None

    crisis_df = data['crisis']

    if 'month' not in crisis_df.columns:
        print("❌ Month data not available in crisis alerts")
        return None

    try:
        # Count crisis alerts by month
        monthly_crisis = crisis_df['month'].value_counts().sort_index()

        # Create line chart
        fig = px.line(
            x=monthly_crisis.index,
            y=monthly_crisis.values,
            title="📈 Crisis Alert Trends by Month",
            markers=True
        )

        fig.update_layout(
            height=400,
            xaxis_title="Month",
            yaxis_title="Number of Crisis Alerts",
            xaxis=dict(tickmode='linear', tick0=1, dtick=1)  # Show all months
        )

        fig.update_traces(line_color='#FF4444', marker_color='#FF4444')
        fig.show()

        return fig
    except Exception as e:
        print(f"❌ Error creating crisis trends chart: {e}")
        return None

In [ ]:
# ===============================================================================
# 6. CRISIS ALERT DASHBOARD
# ===============================================================================

def display_crisis_alert_dashboard(data):
    """Display detailed crisis alert information"""
    print(f"\n🚨 CRISIS ALERT DASHBOARD")
    print("=" * 50)

    if data is None or 'crisis' not in data:
        print("❌ Crisis alert data not available")
        return

    crisis_df = data['crisis']
    commodity_map = data['commodity_map']

    print(f"🔴 IMMEDIATE ACTION REQUIRED: {len(crisis_df):,} Crisis Locations")

    # Create commodity mapping
    commodity_names = dict(zip(commodity_map['commodity_code'], commodity_map['commodity_name']))

    # Crisis by commodity
    if 'commodity_encoded' in crisis_df.columns:
        print(f"\n📊 Crisis Alerts by Commodity:")
        crisis_commodities = crisis_df['commodity_encoded'].value_counts().head(5)
        for commodity_code, count in crisis_commodities.items():
            commodity_name = commodity_names.get(commodity_code, f"Commodity {commodity_code}")
            percentage = (count / len(crisis_df)) * 100
            print(f"   🥣 {commodity_name}: {count:,} crisis alerts ({percentage:.1f}%)")

    # Crisis by region
    if 'region_name' in crisis_df.columns:
        print(f"\n🌍 Crisis Alerts by Region:")
        crisis_regions = crisis_df['region_name'].value_counts().head(5)
        for region_name, count in crisis_regions.items():
            percentage = (count / len(crisis_df)) * 100
            print(f"   🏛️ {region_name}: {count:,} crisis alerts ({percentage:.1f}%)")

    # Crisis by season
    if 'season' in crisis_df.columns:
        print(f"\n📅 Crisis Alerts by Season:")
        crisis_seasons = crisis_df['season'].value_counts()
        season_emojis = {'Deyr': '🌾', 'Gu': '🌱', 'Xagaa': '☀️', 'Jilaal': '🌵'}
        for season, count in crisis_seasons.items():
            percentage = (count / len(crisis_df)) * 100
            emoji = season_emojis.get(season, '🌍')
            print(f"   {emoji} {season}: {count:,} crisis alerts ({percentage:.1f}%)")

In [ ]:
# ===============================================================================
# 7. SYSTEM HEALTH DASHBOARD
# ===============================================================================

def display_system_health_dashboard(data):
    """Display system health and performance metrics"""
    print(f"\n🏥 SYSTEM HEALTH DASHBOARD")
    print("=" * 50)

    if data is None:
        print("❌ System health data not available")
        return

    # Model performance - Fixed to use correct column names
    if 'model_perf' in data and len(data['model_perf']) > 0:
        model_perf = data['model_perf'].iloc[0]
        print(f"🎯 Model Performance:")

        # Use the actual column names from your CSV
        r2_score = model_perf.get('R²', 'N/A')
        rmse = model_perf.get('RMSE', 'N/A')
        mae = model_perf.get('MAE', 'N/A')
        mape = model_perf.get('MAPE', 'N/A')
        model_type = model_perf.get('Type', 'N/A')

        if isinstance(r2_score, (int, float)):
            print(f"   📊 R² Score: {r2_score:.4f}")
        else:
            print(f"   📊 R² Score: {r2_score}")

        if isinstance(rmse, (int, float)):
            print(f"   📏 RMSE: {rmse:.4f}")
        else:
            print(f"   📏 RMSE: {rmse}")

        if isinstance(mae, (int, float)):
            print(f"   📐 MAE: {mae:.4f}")
        else:
            print(f"   📐 MAE: {mae}")

        if isinstance(mape, (int, float)):
            print(f"   📊 MAPE: {mape:.2f}%")
        else:
            print(f"   📊 MAPE: {mape}")

        print(f"   🤖 Model: {model_type}")

In [ ]:
# ===============================================================================
# 8. INTERACTIVE VISUALIZATIONS
# ===============================================================================

def create_alert_severity_chart(data):
    """Create alert severity distribution chart"""
    if data is None or 'alerts' not in data:
        print("❌ Cannot create severity chart - alerts data not available")
        return None

    alerts_df = data['alerts']

    try:
        severity_counts = alerts_df['alert_level'].value_counts()

        # Color mapping for severity levels
        colors = {'CRISIS': '#FF4444', 'ALERT': '#FF8800', 'WARNING': '#FFBB00'}

        fig = px.pie(
            values=severity_counts.values,
            names=severity_counts.index,
            title="🚨 Alert Severity Distribution",
            color=severity_counts.index,
            color_discrete_map=colors
        )

        fig.update_traces(textposition='inside', textinfo='percent+label')
        fig.update_layout(height=400, showlegend=True)
        fig.show()

        return fig
    except Exception as e:
        print(f"❌ Error creating severity chart: {e}")
        return None

def create_commodity_risk_chart(data):
    """Create commodity risk analysis chart"""
    if data is None or 'alerts' not in data or 'commodity_map' not in data:
        print("❌ Cannot create commodity chart - required data not available")
        return None

    alerts_df = data['alerts']
    commodity_map = data['commodity_map']

    try:
        # Get commodity names
        commodity_dict = dict(zip(commodity_map['commodity_code'], commodity_map['commodity_name']))

        # Commodity analysis
        commodity_analysis = alerts_df.groupby(['commodity_encoded', 'alert_level']).size().unstack(fill_value=0)
        commodity_analysis['Total'] = commodity_analysis.sum(axis=1)
        commodity_analysis = commodity_analysis.sort_values('Total', ascending=False).head(10)

        # Add commodity names
        commodity_analysis = commodity_analysis.reset_index()
        commodity_analysis['Commodity'] = [commodity_dict.get(code, f"Commodity {code}")
                                         for code in commodity_analysis['commodity_encoded']]

        # Create stacked bar chart
        alert_columns = [col for col in ['CRISIS', 'ALERT', 'WARNING'] if col in commodity_analysis.columns]

        fig = px.bar(
            commodity_analysis,
            x='Commodity',
            y=alert_columns,
            title="🌾 Top 10 Commodities - Alert Distribution",
            color_discrete_map={'CRISIS': '#FF4444', 'ALERT': '#FF8800', 'WARNING': '#FFBB00'}
        )

        fig.update_layout(
            height=500,
            xaxis_tickangle=-45,
            xaxis_title="Commodity",
            yaxis_title="Number of Alerts"
        )
        fig.show()

        return fig
    except Exception as e:
        print(f"❌ Error creating commodity chart: {e}")
        return None

def create_seasonal_pattern_chart(data):
    """Create seasonal alert pattern chart"""
    if data is None or 'alerts' not in data:
        print("❌ Cannot create seasonal chart - alerts data not available")
        return None

    alerts_df = data['alerts']

    try:
        # Check for season column (might be 'season' or 'current_season')
        season_col = 'season' if 'season' in alerts_df.columns else 'current_season'

        if season_col not in alerts_df.columns:
            print("❌ No season data found in alerts")
            return None

        seasonal_counts = alerts_df[season_col].value_counts()

        # Season color mapping
        season_colors = {'Deyr': '#8B4513', 'Gu': '#228B22', 'Xagaa': '#FFD700', 'Jilaal': '#CD853F'}

        fig = px.bar(
            x=seasonal_counts.index,
            y=seasonal_counts.values,
            title="📅 Seasonal Alert Patterns",
            color=seasonal_counts.index,
            color_discrete_map=season_colors
        )

        fig.update_layout(
            height=400,
            xaxis_title="Season",
            yaxis_title="Number of Alerts",
            showlegend=False
        )
        fig.show()

        return fig
    except Exception as e:
        print(f"❌ Error creating seasonal chart: {e}")
        return None

def create_regional_alert_map(data):
    """Create geographic map with color-coded regional risk levels"""
    if data is None or 'alerts' not in data:
        print("❌ Cannot create map - alerts data not available")
        return None

    alerts_df = data['alerts']

    # Check if alerts data has required geographic columns
    if not all(col in alerts_df.columns for col in ['latitude', 'longitude', 'region_encoded', 'region_name']):
        print("❌ Geographic coordinates or region data not available in alerts data")
        print(f"Available columns: {list(alerts_df.columns)}")
        return None

    try:
        # Create regional risk summary from alerts
        regional_risk = alerts_df.groupby(['region_encoded', 'region_name']).agg({
            'alert_level': lambda x: x.value_counts().index[0],  # Most frequent alert level
            'latitude': 'first',  # Take first latitude for each region
            'longitude': 'first',  # Take first longitude for each region
            'commodity_encoded': 'count'  # Total alerts
        }).reset_index()

        regional_risk.columns = ['region_encoded', 'region_name', 'risk_level', 'latitude', 'longitude', 'total_alerts']

        # Color mapping for risk levels
        color_map = {
            'CRISIS': '#FF0000',    # Red
            'ALERT': '#FF8800',     # Orange
            'WARNING': '#FFDD00'    # Yellow
        }

        # Create the map
        fig = px.scatter_mapbox(
            regional_risk,
            lat='latitude',
            lon='longitude',
            size='total_alerts',
            color='risk_level',
            color_discrete_map=color_map,
            hover_name='region_name',
            hover_data={
                'risk_level': True,
                'total_alerts': True,
                'latitude': False,
                'longitude': False
            },
            title="📍 Regional Risk-Level Geographic Alert Map",
            mapbox_style='open-street-map',
            height=600,
            size_max=20
        )

        fig.update_layout(
            mapbox=dict(center=dict(lat=5.15, lon=46.2), zoom=5.5),
            margin={"r":0,"t":30,"l":0,"b":0},
            legend_title="Alert Risk Level"
        )

        fig.show()
        return fig
    except Exception as e:
        print(f"❌ Error creating regional geographic map: {e}")
        return None

def create_regional_crisis_heatmap(data):
    """Create regional crisis intensity heatmap"""
    if data is None or 'regional_summary' not in data:
        print("❌ Cannot create regional heatmap - regional data not available")
        return None

    regional_summary = data['regional_summary']

    try:
        # Prepare data for heatmap
        heatmap_data = regional_summary[['Crisis_Count', 'Alert_Count', 'Warning_Count']].copy()

        fig = px.imshow(
            heatmap_data.T,
            y=['Crisis', 'Alert', 'Warning'],
            x=heatmap_data.index,
            color_continuous_scale='Reds',
            title="🌍 Regional Alert Intensity Heatmap",
            aspect='auto'
        )

        fig.update_layout(
            height=400,
            xaxis_title="Region",
            xaxis_tickangle=-45
        )
        fig.show()

        return fig
    except Exception as e:
        print(f"❌ Error creating regional heatmap: {e}")
        return None

In [ ]:
# ===============================================================================
# 9. MAIN DASHBOARD EXECUTION
# ===============================================================================

def run_complete_dashboard():
    """Execute complete dashboard display with visualizations"""
    print("🚀 RUNNING COMPLETE DASHBOARD...")

    if data is None:
        print("❌ Cannot run dashboard - data loading failed")
        return

    # Run all dashboard sections
    display_dashboard_overview(data)
    display_commodity_risk_analysis(data)
    display_regional_risk_analysis(data)
    display_crisis_alert_dashboard(data)
    display_system_health_dashboard(data)

    # Run interactive visualizations
    print(f"\n📊 GENERATING INTERACTIVE VISUALIZATIONS...")
    print("=" * 50)

    create_alert_severity_chart(data)
    create_commodity_risk_chart(data)
    create_seasonal_pattern_chart(data)
    create_regional_alert_map(data)
    create_regional_crisis_heatmap(data)
    create_crisis_trends_by_month(data)

    print(f"\n🎉 " + "=" * 60)
    print("   ✅ SOMALIA FOOD PRICE ALERT DASHBOARD READY")
    print("   📊 All metrics loaded and displayed successfully")
    print("   🚨 Crisis alerts identified and prioritized")
    print("   🌍 Regional analysis complete")
    print("   📈 System health validated")
    print("   📊 Interactive visualizations generated")
    print("🎉 " + "=" * 60)

# Execute the complete dashboard
if __name__ == "__main__":
    run_complete_dashboard()

# Download Dashboard

In [ ]:
# Download Dashboard
import plotly.io as pio
import io
from contextlib import redirect_stdout

def download_dashboard():
    """Download complete dashboard with overview and charts"""

    # Capture dashboard overview output
    overview_output = io.StringIO()
    with redirect_stdout(overview_output):
        display_dashboard_overview(data)
        display_commodity_risk_analysis(data)
        display_regional_risk_analysis(data)
        display_crisis_alert_dashboard(data)
        display_system_health_dashboard(data)

    # Create charts silently
    charts = []
    chart_functions = [
        ("Alert Severity", create_alert_severity_chart),
        ("Commodity Risk", create_commodity_risk_chart),
        ("Seasonal Patterns", create_seasonal_pattern_chart),
        ("Regional Map", create_regional_alert_map),
        ("Regional Heatmap", create_regional_crisis_heatmap),
        ("Crisis Trends by Month", create_crisis_trends_by_month)
    ]

    for name, func in chart_functions:
        try:
            with redirect_stdout(io.StringIO()):
                chart = func(data)
            if chart:
                charts.append(f'<h2>{name}</h2>{pio.to_html(chart, include_plotlyjs="cdn", full_html=False)}')
        except:
            pass

    # Create HTML
    html = f"""<!DOCTYPE html>
<html><head><title>Somalia Dashboard</title>
<style>body{{font-family:Arial;margin:20px;}}pre{{background:#f5f5f5;padding:15px;border-radius:5px;}}</style>
</head><body>
<h1>Somalia Food Price Alert Dashboard</h1>
<h2>Overview</h2>
<pre>{overview_output.getvalue()}</pre>
<h2>Charts</h2>
{''.join(charts)}
</body></html>"""

    # Download
    with open('dashboard.html', 'w', encoding='utf-8') as f:
        f.write(html)
    print("Dashboard downloaded!")

download_dashboard()

In [ ]:
# ===============================================================================
# 10. THESIS SCOPE AND LIMITATIONS ANALYSIS
# ===============================================================================

def display_thesis_scope_limitations():
    """Display thesis scope and future work recommendations"""

    print("\nTHESIS SCOPE AND FUTURE WORK RECOMMENDATIONS")
    print("=" * 80)

    # Study objectives addressed
    print("STUDY OBJECTIVES ADDRESSED:")
    print("=" * 60)
    print("   • Analyzed historical food price and economic indicators contributing to commodity price fluctuation across Somalia")
    print("   • Built and compared standalone and hybrid machine learning models for commodity price forecasting")
    print("   • Developed price-threshold-based alert mechanism highlighting regions and commodities at food insecurity risk")
    print("   • Provided actionable insights and recommendations for policymakers and stakeholders")

    # Current implementation scope
    print("\nCURRENT IMPLEMENTATION SCOPE:")
    print("=" * 60)
    print("   • Analysis period: Test dataset (May 15, 2024 - March 15, 2025) covering 12 regions in Somalia")
    print("   • Model performance: Random Forest achieved R² = 95.81%")
    print("   • Alert generation: 1,595 alerts across 30 markets")
    print("   • 90th percentile methodology: 764 high-priority alerts (14.3%)")
    print("   • Crisis identification: 400 locations requiring immediate response")
    print("   • Regional coverage: 12 regions with Sool and Nugaal highest risk")

    # Thesis contributions
    print("\nTHESIS CONTRIBUTIONS:")
    print("=" * 60)
    print("   • Developed machine learning-based food price forecasting model")
    print("   • Implemented 90th percentile threshold methodology with hierarchical fallbacks")
    print("   • Created district-to-regional administrative mapping for humanitarian operations")
    print("   • Built interactive dashboard for stakeholder decision-making")
    print("   • Demonstrated technical feasibility of automated early warning systems")

    # Future work
    print("\nFUTURE WORK RECOMMENDATIONS:")
    print("=" * 60)
    print("   • Expand temporal validation to include historical crisis periods")
    print("   • Validate alert system performance during 2011, 2017 Somalia famines")
    print("   • Incorporate additional external indicators (weather, conflict data)")
    print("   • Develop real-time data integration capabilities")
    print("   • Conduct field validation with WFP and humanitarian organizations")
    print("   • Extend coverage to remaining Somalia regions")

    # Deployment readiness
    print("\nDEPLOYMENT READINESS:")
    print("=" * 60)
    print("   • Technical framework: Complete")
    print("   • Proof of concept: Validated with 25.1% crisis rate detection")
    print("   • Operational deployment requires field validation and stakeholder training")


# Call the function
display_thesis_scope_limitations()